<span style="font-size:2em; font-weight:bold;">Objective 2: Evaluation of Ground Motion Spatial Correlation in New Zealand Through Empirical Approaches - Electronic Supplement A & B</span>
***

This code was written to conduct analyses as outlined in electronic supplements A and B which covers the following:

1) **Analysis and assessment of ground-motion selection on spatial correlations**

This analysis includes understanding the impact on the length scales and/or derived models if the minimum number of records is varied e.g., n > 20, 30, 40, 50, 60 and 80. In addition, models are fitted to assess the impact of considering all data versus various tectonic classes such as subduction and crustal events.

2) **Analysis and assessment of ground-motion model selection on spatial correlations**

This analysis involves utilizing the NGA-West2 and NGA-Sub models to validate whether or not the correlation structure varies. In addition, two NZ-specific GMMs are also analyzed.

# 1.0 Import modules and data

## 1.1 Import modules

In [ ]:
# Repository paths (works when Jupyter starts at the repository root or in notebooks/).
from pathlib import Path
import sys

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (parent for parent in (_current, *_current.parents)
     if (parent / "src" / "cbb21_cossim.py").is_file()
     and (parent / "data").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Repository root not found. Start Jupyter in the repository or notebooks/ directory.")

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = REPO_ROOT / "data"
RESIDUALS_DIR = DATA_DIR / "residuals"
SCM_DIR = DATA_DIR / "nzwide_scm"
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib as mpl
import matplotlib.gridspec as gridspec
import string

mpl.rcParams.update(mpl.rcParamsDefault)
import matplotlib.ticker as mtick
from matplotlib.ticker import AutoMinorLocator, LogLocator, NullFormatter, FuncFormatter
from scipy.interpolate import RegularGridInterpolator
from scipy.stats import bootstrap, gaussian_kde
from scipy.special import logsumexp

# scikitlearn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import CCA
import seaborn as sns
from itertools import combinations
from sklearn.neighbors import BallTree

## 1.2 Function definitions

### 1.2.1 Distance metrics

In [ ]:
# Angular distance given angle relative to earthquake
def getAngDist(theta1,theta2):
    cos_angle = np.cos(np.abs(theta1-theta2))
    distA = np.arccos(np.clip(cos_angle,-1,1))
    return distA

# Soil dissimilarity as a function of the absolute difference Vs30
def getSoilDiss(vs30_1,vs30_2):
    # vs1 = np.clip(vs30_1, 0, 1130)
    # vs2 = np.clip(vs30_2, 0, 1130)
    distS = np.abs(vs30_1-vs30_2)
    return distS

# Circular Median for angular distance
def circmedian(angles, period = 2*np.pi):
    """
    Compute the circular median of angles in radians on [0, 2π)
    """
    if len(angles) == 0:
        return np.nan

    # Wrap angles into [0, 2π)
    angles = np.mod(angles, period)
    sorted_angles = np.sort(angles)

    # Total circular deviation for each candidate
    def angular_distance(a, b):
        diff = np.abs(a - b)
        return np.minimum(diff, period - diff)

    total_deviation = np.array([
        np.sum(angular_distance(angle, sorted_angles)) for angle in sorted_angles
    ])

    return sorted_angles[np.argmin(total_deviation)]

# Circular Mean for angular distance
def circmean(angles, period=2*np.pi):
    """
    Compute the circular mean of angles in radians on [0, 2π)
    """
    angles = np.mod(angles, period)
    sin_sum = np.sum(np.sin(angles))
    cos_sum = np.sum(np.cos(angles))
    return np.arctan2(sin_sum, cos_sum) % period

# source: https://docs.openquake.org/oq-engine/2.7/_modules/openquake/hmtk/seismicity/utils.html#haversine
# def haversine_oq(lon1, lat1, lon2, lat2, radians=False, earth_rad=6371.227):
def haversine_oq(lon1, lat1, lon2, lat2, radians=False, earth_rad=6371):
    """
    Allows to calculate geographical distance
    using the haversine formula.

    :param lon1: longitude of the first set of locations
    :type lon1: numpy.ndarray
    :param lat1: latitude of the frist set of locations
    :type lat1: numpy.ndarray
    :param lon2: longitude of the second set of locations
    :type lon2: numpy.float64
    :param lat2: latitude of the second set of locations
    :type lat2: numpy.float64
    :keyword radians: states if locations are given in terms of radians
    :type radians: bool
    :keyword earth_rad: radius of the earth in km
    :type earth_rad: float
    :returns: geographical distance in km
    :rtype: numpy.ndarray
    """
    if not radians:
        cfact = np.pi / 180.
        lon1 = cfact * lon1
        lat1 = cfact * lat1
        lon2 = cfact * lon2
        lat2 = cfact * lat2

    # Number of locations in each set of points
    if not np.shape(lon1):
        nlocs1 = 1
        lon1 = np.array([lon1])
        lat1 = np.array([lat1])
    else:
        nlocs1 = np.max(np.shape(lon1))
    if not np.shape(lon2):
        nlocs2 = 1
        lon2 = np.array([lon2])
        lat2 = np.array([lat2])
    else:
        nlocs2 = np.max(np.shape(lon2))
    # Pre-allocate array
    distance = np.zeros((nlocs1, nlocs2))
    i = 0
    while i < nlocs2:
        # Perform distance calculation
        dlat = lat1 - lat2[i]
        dlon = lon1 - lon2[i]
        aval = (np.sin(dlat / 2.) ** 2.) + (np.cos(lat1) * np.cos(lat2[i]) *
                                            (np.sin(dlon / 2.) ** 2.))
        distance[:, i] = (2. * earth_rad * np.arctan2(np.sqrt(aval),
                                                      np.sqrt(1 - aval))).T
        i += 1
    return distance

### 1.2.2 Non-stationary correlation computation (Chen et al. 2021)

In [ ]:
def pearsoncorr_BM23(im_sim_all, nj_lim, hmax):
    """
    Optimized (same-structure) correlation cloud builder:
    - Keeps uncentered Pearson definition
    - Uses circmedian like your original
    - Returns the same columns (ordering preserved)
    """
    EARTH_R_KM = 6371.0088

    # Group by station (keep only the columns we actually use in the loop)
    needed_cols = ['evid', 'dw_bar', 'epicenter_azimuth']
    station_groups_full = dict(tuple(im_sim_all.groupby("sta")))
    station_groups = {sta: g[needed_cols].copy() for sta, g in station_groups_full.items()}

    station_meta = (
        im_sim_all.drop_duplicates('sta')
                  .set_index('sta')[['sta_lat','sta_lon','vs30','Z1.0','TRI','k0','TWI','geoage']]
    )
    stations = list(station_groups.keys())
    S = len(stations)

    # Build quick-lookup caches
    ev_sets = {sta: set(g['evid'].values) for sta, g in station_groups.items()}

    lat = station_meta['sta_lat'].to_dict()
    lon = station_meta['sta_lon'].to_dict()
    vs30 = station_meta['vs30'].to_dict()
    Z1   = station_meta['Z1.0'].to_dict()
    TRI  = station_meta['TRI'].to_dict()
    K0   = station_meta['k0'].to_dict()
    TWI  = station_meta['TWI'].to_dict()
    GEO  = station_meta['geoage'].to_dict()

    # spatial pre-pruning: generate only pairs with h <= hmax
    try:
        coords_rad = np.deg2rad(np.c_[ [lat[s] for s in stations], [lon[s] for s in stations] ])
        tree = BallTree(coords_rad, metric='haversine')
        # query radius in radians
        rad = (hmax / EARTH_R_KM) if np.isfinite(hmax) else np.pi
        nbrs = tree.query_radius(coords_rad, r=rad, return_distance=False)

        # Build unique i<j pairs from neighbor lists
        candidate_pairs = []
        for i, neigh in enumerate(nbrs):
            for j in neigh:
                if j > i:  # ensure i<j
                    candidate_pairs.append((i, j))
    except Exception:
        # Fallback: all pairs, distance check inside loop
        candidate_pairs = list(combinations(range(S), 2))

    # Accumulators
    pspear = []
    dsep = []
    stat_i_id = []
    stat_j_id = []
    nj = []
    stat_vs30_i = []
    stat_vs30_j = []
    stat_Z1_i = []
    stat_Z1_j = []
    stat_TRI_i = []
    stat_TRI_j = []
    stat_K0_i = []
    stat_K0_j = []
    stat_TWI_i = []
    stat_TWI_j = []
    stat_geoa_i = []
    stat_geoa_j = []
    stat_lat_i = []
    stat_lon_i = []
    stat_lat_j = []
    stat_lon_j = []
    theta_i = []
    theta_j = []

    # Loop over (unique) station pairs
    for i_idx, j_idx in candidate_pairs:
        sta_i = stations[i_idx]
        sta_j = stations[j_idx]

        # Distance early
        if not np.isfinite(hmax) or 'BallTree' in globals():
            pass
        else:
            distance_temp = haversine_oq(
                np.array([lon[sta_j]]), np.array([lat[sta_j]]),
                np.array([lon[sta_i]]), np.array([lat[sta_i]])
            )[0][0]
            if (distance_temp <= 0) or (distance_temp > hmax):
                continue

        # Overlap check via set intersection
        n_overlap = len(ev_sets[sta_i] & ev_sets[sta_j])
        if n_overlap < nj_lim:
            continue

        # Align on evid once (fast inner merge)
        gi = station_groups[sta_i]
        gj = station_groups[sta_j]
        aligned = gi.merge(gj, on='evid', how='inner', suffixes=('_i', '_j'))
        n_j_temp = len(aligned)
        if n_j_temp < nj_lim:
            continue

        # Distance (compute precisely once if using BallTree or fallback precise)
        distance_temp = haversine_oq(
            np.array([lon[sta_j]]), np.array([lat[sta_j]]),
            np.array([lon[sta_i]]), np.array([lat[sta_i]])
        )[0][0]
        if (distance_temp <= 0) or (distance_temp > hmax):
            continue

        # Uncentered Pearson p_hat
        xi = aligned['dw_bar_i'].to_numpy()
        xj = aligned['dw_bar_j'].to_numpy()
        num = np.dot(xi, xj)
        den = np.sqrt(np.dot(xi, xi) * np.dot(xj, xj))
        if den == 0:
            # degenerate; skip
            continue
        p_temp = num / den

        # Median azimuths over the *overlapped* events
        theta1_temp = circmedian(aligned['epicenter_azimuth_j'])
        theta2_temp = circmedian(aligned['epicenter_azimuth_i'])

        # Append values
        pspear.append(p_temp)
        dsep.append(distance_temp)
        stat_i_id.append(sta_i)
        stat_j_id.append(sta_j)
        nj.append(n_j_temp)

        stat_vs30_i.append(vs30[sta_i])
        stat_vs30_j.append(vs30[sta_j])
        stat_Z1_i.append(Z1[sta_i])
        stat_Z1_j.append(Z1[sta_j])
        stat_TRI_i.append(TRI[sta_i])
        stat_TRI_j.append(TRI[sta_j])
        stat_K0_i.append(K0[sta_i])
        stat_K0_j.append(K0[sta_j])
        stat_TWI_i.append(TWI[sta_i])
        stat_TWI_j.append(TWI[sta_j])
        stat_geoa_i.append(GEO[sta_i])
        stat_geoa_j.append(GEO[sta_j])
        theta_i.append(theta1_temp)
        theta_j.append(theta2_temp)
        stat_lat_i.append(lat[sta_i])
        stat_lon_i.append(lon[sta_i])
        stat_lat_j.append(lat[sta_j])
        stat_lon_j.append(lon[sta_j])

    # Assemble DataFrame
    cbsparcorr = pd.DataFrame()
    cbsparcorr['p_hat'] = pspear
    cbsparcorr['h'] = dsep
    cbsparcorr['sta_j'] = stat_j_id
    cbsparcorr['sta_k'] = stat_i_id
    cbsparcorr['n_j'] = nj
    cbsparcorr['vs30_j'] = stat_vs30_j
    cbsparcorr['vs30_k'] = stat_vs30_i
    cbsparcorr['dS'] = getSoilDiss(np.array(stat_vs30_i), np.array(stat_vs30_j))
    cbsparcorr['Z1_j'] = stat_Z1_j
    cbsparcorr['Z1_k'] = stat_Z1_i
    cbsparcorr['dZ1'] = np.abs(np.array(stat_Z1_i) - np.array(stat_Z1_j))
    cbsparcorr['TRI_j'] = stat_TRI_j
    cbsparcorr['TRI_k'] = stat_TRI_i
    cbsparcorr['dTRI'] = np.abs(np.array(stat_TRI_i) - np.array(stat_TRI_j))
    cbsparcorr['K0_j'] = stat_K0_j
    cbsparcorr['K0_k'] = stat_K0_i
    cbsparcorr['dK0'] = np.abs(np.array(stat_K0_i) - np.array(stat_K0_j))
    cbsparcorr['TWI_j'] = stat_TWI_j
    cbsparcorr['TWI_k'] = stat_TWI_i
    cbsparcorr['dTWI'] = np.abs(np.array(stat_TWI_i) - np.array(stat_TWI_j))
    cbsparcorr['geoa_j'] = stat_geoa_j
    cbsparcorr['geoa_k'] = stat_geoa_i
    cbsparcorr['dGEO'] = np.abs(np.array(stat_geoa_i) - np.array(stat_geoa_j))
    cbsparcorr['theta_i'] = theta_i
    cbsparcorr['theta_j'] = theta_j
    cbsparcorr['dA'] = getAngDist(cbsparcorr.theta_i, cbsparcorr.theta_j) * 180 / np.pi
    cbsparcorr['stat_lon_j'] = stat_lon_j
    cbsparcorr['stat_lat_j'] = stat_lat_j
    cbsparcorr['stat_lon_k'] = stat_lon_i
    cbsparcorr['stat_lat_k'] = stat_lat_i

    # Final filtering
    cbsparcorr = cbsparcorr[cbsparcorr.h > 0]
    if np.isfinite(hmax):
        cbsparcorr = cbsparcorr[cbsparcorr.h <= hmax]
    cbsparcorr = cbsparcorr[cbsparcorr.n_j >= nj_lim]
    cbsparcorr = cbsparcorr.sort_values('h').reset_index(drop=True)

    # Fisher transform
    # clip for numerical safety (avoid +/-1 exactly)
    p = np.clip(cbsparcorr['p_hat'].to_numpy(), -1 + 1e-12, 1 - 1e-12)
    cbsparcorr["z_p_hat"] = 0.5 * np.log((1 + p) / (1 - p))

    return cbsparcorr

### 1.2.3 Spatial Correlation Model

In [ ]:
# Euclidean Distance Correlation Model
def getpE(dE,lE,gamma_E):
    pE = np.zeros((len(dE),1))
    pE[:,0] = np.exp(-(dE/lE)**gamma_E)
    return pE

# Angular Distance Correlation Model
def getpA(dA,lA):
    pA = (1+dA/lA)*(1-dA/180)**(180/lA)
    return pA

# Soil Dissimilarity Correlation Model
def getpS(dS,lS):
    pS = np.exp(-dS/lS)
    return pS

# Basin Dissimilarity Correlation Model
def getpZ(dZ,lZ):
    pZ = np.exp(-dZ/lZ)
    return pZ

# Topographic Dissimilarity Correlation Model (Exponential)
def getpT(dT,lT):
    pT = np.exp(-dT/lT)
    return pT

# combine and call model: EA
def MpEA(pE,pA):
    pEA = pE*pA
    return pEA

# combine and call model: EAS
def MpEAS(pE,pA,pS,w):
    pEAS = pE*(w*pA+(1-w)*pS)
    return pEAS

# combine and call model: EASZ
def MpEASZ(pE,pA,pS,pZ,w,wS):
    pEASZ = pE*(w*pA+(1-w)*(pS*wS + pZ*(1-wS)))
    return pEASZ

# combine and call model: EAST
def MpEAST(pE,pA,pS,pT,w,wS):
    pEAST = pE*(w*pA+(1-w)*(pS*wS + pT*(1-wS)))
    return pEAST

# combine and call model: EASZT
def MpEASZT(pE, pA, pS, pZ, pT, w, wS, wZ, wT):
    pS_w = pS*wS
    pZ_w = pZ*wZ
    pT_w = pT*wT
    pEASZT = pE*(w*pA+(1-w)*(pS_w+pZ_w+pT_w))
    return pEASZT

def BM23EXT_CBB21(cbb21_pSAobs_T, period, parameters):

    # Fisher z residuals with clipping
    def fisher_residual(p, obs, n):
        p = np.clip(p, -0.9999, 0.9999)  # avoid ±1
        z_p = 0.5 * np.log((1 + p) / (1 - p))
        return (obs - z_p) * np.sqrt(n - 2)

    models = ["E","EA","EAS","EASZ","EAST","EASZT"]

    # Storage: each model has mean, mean5, mean95, std, std5, std95
    results = {m: {"mean": [], "mean5": [], "mean95": [],
                   "std": [], "std5": [], "std95": []}
               for m in models}

    for i in range(len(period)):
        obs = cbb21_pSAobs_T[i].z_p_hat
        n   = cbb21_pSAobs_T[i].n_j

        # --- Model E ---
        for suffix, key in [("", "mean"), ("_5", "mean5"), ("_95", "mean95")]:
            pEE = getpE(cbb21_pSAobs_T[i].h,
                        getattr(parameters, f"modelE_LE{suffix}")[i],
                        getattr(parameters, f"modelE_gammaE{suffix}")[i]).reshape(-1)
            eE  = fisher_residual(pEE, obs, n)
            results["E"][key].append(np.nanmean(eE))
            results["E"][key.replace("mean","std")].append(np.nanstd(eE))

        # --- Model EA ---
        for suffix, key in [("", "mean"), ("_5", "mean5"), ("_95", "mean95")]:
            pEAA = getpE(cbb21_pSAobs_T[i].h,
                         getattr(parameters, f"modelEA_LE{suffix}")[i],
                         getattr(parameters, f"modelEA_gammaE{suffix}")[i]).reshape(-1)
            pAA  = getpA(cbb21_pSAobs_T[i].dA,
                         getattr(parameters, f"modelEA_LA{suffix}")[i])
            pEA  = MpEA(pEAA, pAA)
            eEA  = fisher_residual(pEA, obs, n)
            results["EA"][key].append(np.nanmean(eEA))
            results["EA"][key.replace("mean","std")].append(np.nanstd(eEA))

        # --- Model EAS ---
        for suffix, key in [("", "mean"), ("_5", "mean5"), ("_95", "mean95")]:
            pE   = getpE(cbb21_pSAobs_T[i].h,
                         getattr(parameters, f"modelEAS_LE{suffix}")[i],
                         getattr(parameters, f"modelEAS_gammaE{suffix}")[i]).reshape(-1)
            pA   = getpA(cbb21_pSAobs_T[i].dA,
                         getattr(parameters, f"modelEAS_LA{suffix}")[i])
            pS   = getpS(cbb21_pSAobs_T[i].dS,
                         getattr(parameters, f"modelEAS_LS{suffix}")[i])
            pEAS = MpEAS(pE, pA, pS,
                         getattr(parameters, f"modelEAS_w{suffix}")[i])
            eEAS = fisher_residual(pEAS, obs, n)
            results["EAS"][key].append(np.nanmean(eEAS))
            results["EAS"][key.replace("mean","std")].append(np.nanstd(eEAS))

        # --- Model EASZ ---
        for suffix, key in [("", "mean"), ("_5", "mean5"), ("_95", "mean95")]:
            pE   = getpE(cbb21_pSAobs_T[i].h,
                         getattr(parameters, f"modelEASZ_LE{suffix}")[i],
                         getattr(parameters, f"modelEASZ_gammaE{suffix}")[i]).reshape(-1)
            pA   = getpA(cbb21_pSAobs_T[i].dA,
                         getattr(parameters, f"modelEASZ_LA{suffix}")[i])
            pS   = getpS(cbb21_pSAobs_T[i].dS,
                         getattr(parameters, f"modelEASZ_LS{suffix}")[i])
            pZ   = getpZ(cbb21_pSAobs_T[i].dZ1,
                         getattr(parameters, f"modelEASZ_LZ{suffix}")[i])
            pEASZ = MpEASZ(pE, pA, pS, pZ,
                           getattr(parameters, f"modelEASZ_w{suffix}")[i],
                           getattr(parameters, f"modelEASZ_wS{suffix}")[i])
            eEASZ = fisher_residual(pEASZ, obs, n)
            results["EASZ"][key].append(np.nanmean(eEASZ))
            results["EASZ"][key.replace("mean","std")].append(np.nanstd(eEASZ))

        # --- Model EAST ---
        for suffix, key in [("", "mean"), ("_5", "mean5"), ("_95", "mean95")]:
            pE   = getpE(cbb21_pSAobs_T[i].h,
                         getattr(parameters, f"modelEAST_LE{suffix}")[i],
                         getattr(parameters, f"modelEAST_gammaE{suffix}")[i]).reshape(-1)
            pA   = getpA(cbb21_pSAobs_T[i].dA,
                         getattr(parameters, f"modelEAST_LA{suffix}")[i])
            pS   = getpS(cbb21_pSAobs_T[i].dS,
                         getattr(parameters, f"modelEAST_LS{suffix}")[i])
            pT   = getpT(cbb21_pSAobs_T[i].dTRI,
                         getattr(parameters, f"modelEAST_LT{suffix}")[i])
            pEAST = MpEAST(pE, pA, pS, pT,
                           getattr(parameters, f"modelEAST_w{suffix}")[i],
                           getattr(parameters, f"modelEAST_wS{suffix}")[i])
            eEAST = fisher_residual(pEAST, obs, n)
            results["EAST"][key].append(np.nanmean(eEAST))
            results["EAST"][key.replace("mean","std")].append(np.nanstd(eEAST))

        # --- Model EASZT ---
        for suffix, key in [("", "mean"), ("_5", "mean5"), ("_95", "mean95")]:
            pE   = getpE(cbb21_pSAobs_T[i].h,
                         getattr(parameters, f"modelEASZT_LE{suffix}")[i],
                         getattr(parameters, f"modelEASZT_gammaE{suffix}")[i]).reshape(-1)
            pA   = getpA(cbb21_pSAobs_T[i].dA,
                         getattr(parameters, f"modelEASZT_LA{suffix}")[i])
            pS   = getpS(cbb21_pSAobs_T[i].dS,
                         getattr(parameters, f"modelEASZT_LS{suffix}")[i])
            pZ   = getpZ(cbb21_pSAobs_T[i].dZ1,
                         getattr(parameters, f"modelEASZT_LZ{suffix}")[i])
            pT   = getpT(cbb21_pSAobs_T[i].dTRI,
                         getattr(parameters, f"modelEASZT_LT{suffix}")[i])
            pEASZT = MpEASZT(pE, pA, pS, pZ, pT,
                             getattr(parameters, f"modelEASZT_w{suffix}")[i],
                             getattr(parameters, f"modelEASZT_wS{suffix}")[i],
                             getattr(parameters, f"modelEASZT_wZ{suffix}")[i],
                             getattr(parameters, f"modelEASZT_wT{suffix}")[i])
            eEASZT = fisher_residual(pEASZT, obs, n)
            results["EASZT"][key].append(np.nanmean(eEASZT))
            results["EASZT"][key.replace("mean","std")].append(np.nanstd(eEASZT))

    return results

## 1.3 Import Data

### 1.3.1 Import within-event residuals

In [ ]:
## Input directories
# input directory NGA-West2 and NGA-Sub
input_directory_ngaw2 = RESIDUALS_DIR
input_directory_ngas = RESIDUALS_DIR

# input directory NZ-GMM (Backbone model)
input_directory_nzgmm = RESIDUALS_DIR

# input directory n > 20
input_directory_n20 = RESIDUALS_DIR

## Within-event residual files per GMM
im_dwbar_ASK14 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_ASK14_dwbar_ay92.csv', dtype={"sta": "string"})
im_dwbar_BSSA14 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_BSSA14_dwbar_ay92.csv', dtype={"sta": "string"})
im_dwbar_CB14 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_CB14_dwbar_ay92.csv', dtype={"sta": "string"})
im_dwbar_CY14 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_CY14_dwbar_ay92.csv', dtype={"sta": "string"})
im_dwbar_AG20 = pd.read_csv(input_directory_ngas / r'NZGMDB_v4p3_AG20_dwbar_ay92.csv', dtype={"sta": "string"})
im_dwbar_P20 = pd.read_csv(input_directory_ngas / r'NZGMDB_v4p3_P20_dwbar_ay92.csv', dtype={"sta": "string"})
im_dwbar_K20 = pd.read_csv(input_directory_ngas / r'NZGMDB_v4p3_K20_dwbar_ay92.csv', dtype={"sta": "string"})
im_dwbar_ASK22 = pd.read_csv(input_directory_nzgmm / r'NZGMDB_v4p3_ASK22_dwbar_ay92.csv', dtype={"sta": "string"})
im_dwbar_S22 = pd.read_csv(input_directory_nzgmm / r'NZGMDB_v4p3_S22_dwbar_ay92.csv', dtype={"sta": "string"})
im_dwbar_CY14_n20 = pd.read_csv(input_directory_n20 / r'NZGMDB_v4p3_CY14_mw5_ay92_dwbar.csv', dtype={"sta": "string"})
im_dwbar_CY14_n40 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_CY14_dwbar_n40_ay92.csv', dtype={"sta": "string"})
im_dwbar_CY14_n50 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_CY14_dwbar_n50_ay92.csv', dtype={"sta": "string"})
im_dwbar_CY14_n60 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_CY14_dwbar_n60_ay92.csv', dtype={"sta": "string"})
im_dwbar_CY14_n80 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_CY14_dwbar_n80_ay92.csv', dtype={"sta": "string"})

### 1.3.2 Append additional information

In [ ]:
# import stations with metadata
stations_TRI = pd.read_csv(RESIDUALS_DIR / "site_table_TRI_multiscale_group_TWI.csv", dtype={"sta": "string"})
stations_TRI['k0'] = stations_TRI['k0_mean']
stations_TRI['sta_lat'] = stations_TRI['lat']
stations_TRI['sta_lon'] = stations_TRI['lon']
stations_TRI['geoage'] = stations_TRI['geoMA_min']
stations_filt = pd.DataFrame()
stations_filt = stations_TRI[['sta','sta_lat','sta_lon','k0','TWI','geoage']]

# append values to within-event residuals
im_dwbar_ASK14 = im_dwbar_ASK14.merge(stations_filt[['sta','sta_lat','sta_lon','k0']], on = 'sta', how = 'left')
im_dwbar_BSSA14 = im_dwbar_BSSA14.merge(stations_filt[['sta','sta_lat','sta_lon','k0']], on = 'sta', how = 'left')
im_dwbar_CB14 = im_dwbar_CB14.merge(stations_filt[['sta','sta_lat','sta_lon','k0']], on = 'sta', how = 'left')
im_dwbar_CY14 = im_dwbar_CY14.merge(stations_filt[['sta','sta_lat','sta_lon','k0']], on = 'sta', how = 'left')
im_dwbar_AG20 = im_dwbar_AG20.merge(stations_filt[['sta','sta_lat','sta_lon','k0']], on = 'sta', how = 'left')
im_dwbar_P20 = im_dwbar_P20.merge(stations_filt[['sta','sta_lat','sta_lon','k0']], on = 'sta', how = 'left')
im_dwbar_K20 = im_dwbar_K20.merge(stations_filt[['sta','sta_lat','sta_lon','k0']], on = 'sta', how = 'left')
im_dwbar_ASK22 = im_dwbar_ASK22.merge(stations_filt[['sta','sta_lat','sta_lon','k0']], on = 'sta', how = 'left')
im_dwbar_S22 = im_dwbar_S22.merge(stations_filt[['sta','sta_lat','sta_lon','k0']], on = 'sta', how = 'left')
im_dwbar_CY14_n20 = im_dwbar_CY14_n20.merge(stations_filt[['sta','k0','TWI','geoage']], on = 'sta', how = 'left')
im_dwbar_CY14_n40 = im_dwbar_CY14_n40.merge(stations_filt[['sta','sta_lat','sta_lon','k0']], on = 'sta', how = 'left')
im_dwbar_CY14_n50 = im_dwbar_CY14_n50.merge(stations_filt[['sta','sta_lat','sta_lon','k0']], on = 'sta', how = 'left')
im_dwbar_CY14_n60 = im_dwbar_CY14_n60.merge(stations_filt[['sta','sta_lat','sta_lon','k0']], on = 'sta', how = 'left')
im_dwbar_CY14_n80 = im_dwbar_CY14_n80.merge(stations_filt[['sta','sta_lat','sta_lon','k0']], on = 'sta', how = 'left')

### 1.3.2 Import SCM parameters

In [ ]:
# NGA-W2 GMM SCM parameters (n >= 30)
SCM_params_ASK14 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_ASK14_dwbar_ay92_param_summary.csv')
SCM_params_BSSA14 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_BSSA14_dwbar_ay92_param_summary.csv')
SCM_params_CB14 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_CB14_dwbar_ay92_param_summary.csv')
SCM_params_CY14 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_CY14_dwbar_ay92_param_summary.csv')

# NGA-Sub GMM SCM parameters (n >= 30)
SCM_params_AG20 =  pd.read_csv(input_directory_ngas / r'NZGMDB_v4p3_AG20_dwbar_ay92_param_summary.csv')
SCM_params_P20 =  pd.read_csv(input_directory_ngas / r'NZGMDB_v4p3_P20_dwbar_ay92_param_summary.csv')
SCM_params_K20 =  pd.read_csv(input_directory_ngas / r'NZGMDB_v4p3_K20_dwbar_ay92_param_summary.csv')

# NZ-Specific GMM SCM parameters (n >= 30)
SCM_params_ASK22 = pd.read_csv(input_directory_nzgmm / r'NZGMDB_v4p3_ASK22_dwbar_ay92_param_summary.csv')
SCM_params_S22 = pd.read_csv(input_directory_nzgmm / r'NZGMDB_v4p3_S22_dwbar_ay92_param_summary.csv')

# CY14 GMM scm parameters with various n's
SCM_params_CY14_n20 = pd.read_csv(input_directory_n20 / r'NZGMDB_v4p3_CY14_mw5_ay92_dwbar_param_summary.csv')
SCM_params_CY14_n30 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_CY14_dwbar_ay92_param_summary.csv')
SCM_params_CY14_n40 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_CY14_dwbar_n40_ay92_param_summary.csv')
SCM_params_CY14_n50 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_CY14_dwbar_n50_ay92_param_summary.csv')
SCM_params_CY14_n60 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_CY14_dwbar_n60_ay92_param_summary.csv')
SCM_params_CY14_n80 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_CY14_dwbar_n80_ay92_param_summary.csv')

# fitted period
pSA_T  = [0.01, 0.03, 0.06, 0.1, 0.3, 0.6, 1, 2, 3, 4, 5, 6]
pSAobs_T = ['PGA', 'pSA_0.03', 'pSA_0.06', 'pSA_0.1','pSA_0.3', 'pSA_0.6', 'pSA_1.0', 'pSA_2.0', 'pSA_3.0', 'pSA_4.0', 'pSA_5.0', 'pSA_6.0']

### 1.3.4 Import IM flatfile

In [ ]:
## Load NZ GMDB Flatfile
gm_im_table_50 = pd.read_csv(RESIDUALS_DIR / "ground_motion_im_table_rotd50_flat.csv", low_memory=False, dtype={"sta": "string"})

## Remove Duplicates BUT with HN channels as a priority
gm_im_table_50.sort_values('chan',ascending=False,inplace=True)
gm_im_table_50.drop_duplicates(subset=['evid','sta'],keep='first',inplace=True)
gm_im_table_50.sort_values('datetime',ascending=True,inplace=True)

## only keep evid, sta and mag columns
gm_im_table_50 = gm_im_table_50[['evid','sta','mag','r_rup']]

# merge with within-event residual files
im_dwbar_CY14_n20 = im_dwbar_CY14_n20.merge(gm_im_table_50, on = ['evid','sta'], how = 'left')
im_dwbar_CY14_n30 = im_dwbar_CY14.merge(gm_im_table_50, on = ['evid','sta'], how = 'left')
im_dwbar_CY14_n40 = im_dwbar_CY14_n40.merge(gm_im_table_50, on = ['evid','sta'], how = 'left')
im_dwbar_CY14_n50 = im_dwbar_CY14_n50.merge(gm_im_table_50, on = ['evid','sta'], how = 'left')
im_dwbar_CY14_n60 = im_dwbar_CY14_n60.merge(gm_im_table_50, on = ['evid','sta'], how = 'left')
im_dwbar_CY14_n80 = im_dwbar_CY14_n80.merge(gm_im_table_50, on = ['evid','sta'], how = 'left')

# crustal and subduction
im_dwbar_ASK14 = im_dwbar_ASK14.merge(gm_im_table_50, on = ['evid','sta'], how = 'left')
im_dwbar_AG20 = im_dwbar_AG20.merge(gm_im_table_50, on = ['evid','sta'], how = 'left')

## plots
# crustal vs subduction datasets
dataframes = [
    im_dwbar_ASK14,
    im_dwbar_AG20
]

fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharey=True, dpi = 200)
axes = axes.flatten()
labels = [f"({letter})" for letter in string.ascii_lowercase[:len(axes)]]
for ax, df, label in zip(axes, dataframes, labels):
    ax.scatter(df['r_rup'], df['mag'], alpha=0.6, edgecolor='k')
    ax.set_xscale('log')
    ax.set_xlabel(R'$R_{rup}$ (km)')
    ax.set_ylabel('Magnitude')
    ax.grid(True, which="both", ls="--", alpha=0.5)
    ax.text(-0.02, 1.0, label, transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top', ha='right')
plt.tight_layout()
plt.show()

# minimum observation ground moiton datasets
dataframes = [
    im_dwbar_CY14_n20,
    im_dwbar_CY14_n30,
    im_dwbar_CY14_n40,
    im_dwbar_CY14_n50,
    im_dwbar_CY14_n60,
    im_dwbar_CY14_n80
]

fig, axes = plt.subplots(2, 3, figsize=(12, 8), sharey=True, dpi = 200)

axes = axes.flatten()
labels = [f"({letter})" for letter in string.ascii_lowercase[:len(axes)]]
for ax, df, label in zip(axes, dataframes, labels):
    ax.scatter(df['r_rup'], df['mag'], alpha=0.6, edgecolor='k')
    ax.set_xscale('log')
    ax.set_xlabel(R'$R_{rup}$ (km)')
    ax.set_ylabel('Magnitude')
    ax.grid(True, which="both", ls="--", alpha=0.5)
    ax.text(-0.04, 1.00, label, transform=ax.transAxes,
            fontsize=12, fontweight='bold', va='top', ha='right')

plt.tight_layout()
plt.show()

### 13.5 Model comparisons LPPD

In [ ]:
# Crustal LPPDPs
LPPDP_reldiff_ASK14 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_ASK14_dwbar_ay92_LPPDP.csv')
LPPDP_reldiff_BSSA14 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_BSSA14_dwbar_ay92_LPPDP.csv')
LPPDP_reldiff_CB14 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_CB14_dwbar_ay92_LPPDP.csv')
LPPDP_reldiff_CY14 = pd.read_csv(input_directory_ngaw2 / r'NZGMDB_v4p3_CY14_dwbar_ay92_LPPDP.csv')
LPPDP_reldiff_ASK22 = pd.read_csv(input_directory_nzgmm / r'NZGMDB_v4p3_ASK22_dwbar_ay92_LPPDP.csv')
LPPDP_reldiff_S22 = pd.read_csv(input_directory_nzgmm / r'NZGMDB_v4p3_S22_dwbar_ay92_LPPDP.csv')

# Subduction LPPDPs
LPPDP_reldiff_AG20 = pd.read_csv(input_directory_ngas / r'NZGMDB_v4p3_AG20_dwbar_ay92_LPPDP.csv')
LPPDP_reldiff_K20 = pd.read_csv(input_directory_ngas / r'NZGMDB_v4p3_K20_dwbar_ay92_LPPDP.csv')
LPPDP_reldiff_P20 = pd.read_csv(input_directory_ngas / r'NZGMDB_v4p3_P20_dwbar_ay92_LPPDP.csv')

# 2.0 Compute Correlation Cloud (Chen et al., 2021)

## 2.1 Correlation cloud parameters

In [ ]:
n_rec = 6
hmax = 25

## 2.2 NGA-West 2 GMMs

In [ ]:
cbb21_ASK14 = []
for i in tqdm(range(len(pSAobs_T))):
    im_dwbar_ASK14['dw_bar'] = im_dwbar_ASK14[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_ASK14,n_rec,hmax)
    cbb21_ASK14.append(cbb21)

cbb21_BSSA14 = []
for i in tqdm(range(len(pSAobs_T))):
    im_dwbar_BSSA14['dw_bar'] = im_dwbar_BSSA14[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_BSSA14,n_rec,hmax)
    cbb21_BSSA14.append(cbb21)

cbb21_CB14 = []
for i in tqdm(range(len(pSAobs_T))):
    im_dwbar_CB14['dw_bar'] = im_dwbar_CB14[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_CB14,n_rec,hmax)
    cbb21_CB14.append(cbb21)

cbb21_CY14 = []
for i in tqdm(range(len(pSAobs_T))):
    im_dwbar_CY14['dw_bar'] = im_dwbar_CY14[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_CY14,n_rec,hmax)
    cbb21_CY14.append(cbb21)

## 2.3 NGA-Sub GMMs

In [ ]:
cbb21_AG20 = []
for i in tqdm(range(len(pSAobs_T))):
    im_dwbar_AG20['dw_bar'] = im_dwbar_AG20[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_AG20,n_rec,hmax)
    cbb21_AG20.append(cbb21)

cbb21_P20 = []
for i in tqdm(range(len(pSAobs_T))):
    im_dwbar_P20['dw_bar'] = im_dwbar_P20[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_P20,n_rec,hmax)
    cbb21_P20.append(cbb21)

cbb21_K20 = []
for i in tqdm(range(len(pSAobs_T))):
    im_dwbar_K20['dw_bar'] = im_dwbar_K20[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_K20,n_rec,hmax)
    cbb21_K20.append(cbb21)

## 2.4 NZ-Backbone GMMs

In [ ]:
cbb21_ASK22 = []
for i in tqdm(range(len(pSAobs_T))):
    im_dwbar_ASK22['dw_bar'] = im_dwbar_ASK22[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_ASK22,n_rec,hmax)
    cbb21_ASK22.append(cbb21)

cbb21_S22 = []
for i in tqdm(range(len(pSAobs_T))):
    im_dwbar_S22['dw_bar'] = im_dwbar_S22[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_S22,n_rec,hmax)
    cbb21_S22.append(cbb21)

## 2.5 Chiou & Young's 2014 GMM w/ varying observations

In [ ]:
cbb21_CY14_n20 = []
for i in tqdm(range(len(pSAobs_T))):
    im_dwbar_CY14_n20['dw_bar'] = im_dwbar_CY14_n20[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_CY14_n20,n_rec,hmax)
    cbb21_CY14_n20.append(cbb21)

cbb21_CY14_n30 = []
for i in tqdm(range(len(pSAobs_T))):
    im_dwbar_CY14['dw_bar'] = im_dwbar_CY14[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_CY14,n_rec,hmax)
    cbb21_CY14_n30.append(cbb21)

cbb21_CY14_n40 = []
for i in tqdm(range(len(pSAobs_T))):
    im_dwbar_CY14_n40['dw_bar'] = im_dwbar_CY14_n40[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_CY14_n40,n_rec,hmax)
    cbb21_CY14_n40.append(cbb21)

cbb21_CY14_n50 = []
for i in tqdm(range(len(pSAobs_T))):
    im_dwbar_CY14_n50['dw_bar'] = im_dwbar_CY14_n50[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_CY14_n50,n_rec,hmax)
    cbb21_CY14_n50.append(cbb21)

cbb21_CY14_n60 = []
for i in tqdm(range(len(pSAobs_T))):
    im_dwbar_CY14_n60['dw_bar'] = im_dwbar_CY14_n60[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_CY14_n60,n_rec,hmax)
    cbb21_CY14_n60.append(cbb21)

cbb21_CY14_n80 = []
for i in tqdm(range(len(pSAobs_T))):
    im_dwbar_CY14_n80['dw_bar'] = im_dwbar_CY14_n80[pSAobs_T[i]]
    cbb21 = pearsoncorr_BM23(im_dwbar_CY14_n80,n_rec,hmax)
    cbb21_CY14_n80.append(cbb21)

# 3.0 SCM Parameter plots

## 3.1 Crustal GMMs: NGA-West2 and NZ-GMM

### 3.1.1 Model E

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(12,4),dpi=200, constrained_layout=True)
ax[0].loglog(pSA_T, SCM_params_ASK14['modelE_LE'],'.-',label = 'ASK14', lw = 1)
ax[0].fill_between(pSA_T, SCM_params_ASK14['modelE_LE_5'], SCM_params_ASK14['modelE_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_BSSA14['modelE_LE'],'.-',label = 'BSSA14')
ax[0].fill_between(pSA_T, SCM_params_BSSA14['modelE_LE_5'], SCM_params_BSSA14['modelE_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_CB14['modelE_LE'],'.-', label='CB14')
ax[0].fill_between(pSA_T, SCM_params_CB14['modelE_LE_5'], SCM_params_CB14['modelE_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_CY14['modelE_LE'],'.-', label='CY14')
ax[0].fill_between(pSA_T, SCM_params_CY14['modelE_LE_5'], SCM_params_CY14['modelE_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_ASK22['modelE_LE'],'.-', label='A22')
ax[0].fill_between(pSA_T, SCM_params_ASK22['modelE_LE_5'], SCM_params_ASK22['modelE_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_S22['modelE_LE'],'.-', label='S22')
ax[0].fill_between(pSA_T, SCM_params_S22['modelE_LE_5'], SCM_params_S22['modelE_LE_95'], alpha = 0.2)

ax[1].semilogx(pSA_T, SCM_params_ASK14['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, SCM_params_ASK14['modelE_gammaE_5'], SCM_params_ASK14['modelE_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_BSSA14['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, SCM_params_BSSA14['modelE_gammaE_5'], SCM_params_BSSA14['modelE_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_CB14['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, SCM_params_CB14['modelE_gammaE_5'], SCM_params_CB14['modelE_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_CY14['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, SCM_params_CY14['modelE_gammaE_5'], SCM_params_CY14['modelE_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_ASK22['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, SCM_params_ASK22['modelE_gammaE_5'], SCM_params_ASK22['modelE_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_S22['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, SCM_params_S22['modelE_gammaE_5'], SCM_params_S22['modelE_gammaE_95'], alpha = 0.2)

for i in range(2):
    ax[i].set_xlabel('Period, T(s)')
    ax[i].set_xlim([min(pSA_T),max(pSA_T)])
    ax[i].grid(which = 'both')
    ax[i].set_xlim(0.01,6)
# ax[0].set_yscale('log')
ax[0].set_ylabel(r'$L_E$ (km)')
ax[1].set_ylabel(r'$\gamma_E$')
ax[0].set_ylim([1,500])
ax[1].set_ylim([0,1])
ax[0].legend(loc = 'upper left', ncol = 3)
plt.show()

### 3.1.2 Model EA

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(12,4),dpi=200, constrained_layout=True)
ax[0].loglog(pSA_T, SCM_params_ASK14['modelEA_LE'],'.-',label = 'ASK14', lw = 1)
ax[0].fill_between(pSA_T, SCM_params_ASK14['modelEA_LE_5'], SCM_params_ASK14['modelEA_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_BSSA14['modelEA_LE'],'.-',label = 'BSSA14')
ax[0].fill_between(pSA_T, SCM_params_BSSA14['modelEA_LE_5'], SCM_params_BSSA14['modelEA_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_CB14['modelEA_LE'],'.-', label='CB14')
ax[0].fill_between(pSA_T, SCM_params_CB14['modelEA_LE_5'], SCM_params_CB14['modelEA_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_CY14['modelEA_LE'],'.-', label='CY14')
ax[0].fill_between(pSA_T, SCM_params_CY14['modelEA_LE_5'], SCM_params_CY14['modelEA_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_ASK22['modelEA_LE'],'.-', label='A22')
ax[0].fill_between(pSA_T, SCM_params_ASK22['modelEA_LE_5'], SCM_params_ASK22['modelEA_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_S22['modelEA_LE'],'.-', label='S22')
ax[0].fill_between(pSA_T, SCM_params_S22['modelEA_LE_5'], SCM_params_S22['modelEA_LE_95'], alpha = 0.2)

ax[1].semilogx(pSA_T, SCM_params_ASK14['modelEA_gammaE'],'.-',label = 'ASK14', lw = 1)
ax[1].fill_between(pSA_T, SCM_params_ASK14['modelEA_gammaE_5'], SCM_params_ASK14['modelEA_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_BSSA14['modelEA_gammaE'],'.-',label = 'BSSA14')
ax[1].fill_between(pSA_T, SCM_params_BSSA14['modelEA_gammaE_5'], SCM_params_BSSA14['modelEA_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_CB14['modelEA_gammaE'],'.-', label='CB14')
ax[1].fill_between(pSA_T, SCM_params_CB14['modelEA_gammaE_5'], SCM_params_CB14['modelEA_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_CY14['modelEA_gammaE'],'.-', label='CY14')
ax[1].fill_between(pSA_T, SCM_params_CY14['modelEA_gammaE_5'], SCM_params_CY14['modelEA_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_ASK22['modelEA_gammaE'],'.-', label='A22')
ax[1].fill_between(pSA_T, SCM_params_ASK22['modelEA_gammaE_5'], SCM_params_ASK22['modelEA_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_S22['modelEA_gammaE'],'.-', label='S22')
ax[1].fill_between(pSA_T, SCM_params_S22['modelEA_gammaE_5'], SCM_params_S22['modelEA_gammaE_95'], alpha = 0.2)

ax[2].semilogx(pSA_T, SCM_params_ASK14['modelEA_LA'],'.-',label = 'ASK14', lw = 1)
ax[2].fill_between(pSA_T, SCM_params_ASK14['modelEA_LA_5'], SCM_params_ASK14['modelEA_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, SCM_params_BSSA14['modelEA_LA'],'.-',label = 'BSSA14')
ax[2].fill_between(pSA_T, SCM_params_BSSA14['modelEA_LA_5'], SCM_params_BSSA14['modelEA_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, SCM_params_CB14['modelEA_LA'],'.-', label='CB14')
ax[2].fill_between(pSA_T, SCM_params_CB14['modelEA_LA_5'], SCM_params_CB14['modelEA_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, SCM_params_CY14['modelEA_LA'],'.-', label='CY14')
ax[2].fill_between(pSA_T, SCM_params_CY14['modelEA_LA_5'], SCM_params_CY14['modelEA_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, SCM_params_ASK22['modelEA_LA'],'.-', label='A22')
ax[2].fill_between(pSA_T, SCM_params_ASK22['modelEA_LA_5'], SCM_params_ASK22['modelEA_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, SCM_params_S22['modelEA_LA'],'.-', label='S22')
ax[2].fill_between(pSA_T, SCM_params_S22['modelEA_LA_5'], SCM_params_S22['modelEA_LA_95'], alpha = 0.2)

for i in range(3):
    ax[i].set_xlabel('Period, T(s)')
    ax[i].set_xlim([min(pSA_T),max(pSA_T)])
    ax[i].grid(which = 'both')
    ax[i].set_xlim(0.01,6)
ax[0].set_ylabel(r'$L_E$ (km)')
ax[1].set_ylabel(r'$\gamma_E$')
ax[2].set_ylabel(r'$L_A(\degree)$')
ax[0].set_ylim([1,500])
ax[1].set_ylim([0,1])
ax[2].set_ylim([0,45])
ax[1].legend(loc = 'upper center', ncol = 3)
plt.show()

### 3.1.3 Model EAS

In [ ]:
fig = plt.figure(figsize=(12,6),dpi=200, constrained_layout=True)
gs = fig.add_gridspec(2, 3)
ax00 = fig.add_subplot(gs[0,0])
ax01 = fig.add_subplot(gs[0,1])
ax02 = fig.add_subplot(gs[0,2])
ax10 = fig.add_subplot(gs[1,0])
ax11 = fig.add_subplot(gs[1,1:3])

ax00.loglog(pSA_T, SCM_params_ASK14['modelEAS_LE'],'.-',label = 'ASK14', lw = 1)
ax00.fill_between(pSA_T, SCM_params_ASK14['modelEAS_LE_5'], SCM_params_ASK14['modelEAS_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_BSSA14['modelEAS_LE'],'.-',label = 'BSSA14')
ax00.fill_between(pSA_T, SCM_params_BSSA14['modelEAS_LE_5'], SCM_params_BSSA14['modelEAS_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CB14['modelEAS_LE'],'.-', label='CB14')
ax00.fill_between(pSA_T, SCM_params_CB14['modelEAS_LE_5'], SCM_params_CB14['modelEAS_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14['modelEAS_LE'],'.-', label='CY14')
ax00.fill_between(pSA_T, SCM_params_CY14['modelEAS_LE_5'], SCM_params_CY14['modelEAS_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_ASK22['modelEAS_LE'],'.-', label='A22')
ax00.fill_between(pSA_T, SCM_params_ASK22['modelEAS_LE_5'], SCM_params_ASK22['modelEAS_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_S22['modelEAS_LE'],'.-', label='S22')
ax00.fill_between(pSA_T, SCM_params_S22['modelEAS_LE_5'], SCM_params_S22['modelEAS_LE_95'], alpha = 0.2)

ax01.semilogx(pSA_T, SCM_params_ASK14['modelEAS_gammaE'],'.-',label = 'ASK14', lw = 1)
ax01.fill_between(pSA_T, SCM_params_ASK14['modelEAS_gammaE_5'], SCM_params_ASK14['modelEAS_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_BSSA14['modelEAS_gammaE'],'.-', label = 'BSSA14')
ax01.fill_between(pSA_T, SCM_params_BSSA14['modelEAS_gammaE_5'], SCM_params_BSSA14['modelEAS_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CB14['modelEAS_gammaE'],'.-', label='CB14')
ax01.fill_between(pSA_T, SCM_params_CB14['modelEAS_gammaE_5'], SCM_params_CB14['modelEAS_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14['modelEAS_gammaE'],'.-', label='CY14')
ax01.fill_between(pSA_T, SCM_params_CY14['modelEAS_gammaE_5'], SCM_params_CY14['modelEAS_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_ASK22['modelEAS_gammaE'],'.-', label='A22')
ax01.fill_between(pSA_T, SCM_params_ASK22['modelEAS_gammaE_5'], SCM_params_ASK22['modelEAS_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_S22['modelEAS_gammaE'],'.-', label='S22')
ax01.fill_between(pSA_T, SCM_params_S22['modelEAS_gammaE_5'], SCM_params_S22['modelEAS_gammaE_95'], alpha = 0.2)

ax02.semilogx(pSA_T, SCM_params_ASK14['modelEAS_LA'],'.-',label = 'ASK14', lw = 1)
ax02.fill_between(pSA_T, SCM_params_ASK14['modelEAS_LA_5'], SCM_params_ASK14['modelEAS_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_BSSA14['modelEAS_LA'],'.-',label = 'BSSA14')
ax02.fill_between(pSA_T, SCM_params_BSSA14['modelEAS_LA_5'], SCM_params_BSSA14['modelEAS_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CB14['modelEAS_LA'],'.-', label='CB14')
ax02.fill_between(pSA_T, SCM_params_CB14['modelEAS_LA_5'], SCM_params_CB14['modelEAS_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14['modelEAS_LA'],'.-', label='CY14')
ax02.fill_between(pSA_T, SCM_params_CY14['modelEAS_LA_5'], SCM_params_CY14['modelEAS_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_ASK22['modelEAS_LA'],'.-', label='A22')
ax02.fill_between(pSA_T, SCM_params_ASK22['modelEAS_LA_5'], SCM_params_ASK22['modelEAS_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_S22['modelEAS_LA'],'.-', label='S22')
ax02.fill_between(pSA_T, SCM_params_S22['modelEAS_LA_5'], SCM_params_S22['modelEAS_LA_95'], alpha = 0.2)

ax10.loglog(pSA_T, SCM_params_ASK14['modelEAS_LS'],'.-',label = 'ASK14', lw = 1)
ax10.fill_between(pSA_T, SCM_params_ASK14['modelEAS_LS_5'], SCM_params_ASK14['modelEAS_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_BSSA14['modelEAS_LS'],'.-',label = 'BSSA14')
ax10.fill_between(pSA_T, SCM_params_BSSA14['modelEAS_LS_5'], SCM_params_BSSA14['modelEAS_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CB14['modelEAS_LS'],'.-', label='CB14')
ax10.fill_between(pSA_T, SCM_params_CB14['modelEAS_LS_5'], SCM_params_CB14['modelEAS_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14['modelEAS_LS'],'.-', label='CY14')
ax10.fill_between(pSA_T, SCM_params_CY14['modelEAS_LS_5'], SCM_params_CY14['modelEAS_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_ASK22['modelEAS_LS'],'.-', label='A22')
ax10.fill_between(pSA_T, SCM_params_ASK22['modelEAS_LS_5'], SCM_params_ASK22['modelEAS_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_S22['modelEAS_LS'],'.-', label='S22')
ax10.fill_between(pSA_T, SCM_params_S22['modelEAS_LS_5'], SCM_params_S22['modelEAS_LS_95'], alpha = 0.2)

ax11.semilogx(pSA_T, SCM_params_ASK14['modelEAS_w'],'.-',label = 'ASK14', lw = 1)
ax11.fill_between(pSA_T, SCM_params_ASK14['modelEAS_w_5'], SCM_params_ASK14['modelEAS_w_95'], alpha = 0.2)
ax11.semilogx(pSA_T, SCM_params_BSSA14['modelEAS_w'],'.-',label = 'BSSA14')
ax11.fill_between(pSA_T, SCM_params_BSSA14['modelEAS_w_5'], SCM_params_BSSA14['modelEAS_w_95'], alpha = 0.2)
ax11.semilogx(pSA_T, SCM_params_CB14['modelEAS_w'],'.-', label='CB14')
ax11.fill_between(pSA_T, SCM_params_CB14['modelEAS_w_5'], SCM_params_CB14['modelEAS_w_95'], alpha = 0.2)
ax11.semilogx(pSA_T, SCM_params_CY14['modelEAS_w'],'.-', label='CY14')
ax11.fill_between(pSA_T, SCM_params_CY14['modelEAS_w_5'], SCM_params_CY14['modelEAS_w_95'], alpha = 0.2)
ax11.semilogx(pSA_T, SCM_params_ASK22['modelEAS_w'],'.-', label='A22')
ax11.fill_between(pSA_T, SCM_params_ASK22['modelEAS_w_5'], SCM_params_ASK22['modelEAS_w_95'], alpha = 0.2)
ax11.semilogx(pSA_T, SCM_params_S22['modelEAS_w'],'.-', label='S22')
ax11.fill_between(pSA_T, SCM_params_S22['modelEAS_w_5'], SCM_params_S22['modelEAS_w_95'], alpha = 0.2)

ax00.set_xlabel('Period, T(s)')
ax01.set_xlabel('Period, T(s)')
ax02.set_xlabel('Period, T(s)')
ax10.set_xlabel('Period, T(s)')
ax11.set_xlabel('Period, T(s)')

ax00.set_ylabel(r'$L_E$ (km)')
ax01.set_ylabel(r'$\gamma_E$')
ax02.set_ylabel(r'$L_A\ (^\circ)$')
ax10.set_ylabel(r'$L_S$ (m/s)')
ax11.set_ylabel(r'$w$')

ax00.set_xlim(0.01, 6)
ax01.set_xlim(0.01, 6)
ax02.set_xlim(0.01, 6)
ax10.set_xlim(0.01, 6)
ax11.set_xlim(0.01, 6)

ax00.set_ylim(1,500)
ax01.set_ylim(0,1)
ax02.set_ylim(0,45)
ax10.set_ylim(10,5000)
ax11.set_ylim(0,1)

for ax in [ax00, ax01, ax02, ax10, ax11]:
    ax.grid(which='both')

ax01.legend(loc='upper center', ncol=3)

plt.show()

### 3.1.4 Model EASZ

In [ ]:
fig = plt.figure(figsize=(12,8),dpi=200, constrained_layout=True)
gs = fig.add_gridspec(3, 3)
ax00 = fig.add_subplot(gs[0,0])
ax01 = fig.add_subplot(gs[0,1])
ax02 = fig.add_subplot(gs[0,2])
ax10 = fig.add_subplot(gs[1,0])
ax11 = fig.add_subplot(gs[1,1])
ax12 = fig.add_subplot(gs[1,2])
ax30 = fig.add_subplot(gs[2,0:3])

ax00.loglog(pSA_T, SCM_params_ASK14['modelEASZ_LE'],'.-',label = 'ASK14', lw = 1)
ax00.fill_between(pSA_T, SCM_params_ASK14['modelEASZ_LE_5'], SCM_params_ASK14['modelEASZ_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_BSSA14['modelEASZ_LE'],'.-',label = 'BSSA14')
ax00.fill_between(pSA_T, SCM_params_BSSA14['modelEASZ_LE_5'], SCM_params_BSSA14['modelEASZ_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CB14['modelEASZ_LE'],'.-', label='CB14')
ax00.fill_between(pSA_T, SCM_params_CB14['modelEASZ_LE_5'], SCM_params_CB14['modelEASZ_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14['modelEASZ_LE'],'.-', label='CY14')
ax00.fill_between(pSA_T, SCM_params_CY14['modelEASZ_LE_5'], SCM_params_CY14['modelEASZ_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_ASK22['modelEASZ_LE'],'.-', label='A22')
ax00.fill_between(pSA_T, SCM_params_ASK22['modelEASZ_LE_5'], SCM_params_ASK22['modelEASZ_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_S22['modelEASZ_LE'],'.-', label='S22')
ax00.fill_between(pSA_T, SCM_params_S22['modelEASZ_LE_5'], SCM_params_S22['modelEASZ_LE_95'], alpha = 0.2)

ax01.semilogx(pSA_T, SCM_params_ASK14['modelEASZ_gammaE'],'.-',label = 'ASK14', lw = 1)
ax01.fill_between(pSA_T, SCM_params_ASK14['modelEASZ_gammaE_5'], SCM_params_ASK14['modelEASZ_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_BSSA14['modelEASZ_gammaE'],'.-',label = 'BSSA14')
ax01.fill_between(pSA_T, SCM_params_BSSA14['modelEASZ_gammaE_5'], SCM_params_BSSA14['modelEASZ_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CB14['modelEASZ_gammaE'],'.-', label='CB14')
ax01.fill_between(pSA_T, SCM_params_CB14['modelEASZ_gammaE_5'], SCM_params_CB14['modelEASZ_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14['modelEASZ_gammaE'],'.-', label='CY14')
ax01.fill_between(pSA_T, SCM_params_CY14['modelEASZ_gammaE_5'], SCM_params_CY14['modelEASZ_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_ASK22['modelEASZ_gammaE'],'.-', label='A22')
ax01.fill_between(pSA_T, SCM_params_ASK22['modelEASZ_gammaE_5'], SCM_params_ASK22['modelEASZ_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_S22['modelEASZ_gammaE'],'.-', label='S22')
ax01.fill_between(pSA_T, SCM_params_S22['modelEASZ_gammaE_5'], SCM_params_S22['modelEASZ_gammaE_95'], alpha = 0.2)

ax02.semilogx(pSA_T, SCM_params_ASK14['modelEASZ_LA'],'.-',label = 'ASK14', lw = 1)
ax02.fill_between(pSA_T, SCM_params_ASK14['modelEASZ_LA_5'], SCM_params_ASK14['modelEASZ_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_BSSA14['modelEASZ_LA'],'.-',label = 'BSSA14')
ax02.fill_between(pSA_T, SCM_params_BSSA14['modelEASZ_LA_5'], SCM_params_BSSA14['modelEASZ_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CB14['modelEASZ_LA'],'.-', label='CB14')
ax02.fill_between(pSA_T, SCM_params_CB14['modelEASZ_LA_5'], SCM_params_CB14['modelEASZ_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14['modelEASZ_LA'],'.-', label='CY14')
ax02.fill_between(pSA_T, SCM_params_CY14['modelEASZ_LA_5'], SCM_params_CY14['modelEASZ_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_ASK22['modelEASZ_LA'],'.-', label='A22')
ax02.fill_between(pSA_T, SCM_params_ASK22['modelEASZ_LA_5'], SCM_params_ASK22['modelEASZ_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_S22['modelEASZ_LA'],'.-', label='S22')
ax02.fill_between(pSA_T, SCM_params_S22['modelEASZ_LA_5'], SCM_params_S22['modelEASZ_LA_95'], alpha = 0.2)

ax10.loglog(pSA_T, SCM_params_ASK14['modelEASZ_LS'],'.-',label = 'ASK14', lw = 1)
ax10.fill_between(pSA_T, SCM_params_ASK14['modelEASZ_LS_5'], SCM_params_ASK14['modelEASZ_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_BSSA14['modelEASZ_LS'],'.-',label = 'BSSA14')
ax10.fill_between(pSA_T, SCM_params_BSSA14['modelEASZ_LS_5'], SCM_params_BSSA14['modelEASZ_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CB14['modelEASZ_LS'],'.-', label='CB14')
ax10.fill_between(pSA_T, SCM_params_CB14['modelEASZ_LS_5'], SCM_params_CB14['modelEASZ_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14['modelEASZ_LS'],'.-', label='CY14')
ax10.fill_between(pSA_T, SCM_params_CY14['modelEASZ_LS_5'], SCM_params_CY14['modelEASZ_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_ASK22['modelEASZ_LS'],'.-', label='A22')
ax10.fill_between(pSA_T, SCM_params_ASK22['modelEASZ_LS_5'], SCM_params_ASK22['modelEASZ_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_S22['modelEASZ_LS'],'.-', label='S22')
ax10.fill_between(pSA_T, SCM_params_S22['modelEASZ_LS_5'], SCM_params_S22['modelEASZ_LS_95'], alpha = 0.2)

ax11.loglog(pSA_T, SCM_params_ASK14['modelEASZ_LZ'],'.-',label = 'ASK14', lw = 1)
ax11.fill_between(pSA_T, SCM_params_ASK14['modelEASZ_LZ_5'], SCM_params_ASK14['modelEASZ_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_BSSA14['modelEASZ_LZ'],'.-',label = 'BSSA14')
ax11.fill_between(pSA_T, SCM_params_BSSA14['modelEASZ_LZ_5'], SCM_params_BSSA14['modelEASZ_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CB14['modelEASZ_LZ'],'.-', label='CB14')
ax11.fill_between(pSA_T, SCM_params_CB14['modelEASZ_LZ_5'], SCM_params_CB14['modelEASZ_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CY14['modelEASZ_LZ'],'.-', label='CY14')
ax11.fill_between(pSA_T, SCM_params_CY14['modelEASZ_LZ_5'], SCM_params_CY14['modelEASZ_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_ASK22['modelEASZ_LZ'],'.-', label='A22')
ax11.fill_between(pSA_T, SCM_params_ASK22['modelEASZ_LZ_5'], SCM_params_ASK22['modelEASZ_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_S22['modelEASZ_LZ'],'.-', label='S22')
ax11.fill_between(pSA_T, SCM_params_S22['modelEASZ_LZ_5'], SCM_params_S22['modelEASZ_LZ_95'], alpha = 0.2)

ax12.semilogx(pSA_T, SCM_params_ASK14['modelEASZ_wS'],'.-',label = 'ASK14', lw = 1)
ax12.fill_between(pSA_T, SCM_params_ASK14['modelEASZ_wS_5'], SCM_params_ASK14['modelEASZ_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_BSSA14['modelEASZ_wS'],'.-',label = 'BSSA14')
ax12.fill_between(pSA_T, SCM_params_BSSA14['modelEASZ_wS_5'], SCM_params_BSSA14['modelEASZ_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_CB14['modelEASZ_wS'],'.-', label='CB14')
ax12.fill_between(pSA_T, SCM_params_CB14['modelEASZ_wS_5'], SCM_params_CB14['modelEASZ_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_CY14['modelEASZ_wS'],'.-', label='CY14')
ax12.fill_between(pSA_T, SCM_params_CY14['modelEASZ_wS_5'], SCM_params_CY14['modelEASZ_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_ASK22['modelEASZ_wS'],'.-', label='A22')
ax12.fill_between(pSA_T, SCM_params_ASK22['modelEASZ_wS_5'], SCM_params_ASK22['modelEASZ_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_S22['modelEASZ_wS'],'.-', label='S22')
ax12.fill_between(pSA_T, SCM_params_S22['modelEASZ_wS_5'], SCM_params_S22['modelEASZ_wS_95'], alpha = 0.2)

ax30.semilogx(pSA_T, SCM_params_ASK14['modelEASZ_w'],'.-',label = 'ASK14', lw = 1)
ax30.fill_between(pSA_T, SCM_params_ASK14['modelEASZ_w_5'], SCM_params_ASK14['modelEASZ_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_BSSA14['modelEASZ_w'],'.-',label = 'BSSA14')
ax30.fill_between(pSA_T, SCM_params_BSSA14['modelEASZ_w_5'], SCM_params_BSSA14['modelEASZ_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CB14['modelEASZ_w'],'.-', label='CB14')
ax30.fill_between(pSA_T, SCM_params_CB14['modelEASZ_w_5'], SCM_params_CB14['modelEASZ_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CY14['modelEASZ_w'],'.-', label='CY14')
ax30.fill_between(pSA_T, SCM_params_CY14['modelEASZ_w_5'], SCM_params_CY14['modelEASZ_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_ASK22['modelEASZ_w'],'.-', label='A22')
ax30.fill_between(pSA_T, SCM_params_ASK22['modelEASZ_w_5'], SCM_params_ASK22['modelEASZ_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_S22['modelEASZ_w'],'.-', label='S22')
ax30.fill_between(pSA_T, SCM_params_S22['modelEASZ_w_5'], SCM_params_S22['modelEASZ_w_95'], alpha = 0.2)

ax00.set_xlabel('Period, T(s)')
ax01.set_xlabel('Period, T(s)')
ax02.set_xlabel('Period, T(s)')
ax10.set_xlabel('Period, T(s)')
ax11.set_xlabel('Period, T(s)')
ax12.set_xlabel('Period, T(s)')
ax30.set_xlabel('Period, T(s)')

ax00.set_ylabel(r'$L_E$ (km)')
ax01.set_ylabel(r'$\gamma_E$')
ax02.set_ylabel(r'$L_A\ (^\circ)$')
ax10.set_ylabel(r'$L_S$ (m/s)')
ax11.set_ylabel(r'$L_Z$ (m)')
ax12.set_ylabel(r'$w_S$')
ax30.set_ylabel(r'$w$')

ax00.set_xlim(0.01, 6)
ax01.set_xlim(0.01, 6)
ax02.set_xlim(0.01, 6)
ax10.set_xlim(0.01, 6)
ax11.set_xlim(0.01, 6)
ax12.set_xlim(0.01, 6)
ax30.set_xlim(0.01, 6)

ax00.set_ylim(1,500)
ax01.set_ylim(0,1)
ax02.set_ylim(0,45)
ax10.set_ylim(10,5000)
ax11.set_ylim(10,5000)
ax12.set_ylim(0,1)
ax30.set_ylim(0,1)

for ax in [ax00, ax01, ax02, ax10, ax11, ax12, ax30]:
    ax.grid(which='both')

ax01.legend(loc='upper center', ncol=3, fontsize = 9.7)

plt.show()

### 3.1.5 Model EAST

In [ ]:
fig = plt.figure(figsize=(12,8),dpi=200, constrained_layout=True)
gs = fig.add_gridspec(3, 3)
ax00 = fig.add_subplot(gs[0,0])
ax01 = fig.add_subplot(gs[0,1])
ax02 = fig.add_subplot(gs[0,2])
ax10 = fig.add_subplot(gs[1,0])
ax11 = fig.add_subplot(gs[1,1])
ax12 = fig.add_subplot(gs[1,2])
ax30 = fig.add_subplot(gs[2,0:3])

ax00.loglog(pSA_T, SCM_params_ASK14['modelEAST_LE'],'.-',label = 'ASK14', lw = 1)
ax00.fill_between(pSA_T, SCM_params_ASK14['modelEAST_LE_5'], SCM_params_ASK14['modelEAST_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_BSSA14['modelEAST_LE'],'.-',label = 'BSSA14')
ax00.fill_between(pSA_T, SCM_params_BSSA14['modelEAST_LE_5'], SCM_params_BSSA14['modelEAST_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CB14['modelEAST_LE'],'.-', label='CB14')
ax00.fill_between(pSA_T, SCM_params_CB14['modelEAST_LE_5'], SCM_params_CB14['modelEAST_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14['modelEAST_LE'],'.-', label='CY14')
ax00.fill_between(pSA_T, SCM_params_CY14['modelEAST_LE_5'], SCM_params_CY14['modelEAST_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_ASK22['modelEAST_LE'],'.-', label='A22')
ax00.fill_between(pSA_T, SCM_params_ASK22['modelEAST_LE_5'], SCM_params_ASK22['modelEAST_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_S22['modelEAST_LE'],'.-', label='S22')
ax00.fill_between(pSA_T, SCM_params_S22['modelEAST_LE_5'], SCM_params_S22['modelEAST_LE_95'], alpha = 0.2)

ax01.semilogx(pSA_T, SCM_params_ASK14['modelEAST_gammaE'],'.-',label = 'ASK14', lw = 1)
ax01.fill_between(pSA_T, SCM_params_ASK14['modelEAST_gammaE_5'], SCM_params_ASK14['modelEAST_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_BSSA14['modelEAST_gammaE'],'.-',label = 'BSSA14')
ax01.fill_between(pSA_T, SCM_params_BSSA14['modelEAST_gammaE_5'], SCM_params_BSSA14['modelEAST_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CB14['modelEAST_gammaE'],'.-', label='CB14')
ax01.fill_between(pSA_T, SCM_params_CB14['modelEAST_gammaE_5'], SCM_params_CB14['modelEAST_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14['modelEAST_gammaE'],'.-', label='CY14')
ax01.fill_between(pSA_T, SCM_params_CY14['modelEAST_gammaE_5'], SCM_params_CY14['modelEAST_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_ASK22['modelEAST_gammaE'],'.-', label='A22')
ax01.fill_between(pSA_T, SCM_params_ASK22['modelEAST_gammaE_5'], SCM_params_ASK22['modelEAST_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_S22['modelEAST_gammaE'],'.-', label='S22')
ax01.fill_between(pSA_T, SCM_params_S22['modelEAST_gammaE_5'], SCM_params_S22['modelEAST_gammaE_95'], alpha = 0.2)

ax02.semilogx(pSA_T, SCM_params_ASK14['modelEAST_LA'],'.-',label = 'ASK14', lw = 1)
ax02.fill_between(pSA_T, SCM_params_ASK14['modelEAST_LA_5'], SCM_params_ASK14['modelEAST_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_BSSA14['modelEAST_LA'],'.-',label = 'BSSA14')
ax02.fill_between(pSA_T, SCM_params_BSSA14['modelEAST_LA_5'], SCM_params_BSSA14['modelEAST_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CB14['modelEAST_LA'],'.-', label='CB14')
ax02.fill_between(pSA_T, SCM_params_CB14['modelEAST_LA_5'], SCM_params_CB14['modelEAST_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14['modelEAST_LA'],'.-', label='CY14')
ax02.fill_between(pSA_T, SCM_params_CY14['modelEAST_LA_5'], SCM_params_CY14['modelEAST_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_ASK22['modelEAST_LA'],'.-', label='A22')
ax02.fill_between(pSA_T, SCM_params_ASK22['modelEAST_LA_5'], SCM_params_ASK22['modelEAST_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_S22['modelEAST_LA'],'.-', label='S22')
ax02.fill_between(pSA_T, SCM_params_S22['modelEAST_LA_5'], SCM_params_S22['modelEAST_LA_95'], alpha = 0.2)

ax10.loglog(pSA_T, SCM_params_ASK14['modelEAST_LS'],'.-',label = 'ASK14', lw = 1)
ax10.fill_between(pSA_T, SCM_params_ASK14['modelEAST_LS_5'], SCM_params_ASK14['modelEAST_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_BSSA14['modelEAST_LS'],'.-',label = 'BSSA14')
ax10.fill_between(pSA_T, SCM_params_BSSA14['modelEAST_LS_5'], SCM_params_BSSA14['modelEAST_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CB14['modelEAST_LS'],'.-', label='CB14')
ax10.fill_between(pSA_T, SCM_params_CB14['modelEAST_LS_5'], SCM_params_CB14['modelEAST_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14['modelEAST_LS'],'.-', label='CY14')
ax10.fill_between(pSA_T, SCM_params_CY14['modelEAST_LS_5'], SCM_params_CY14['modelEAST_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_ASK22['modelEAST_LS'],'.-', label='A22')
ax10.fill_between(pSA_T, SCM_params_ASK22['modelEAST_LS_5'], SCM_params_ASK22['modelEAST_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_S22['modelEAST_LS'],'.-', label='S22')
ax10.fill_between(pSA_T, SCM_params_S22['modelEAST_LS_5'], SCM_params_S22['modelEAST_LS_95'], alpha = 0.2)

ax11.loglog(pSA_T, SCM_params_ASK14['modelEAST_LT'],'.-',label = 'ASK14', lw = 1)
ax11.fill_between(pSA_T, SCM_params_ASK14['modelEAST_LT_5'], SCM_params_ASK14['modelEAST_LT_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_BSSA14['modelEAST_LT'],'.-',label = 'BSSA14')
ax11.fill_between(pSA_T, SCM_params_BSSA14['modelEAST_LT_5'], SCM_params_BSSA14['modelEAST_LT_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CB14['modelEAST_LT'],'.-', label='CB14')
ax11.fill_between(pSA_T, SCM_params_CB14['modelEAST_LT_5'], SCM_params_CB14['modelEAST_LT_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CY14['modelEAST_LT'],'.-', label='CY14')
ax11.fill_between(pSA_T, SCM_params_CY14['modelEAST_LT_5'], SCM_params_CY14['modelEAST_LT_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_ASK22['modelEAST_LT'],'.-', label='A22')
ax11.fill_between(pSA_T, SCM_params_ASK22['modelEAST_LT_5'], SCM_params_ASK22['modelEAST_LT_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_S22['modelEAST_LT'],'.-', label='S22')
ax11.fill_between(pSA_T, SCM_params_S22['modelEAST_LT_5'], SCM_params_S22['modelEAST_LT_95'], alpha = 0.2)

ax12.semilogx(pSA_T, SCM_params_ASK14['modelEAST_wS'],'.-',label = 'ASK14', lw = 1)
ax12.fill_between(pSA_T, SCM_params_ASK14['modelEAST_wS_5'], SCM_params_ASK14['modelEAST_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_BSSA14['modelEAST_wS'],'.-',label = 'BSSA14')
ax12.fill_between(pSA_T, SCM_params_BSSA14['modelEAST_wS_5'], SCM_params_BSSA14['modelEAST_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_CB14['modelEAST_wS'],'.-', label='CB14')
ax12.fill_between(pSA_T, SCM_params_CB14['modelEAST_wS_5'], SCM_params_CB14['modelEAST_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_CY14['modelEAST_wS'],'.-', label='CY14')
ax12.fill_between(pSA_T, SCM_params_CY14['modelEAST_wS_5'], SCM_params_CY14['modelEAST_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_ASK22['modelEAST_wS'],'.-', label='A22')
ax12.fill_between(pSA_T, SCM_params_ASK22['modelEAST_wS_5'], SCM_params_ASK22['modelEAST_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_S22['modelEAST_wS'],'.-', label='S22')
ax12.fill_between(pSA_T, SCM_params_S22['modelEAST_wS_5'], SCM_params_S22['modelEAST_wS_95'], alpha = 0.2)

ax30.semilogx(pSA_T, SCM_params_ASK14['modelEAST_w'],'.-',label = 'ASK14', lw = 1)
ax30.fill_between(pSA_T, SCM_params_ASK14['modelEAST_w_5'], SCM_params_ASK14['modelEAST_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_BSSA14['modelEAST_w'],'.-',label = 'BSSA14')
ax30.fill_between(pSA_T, SCM_params_BSSA14['modelEAST_w_5'], SCM_params_BSSA14['modelEAST_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CB14['modelEAST_w'],'.-', label='CB14')
ax30.fill_between(pSA_T, SCM_params_CB14['modelEAST_w_5'], SCM_params_CB14['modelEAST_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CY14['modelEAST_w'],'.-', label='CY14')
ax30.fill_between(pSA_T, SCM_params_CY14['modelEAST_w_5'], SCM_params_CY14['modelEAST_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_ASK22['modelEAST_w'],'.-', label='A22')
ax30.fill_between(pSA_T, SCM_params_ASK22['modelEAST_w_5'], SCM_params_ASK22['modelEAST_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_S22['modelEAST_w'],'.-', label='S22')
ax30.fill_between(pSA_T, SCM_params_S22['modelEAST_w_5'], SCM_params_S22['modelEAST_w_95'], alpha = 0.2)

ax00.set_xlabel('Period, T(s)')
ax01.set_xlabel('Period, T(s)')
ax02.set_xlabel('Period, T(s)')
ax10.set_xlabel('Period, T(s)')
ax11.set_xlabel('Period, T(s)')
ax12.set_xlabel('Period, T(s)')
ax30.set_xlabel('Period, T(s)')

ax00.set_ylabel(r'$L_E$ (km)')
ax01.set_ylabel(r'$\gamma_E$')
ax02.set_ylabel(r'$L_A\ (^\circ)$')
ax10.set_ylabel(r'$L_S$ (m/s)')
ax11.set_ylabel(r'$L_T$ (m)')
ax12.set_ylabel(r'$w_S$')
ax30.set_ylabel(r'$w$')

ax00.set_xlim(0.01, 6)
ax01.set_xlim(0.01, 6)
ax02.set_xlim(0.01, 6)
ax10.set_xlim(0.01, 6)
ax11.set_xlim(0.01, 6)
ax12.set_xlim(0.01, 6)
ax30.set_xlim(0.01, 6)

ax00.set_ylim(1,500)
ax01.set_ylim(0,1)
ax02.set_ylim(0,45)
ax10.set_ylim(10,5000)
ax11.set_ylim(1,500)
ax12.set_ylim(0,1)
ax30.set_ylim(0,1)

for ax in [ax00, ax01, ax02, ax10, ax11, ax12, ax30]:
    ax.grid(which='both')

ax01.legend(loc='upper center', ncol=3, fontsize = 9.7)

plt.show()

### 3.1.6 Model EASZT

In [ ]:
fig = plt.figure(figsize=(12,9),dpi=200, constrained_layout=True)
gs = fig.add_gridspec(4, 3)
ax00 = fig.add_subplot(gs[0,0])
ax01 = fig.add_subplot(gs[0,1])
ax02 = fig.add_subplot(gs[0,2])
ax10 = fig.add_subplot(gs[1,0])
ax11 = fig.add_subplot(gs[1,1])
ax12 = fig.add_subplot(gs[1,2])
ax20 = fig.add_subplot(gs[2,0])
ax21 = fig.add_subplot(gs[2,1])
ax22 = fig.add_subplot(gs[2,2])
ax30 = fig.add_subplot(gs[3,0:3])

ax00.loglog(pSA_T, SCM_params_ASK14['modelEASZT_LE'],'.-',label = 'ASK14', lw = 1)
ax00.fill_between(pSA_T, SCM_params_ASK14['modelEASZT_LE_5'], SCM_params_ASK14['modelEASZT_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_BSSA14['modelEASZT_LE'],'.-',label = 'BSSA14')
ax00.fill_between(pSA_T, SCM_params_BSSA14['modelEASZT_LE_5'], SCM_params_BSSA14['modelEASZT_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CB14['modelEASZT_LE'],'.-', label='CB14')
ax00.fill_between(pSA_T, SCM_params_CB14['modelEASZT_LE_5'], SCM_params_CB14['modelEASZT_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14['modelEASZT_LE'],'.-', label='CY14')
ax00.fill_between(pSA_T, SCM_params_CY14['modelEASZT_LE_5'], SCM_params_CY14['modelEASZT_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_ASK22['modelEASZT_LE'],'.-', label='A22')
ax00.fill_between(pSA_T, SCM_params_ASK22['modelEASZT_LE_5'], SCM_params_ASK22['modelEASZT_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_S22['modelEASZT_LE'],'.-', label='S22')
ax00.fill_between(pSA_T, SCM_params_S22['modelEASZT_LE_5'], SCM_params_S22['modelEASZT_LE_95'], alpha = 0.2)

ax01.semilogx(pSA_T, SCM_params_ASK14['modelEASZT_gammaE'],'.-',label = 'ASK14', lw = 1)
ax01.fill_between(pSA_T, SCM_params_ASK14['modelEASZT_gammaE_5'], SCM_params_ASK14['modelEASZT_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_BSSA14['modelEASZT_gammaE'],'.-',label = 'BSSA14')
ax01.fill_between(pSA_T, SCM_params_BSSA14['modelEASZT_gammaE_5'], SCM_params_BSSA14['modelEASZT_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CB14['modelEASZT_gammaE'],'.-', label='CB14')
ax01.fill_between(pSA_T, SCM_params_CB14['modelEASZT_gammaE_5'], SCM_params_CB14['modelEASZT_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14['modelEASZT_gammaE'],'.-', label='CY14')
ax01.fill_between(pSA_T, SCM_params_CY14['modelEASZT_gammaE_5'], SCM_params_CY14['modelEASZT_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_ASK22['modelEAST_gammaE'],'.-', label='A22')
ax01.fill_between(pSA_T, SCM_params_ASK22['modelEASZT_gammaE_5'], SCM_params_ASK22['modelEASZT_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_S22['modelEASZT_gammaE'],'.-', label='S22')
ax01.fill_between(pSA_T, SCM_params_S22['modelEASZT_gammaE_5'], SCM_params_S22['modelEASZT_gammaE_95'], alpha = 0.2)

ax02.semilogx(pSA_T, SCM_params_ASK14['modelEASZT_LA'],'.-',label = 'ASK14', lw = 1)
ax02.fill_between(pSA_T, SCM_params_ASK14['modelEASZT_LA_5'], SCM_params_ASK14['modelEASZT_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_BSSA14['modelEASZT_LA'],'.-',label = 'BSSA14')
ax02.fill_between(pSA_T, SCM_params_BSSA14['modelEASZT_LA_5'], SCM_params_BSSA14['modelEASZT_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CB14['modelEASZT_LA'],'.-', label='CB14')
ax02.fill_between(pSA_T, SCM_params_CB14['modelEASZT_LA_5'], SCM_params_CB14['modelEASZT_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14['modelEASZT_LA'],'.-', label='CY14')
ax02.fill_between(pSA_T, SCM_params_CY14['modelEASZT_LA_5'], SCM_params_CY14['modelEASZT_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_ASK22['modelEASZT_LA'],'.-', label='A22')
ax02.fill_between(pSA_T, SCM_params_ASK22['modelEASZT_LA_5'], SCM_params_ASK22['modelEASZT_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_S22['modelEASZT_LA'],'.-', label='S22')
ax02.fill_between(pSA_T, SCM_params_S22['modelEASZT_LA_5'], SCM_params_S22['modelEASZT_LA_95'], alpha = 0.2)

ax10.loglog(pSA_T, SCM_params_ASK14['modelEASZT_LS'],'.-',label = 'ASK14', lw = 1)
ax10.fill_between(pSA_T, SCM_params_ASK14['modelEASZT_LS_5'], SCM_params_ASK14['modelEASZT_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_BSSA14['modelEASZT_LS'],'.-',label = 'BSSA14')
ax10.fill_between(pSA_T, SCM_params_BSSA14['modelEASZT_LS_5'], SCM_params_BSSA14['modelEASZT_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CB14['modelEASZT_LS'],'.-', label='CB14')
ax10.fill_between(pSA_T, SCM_params_CB14['modelEASZT_LS_5'], SCM_params_CB14['modelEASZT_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14['modelEASZT_LS'],'.-', label='CY14')
ax10.fill_between(pSA_T, SCM_params_CY14['modelEASZT_LS_5'], SCM_params_CY14['modelEASZT_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_ASK22['modelEASZT_LS'],'.-', label='A22')
ax10.fill_between(pSA_T, SCM_params_ASK22['modelEASZT_LS_5'], SCM_params_ASK22['modelEASZT_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_S22['modelEASZT_LS'],'.-', label='S22')
ax10.fill_between(pSA_T, SCM_params_S22['modelEASZT_LS_5'], SCM_params_S22['modelEASZT_LS_95'], alpha = 0.2)

ax11.loglog(pSA_T, SCM_params_ASK14['modelEASZT_LZ'],'.-',label = 'ASK14', lw = 1)
ax11.fill_between(pSA_T, SCM_params_ASK14['modelEASZT_LZ_5'], SCM_params_ASK14['modelEASZT_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_BSSA14['modelEASZT_LZ'],'.-',label = 'BSSA14')
ax11.fill_between(pSA_T, SCM_params_BSSA14['modelEASZT_LZ_5'], SCM_params_BSSA14['modelEASZT_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CB14['modelEASZT_LZ'],'.-', label='CB14')
ax11.fill_between(pSA_T, SCM_params_CB14['modelEASZT_LZ_5'], SCM_params_CB14['modelEASZT_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CY14['modelEASZT_LZ'],'.-', label='CY14')
ax11.fill_between(pSA_T, SCM_params_CY14['modelEASZT_LZ_5'], SCM_params_CY14['modelEASZT_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_ASK22['modelEASZT_LZ'],'.-', label='A22')
ax11.fill_between(pSA_T, SCM_params_ASK22['modelEASZT_LZ_5'], SCM_params_ASK22['modelEASZT_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_S22['modelEASZT_LZ'],'.-', label='S22')
ax11.fill_between(pSA_T, SCM_params_S22['modelEASZT_LZ_5'], SCM_params_S22['modelEASZT_LZ_95'], alpha = 0.2)

ax12.loglog(pSA_T, SCM_params_ASK14['modelEASZT_LT'],'.-',label = 'ASK14', lw = 1)
ax12.fill_between(pSA_T, SCM_params_ASK14['modelEASZT_LT_5'], SCM_params_ASK14['modelEASZT_LT_95'], alpha = 0.2)
ax12.loglog(pSA_T, SCM_params_BSSA14['modelEASZT_LT'],'.-',label = 'BSSA14')
ax12.fill_between(pSA_T, SCM_params_BSSA14['modelEASZT_LT_5'], SCM_params_BSSA14['modelEASZT_LT_95'], alpha = 0.2)
ax12.loglog(pSA_T, SCM_params_CB14['modelEASZT_LT'],'.-', label='CB14')
ax12.fill_between(pSA_T, SCM_params_CB14['modelEASZT_LT_5'], SCM_params_CB14['modelEASZT_LT_95'], alpha = 0.2)
ax12.loglog(pSA_T, SCM_params_CY14['modelEASZT_LT'],'.-', label='CY14')
ax12.fill_between(pSA_T, SCM_params_CY14['modelEASZT_LT_5'], SCM_params_CY14['modelEASZT_LT_95'], alpha = 0.2)
ax12.loglog(pSA_T, SCM_params_ASK22['modelEASZT_LT'],'.-', label='A22')
ax12.fill_between(pSA_T, SCM_params_ASK22['modelEASZT_LT_5'], SCM_params_ASK22['modelEASZT_LT_95'], alpha = 0.2)
ax12.loglog(pSA_T, SCM_params_S22['modelEASZT_LT'],'.-', label='S22')
ax12.fill_between(pSA_T, SCM_params_S22['modelEASZT_LT_5'], SCM_params_S22['modelEASZT_LT_95'], alpha = 0.2)

ax20.semilogx(pSA_T, SCM_params_ASK14['modelEASZT_wS'],'.-',label = 'ASK14', lw = 1)
ax20.fill_between(pSA_T, SCM_params_ASK14['modelEASZT_wS_5'], SCM_params_ASK14['modelEASZT_wS_95'], alpha = 0.2)
ax20.semilogx(pSA_T, SCM_params_BSSA14['modelEASZT_wS'],'.-',label = 'BSSA14')
ax20.fill_between(pSA_T, SCM_params_BSSA14['modelEASZT_wS_5'], SCM_params_BSSA14['modelEASZT_wS_95'], alpha = 0.2)
ax20.semilogx(pSA_T, SCM_params_CB14['modelEASZT_wS'],'.-', label='CB14')
ax20.fill_between(pSA_T, SCM_params_CB14['modelEASZT_wS_5'], SCM_params_CB14['modelEASZT_wS_95'], alpha = 0.2)
ax20.semilogx(pSA_T, SCM_params_CY14['modelEASZT_wS'],'.-', label='CY14')
ax20.fill_between(pSA_T, SCM_params_CY14['modelEASZT_wS_5'], SCM_params_CY14['modelEASZT_wS_95'], alpha = 0.2)
ax20.semilogx(pSA_T, SCM_params_ASK22['modelEASZT_wS'],'.-', label='A22')
ax20.fill_between(pSA_T, SCM_params_ASK22['modelEASZT_wS_5'], SCM_params_ASK22['modelEASZT_wS_95'], alpha = 0.2)
ax20.semilogx(pSA_T, SCM_params_S22['modelEASZT_wS'],'.-', label='S22')
ax20.fill_between(pSA_T, SCM_params_S22['modelEASZT_wS_5'], SCM_params_S22['modelEASZT_wS_95'], alpha = 0.2)

ax21.semilogx(pSA_T, SCM_params_ASK14['modelEASZT_wZ'],'.-',label = 'ASK14', lw = 1)
ax21.fill_between(pSA_T, SCM_params_ASK14['modelEASZT_wZ_5'], SCM_params_ASK14['modelEASZT_wZ_95'], alpha = 0.2)
ax21.semilogx(pSA_T, SCM_params_BSSA14['modelEASZT_wZ'],'.-',label = 'BSSA14')
ax21.fill_between(pSA_T, SCM_params_BSSA14['modelEASZT_wZ_5'], SCM_params_BSSA14['modelEASZT_wZ_95'], alpha = 0.2)
ax21.semilogx(pSA_T, SCM_params_CB14['modelEASZT_wZ'],'.-', label='CB14')
ax21.fill_between(pSA_T, SCM_params_CB14['modelEASZT_wZ_5'], SCM_params_CB14['modelEASZT_wZ_95'], alpha = 0.2)
ax21.semilogx(pSA_T, SCM_params_CY14['modelEASZT_wZ'],'.-', label='CY14')
ax21.fill_between(pSA_T, SCM_params_CY14['modelEASZT_wZ_5'], SCM_params_CY14['modelEASZT_wZ_95'], alpha = 0.2)
ax21.semilogx(pSA_T, SCM_params_ASK22['modelEASZT_wZ'],'.-', label='A22')
ax21.fill_between(pSA_T, SCM_params_ASK22['modelEASZT_wZ_5'], SCM_params_ASK22['modelEASZT_wZ_95'], alpha = 0.2)
ax21.semilogx(pSA_T, SCM_params_S22['modelEASZT_wZ'],'.-', label='S22')
ax21.fill_between(pSA_T, SCM_params_S22['modelEASZT_wZ_5'], SCM_params_S22['modelEASZT_wZ_95'], alpha = 0.2)

ax22.semilogx(pSA_T, SCM_params_ASK14['modelEASZT_wT'],'.-',label = 'ASK14', lw = 1)
ax22.fill_between(pSA_T, SCM_params_ASK14['modelEASZT_wT_5'], SCM_params_ASK14['modelEASZT_wT_95'], alpha = 0.2)
ax22.semilogx(pSA_T, SCM_params_BSSA14['modelEASZT_wT'],'.-',label = 'BSSA14')
ax22.fill_between(pSA_T, SCM_params_BSSA14['modelEASZT_wT_5'], SCM_params_BSSA14['modelEASZT_wT_95'], alpha = 0.2)
ax22.semilogx(pSA_T, SCM_params_CB14['modelEASZT_wT'],'.-', label='CB14')
ax22.fill_between(pSA_T, SCM_params_CB14['modelEASZT_wT_5'], SCM_params_CB14['modelEASZT_wT_95'], alpha = 0.2)
ax22.semilogx(pSA_T, SCM_params_CY14['modelEASZT_wT'],'.-', label='CY14')
ax22.fill_between(pSA_T, SCM_params_CY14['modelEASZT_wT_5'], SCM_params_CY14['modelEASZT_wT_95'], alpha = 0.2)
ax22.semilogx(pSA_T, SCM_params_ASK22['modelEASZT_wT'],'.-', label='A22')
ax22.fill_between(pSA_T, SCM_params_ASK22['modelEASZT_wT_5'], SCM_params_ASK22['modelEASZT_wT_95'], alpha = 0.2)
ax22.semilogx(pSA_T, SCM_params_S22['modelEASZT_wT'],'.-', label='S22')
ax22.fill_between(pSA_T, SCM_params_S22['modelEASZT_wT_5'], SCM_params_S22['modelEASZT_wT_95'], alpha = 0.2)

ax30.semilogx(pSA_T, SCM_params_ASK14['modelEASZT_w'],'.-',label = 'ASK14', lw = 1)
ax30.fill_between(pSA_T, SCM_params_ASK14['modelEASZT_w_5'], SCM_params_ASK14['modelEASZT_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_BSSA14['modelEASZT_w'],'.-',label = 'BSSA14')
ax30.fill_between(pSA_T, SCM_params_BSSA14['modelEASZT_w_5'], SCM_params_BSSA14['modelEASZT_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CB14['modelEASZT_w'],'.-', label='CB14')
ax30.fill_between(pSA_T, SCM_params_CB14['modelEASZT_w_5'], SCM_params_CB14['modelEASZT_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CY14['modelEASZT_w'],'.-', label='CY14')
ax30.fill_between(pSA_T, SCM_params_CY14['modelEASZT_w_5'], SCM_params_CY14['modelEASZT_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_ASK22['modelEASZT_w'],'.-', label='A22')
ax30.fill_between(pSA_T, SCM_params_ASK22['modelEASZT_w_5'], SCM_params_ASK22['modelEASZT_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_S22['modelEASZT_w'],'.-', label='S22')
ax30.fill_between(pSA_T, SCM_params_S22['modelEASZT_w_5'], SCM_params_S22['modelEASZT_w_95'], alpha = 0.2)

ax00.set_xlabel('Period, T(s)')
ax01.set_xlabel('Period, T(s)')
ax02.set_xlabel('Period, T(s)')
ax10.set_xlabel('Period, T(s)')
ax11.set_xlabel('Period, T(s)')
ax12.set_xlabel('Period, T(s)')
ax20.set_xlabel('Period, T(s)')
ax21.set_xlabel('Period, T(s)')
ax21.set_xlabel('Period, T(s)')

ax00.set_ylabel(r'$L_E$ (km)')
ax01.set_ylabel(r'$\gamma_E$')
ax02.set_ylabel(r'$L_A\ (^\circ)$')
ax10.set_ylabel(r'$L_S$ (m/s)')
ax11.set_ylabel(r'$L_Z$ (m)')
ax12.set_ylabel(r'$L_T$ (m)')
ax20.set_ylabel(r'$w_S$')
ax21.set_ylabel(r'$w_Z$')
ax22.set_ylabel(r'$w_T$')
ax30.set_ylabel(r'$w$')

ax00.set_xlim(0.01, 6)
ax01.set_xlim(0.01, 6)
ax02.set_xlim(0.01, 6)
ax10.set_xlim(0.01, 6)
ax11.set_xlim(0.01, 6)
ax12.set_xlim(0.01, 6)
ax20.set_xlim(0.01, 6)
ax21.set_xlim(0.01, 6)
ax22.set_xlim(0.01, 6)
ax30.set_xlim(0.01, 6)

ax00.set_ylim(1,500)
ax01.set_ylim(0,1)
ax02.set_ylim(0,45)
ax10.set_ylim(10,5000)
ax11.set_ylim(10,5000)
ax12.set_ylim(1,500)
ax20.set_ylim(0,1)
ax21.set_ylim(0,1)
ax22.set_ylim(0,1)
ax30.set_ylim(0,1)

for ax in [ax00, ax01, ax02, ax10, ax11, ax12, ax20, ax21, ax22, ax30]:
    ax.grid(which='both')

ax01.legend(loc='upper center', ncol=3, fontsize = 9.7)

plt.show()

## 3.2 Subduction GMMs: NGA-Sub

### 3.2.1 Model E

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(12,4),dpi=200, constrained_layout=True)
ax[0].loglog(pSA_T, SCM_params_AG20['modelE_LE'],'.-',label = 'AG20', lw = 1)
ax[0].fill_between(pSA_T, SCM_params_AG20['modelE_LE_5'], SCM_params_AG20['modelE_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_P20['modelE_LE'],'.-',label = 'P20')
ax[0].fill_between(pSA_T, SCM_params_P20['modelE_LE_5'], SCM_params_P20['modelE_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_K20['modelE_LE'],'.-', label='K20')
ax[0].fill_between(pSA_T, SCM_params_K20['modelE_LE_5'], SCM_params_K20['modelE_LE_95'], alpha = 0.2)

ax[1].semilogx(pSA_T, SCM_params_AG20['modelE_gammaE'],'.-', lw = 1)
ax[1].fill_between(pSA_T, SCM_params_AG20['modelE_gammaE_5'], SCM_params_AG20['modelE_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_P20['modelE_gammaE'],'.-', lw = 1)
ax[1].fill_between(pSA_T, SCM_params_P20['modelE_gammaE_5'], SCM_params_P20['modelE_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_K20['modelE_gammaE'],'.-', lw = 1)
ax[1].fill_between(pSA_T, SCM_params_K20['modelE_gammaE_5'], SCM_params_K20['modelE_gammaE_95'], alpha = 0.2)

for i in range(2):
    ax[i].set_xlabel('Period, T(s)')
    ax[i].set_xlim([min(pSA_T),max(pSA_T)])
    ax[i].grid(which = 'both')
    ax[i].set_xlim(0.01,6)
# ax[0].set_yscale('log')
ax[0].set_ylabel(r'$L_E$ (km)')
ax[1].set_ylabel(r'$\gamma_E$')
ax[0].set_ylim([10,500])
ax[1].set_ylim([0,1])
ax[0].legend(loc = 'upper left', ncol = 3)
plt.show()

### 3.2.2 Model EA

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(12,4),dpi=200, constrained_layout=True)
ax[0].loglog(pSA_T, SCM_params_AG20['modelEA_LE'],'.-',label = 'AG20', lw = 1)
ax[0].fill_between(pSA_T, SCM_params_AG20['modelEA_LE_5'], SCM_params_AG20['modelEA_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_P20['modelEA_LE'],'.-',label = 'P20')
ax[0].fill_between(pSA_T, SCM_params_P20['modelEA_LE_5'], SCM_params_P20['modelEA_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_K20['modelEA_LE'],'.-', label='K20')
ax[0].fill_between(pSA_T, SCM_params_K20['modelEA_LE_5'], SCM_params_K20['modelEA_LE_95'], alpha = 0.2)

ax[1].semilogx(pSA_T, SCM_params_AG20['modelEA_gammaE'],'.-',label = 'AG20', lw = 1)
ax[1].fill_between(pSA_T, SCM_params_AG20['modelEA_gammaE_5'], SCM_params_AG20['modelEA_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_P20['modelEA_gammaE'],'.-',label = 'P20')
ax[1].fill_between(pSA_T, SCM_params_P20['modelEA_gammaE_5'], SCM_params_P20['modelEA_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_K20['modelEA_gammaE'],'.-', label='K20')
ax[1].fill_between(pSA_T, SCM_params_K20['modelEA_gammaE_5'], SCM_params_K20['modelEA_gammaE_95'], alpha = 0.2)

ax[2].semilogx(pSA_T, SCM_params_AG20['modelEA_LA'],'.-',label = 'AG20', lw = 1)
ax[2].fill_between(pSA_T, SCM_params_AG20['modelEA_LA_5'], SCM_params_AG20['modelEA_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, SCM_params_P20['modelEA_LA'],'.-',label = 'P20')
ax[2].fill_between(pSA_T, SCM_params_P20['modelEA_LA_5'], SCM_params_P20['modelEA_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, SCM_params_K20['modelEA_LA'],'.-', label='K20')
ax[2].fill_between(pSA_T, SCM_params_K20['modelEA_LA_5'], SCM_params_K20['modelEA_LA_95'], alpha = 0.2)

for i in range(3):
    ax[i].set_xlabel('Period, T(s)')
    ax[i].set_xlim([min(pSA_T),max(pSA_T)])
    ax[i].grid(which = 'both')
    ax[i].set_xlim(0.01,6)
ax[0].set_ylabel(r'$L_E$ (km)')
ax[1].set_ylabel(r'$\gamma_E$')
ax[2].set_ylabel(r'$L_A(\degree)$')
ax[0].set_ylim([10,500])
ax[1].set_ylim([0,1])
ax[2].set_ylim([0,45])
ax[1].legend(loc = 'upper center', ncol = 3)
plt.show()

### 3.2.3 Model EAS

In [ ]:
fig = plt.figure(figsize=(12,6),dpi=200, constrained_layout=True)
gs = fig.add_gridspec(2, 3)
ax00 = fig.add_subplot(gs[0,0])
ax01 = fig.add_subplot(gs[0,1])
ax02 = fig.add_subplot(gs[0,2])
ax10 = fig.add_subplot(gs[1,0])
ax11 = fig.add_subplot(gs[1,1:3])

ax00.loglog(pSA_T, SCM_params_AG20['modelEAS_LE'],'.-',label = 'AG20', lw = 1)
ax00.fill_between(pSA_T, SCM_params_AG20['modelEAS_LE_5'], SCM_params_AG20['modelEAS_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_P20['modelEAS_LE'],'.-',label = 'P20')
ax00.fill_between(pSA_T, SCM_params_P20['modelEAS_LE_5'], SCM_params_P20['modelEAS_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_K20['modelEAS_LE'],'.-', label='K20')
ax00.fill_between(pSA_T, SCM_params_K20['modelEAS_LE_5'], SCM_params_K20['modelEAS_LE_95'], alpha = 0.2)

ax01.semilogx(pSA_T, SCM_params_AG20['modelEAS_gammaE'],'.-',label = 'AG20', lw = 1)
ax01.fill_between(pSA_T, SCM_params_AG20['modelEAS_gammaE_5'], SCM_params_AG20['modelEAS_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_P20['modelEAS_gammaE'],'.-', label = 'P20')
ax01.fill_between(pSA_T, SCM_params_P20['modelEAS_gammaE_5'], SCM_params_P20['modelEAS_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_K20['modelEAS_gammaE'],'.-', label='K20')
ax01.fill_between(pSA_T, SCM_params_K20['modelEAS_gammaE_5'], SCM_params_K20['modelEAS_gammaE_95'], alpha = 0.2)

ax02.semilogx(pSA_T, SCM_params_AG20['modelEAS_LA'],'.-',label = 'AG20', lw = 1)
ax02.fill_between(pSA_T, SCM_params_AG20['modelEAS_LA_5'], SCM_params_AG20['modelEAS_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_P20['modelEAS_LA'],'.-',label = 'P20')
ax02.fill_between(pSA_T, SCM_params_P20['modelEAS_LA_5'], SCM_params_P20['modelEAS_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_K20['modelEAS_LA'],'.-', label='K20')
ax02.fill_between(pSA_T, SCM_params_K20['modelEAS_LA_5'], SCM_params_K20['modelEAS_LA_95'], alpha = 0.2)

ax10.loglog(pSA_T, SCM_params_AG20['modelEAS_LS'],'.-',label = 'AG20', lw = 1)
ax10.fill_between(pSA_T, SCM_params_AG20['modelEAS_LS_5'], SCM_params_AG20['modelEAS_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_P20['modelEAS_LS'],'.-',label = 'P20')
ax10.fill_between(pSA_T, SCM_params_P20['modelEAS_LS_5'], SCM_params_P20['modelEAS_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_K20['modelEAS_LS'],'.-', label='K20')
ax10.fill_between(pSA_T, SCM_params_K20['modelEAS_LS_5'], SCM_params_K20['modelEAS_LS_95'], alpha = 0.2)

ax11.semilogx(pSA_T, SCM_params_AG20['modelEAS_w'],'.-',label = 'AG20', lw = 1)
ax11.fill_between(pSA_T, SCM_params_AG20['modelEAS_w_5'], SCM_params_AG20['modelEAS_w_95'], alpha = 0.2)
ax11.semilogx(pSA_T, SCM_params_P20['modelEAS_w'],'.-',label = 'P20')
ax11.fill_between(pSA_T, SCM_params_P20['modelEAS_w_5'], SCM_params_P20['modelEAS_w_95'], alpha = 0.2)
ax11.semilogx(pSA_T, SCM_params_K20['modelEAS_w'],'.-', label='K20')
ax11.fill_between(pSA_T, SCM_params_K20['modelEAS_w_5'], SCM_params_K20['modelEAS_w_95'], alpha = 0.2)

ax00.set_xlabel('Period, T(s)')
ax01.set_xlabel('Period, T(s)')
ax02.set_xlabel('Period, T(s)')
ax10.set_xlabel('Period, T(s)')
ax11.set_xlabel('Period, T(s)')

ax00.set_ylabel(r'$L_E$ (km)')
ax01.set_ylabel(r'$\gamma_E$')
ax02.set_ylabel(r'$L_A\ (^\circ)$')
ax10.set_ylabel(r'$L_S$ (m/s)')
ax11.set_ylabel(r'$w$')

ax00.set_xlim(0.01, 6)
ax01.set_xlim(0.01, 6)
ax02.set_xlim(0.01, 6)
ax10.set_xlim(0.01, 6)
ax11.set_xlim(0.01, 6)

ax00.set_ylim(10,500)
ax01.set_ylim(0,1)
ax02.set_ylim(0,45)
ax10.set_ylim(10,5000)
ax11.set_ylim(0,1)

for ax in [ax00, ax01, ax02, ax10, ax11]:
    ax.grid(which='both')

ax01.legend(loc='upper center', ncol=3)

plt.show()

### 3.2.4 Model EASZ

In [ ]:
fig = plt.figure(figsize=(12,8),dpi=200, constrained_layout=True)
gs = fig.add_gridspec(3, 3)
ax00 = fig.add_subplot(gs[0,0])
ax01 = fig.add_subplot(gs[0,1])
ax02 = fig.add_subplot(gs[0,2])
ax10 = fig.add_subplot(gs[1,0])
ax11 = fig.add_subplot(gs[1,1])
ax12 = fig.add_subplot(gs[1,2])
ax30 = fig.add_subplot(gs[2,0:3])

ax00.loglog(pSA_T, SCM_params_AG20['modelEASZ_LE'],'.-',label = 'AG20', lw = 1)
ax00.fill_between(pSA_T, SCM_params_AG20['modelEASZ_LE_5'], SCM_params_AG20['modelEASZ_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_P20['modelEASZ_LE'],'.-',label = 'P20')
ax00.fill_between(pSA_T, SCM_params_P20['modelEASZ_LE_5'], SCM_params_P20['modelEASZ_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_K20['modelEASZ_LE'],'.-', label='K20')
ax00.fill_between(pSA_T, SCM_params_K20['modelEASZ_LE_5'], SCM_params_K20['modelEASZ_LE_95'], alpha = 0.2)

ax01.semilogx(pSA_T, SCM_params_AG20['modelEASZ_gammaE'],'.-',label = 'AG20', lw = 1)
ax01.fill_between(pSA_T, SCM_params_AG20['modelEASZ_gammaE_5'], SCM_params_AG20['modelEASZ_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_P20['modelEASZ_gammaE'],'.-',label = 'P20')
ax01.fill_between(pSA_T, SCM_params_P20['modelEASZ_gammaE_5'], SCM_params_P20['modelEASZ_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_K20['modelEASZ_gammaE'],'.-', label='K20')
ax01.fill_between(pSA_T, SCM_params_K20['modelEASZ_gammaE_5'], SCM_params_K20['modelEASZ_gammaE_95'], alpha = 0.2)

ax02.semilogx(pSA_T, SCM_params_AG20['modelEASZ_LA'],'.-',label = 'AG20', lw = 1)
ax02.fill_between(pSA_T, SCM_params_AG20['modelEASZ_LA_5'], SCM_params_AG20['modelEASZ_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_P20['modelEASZ_LA'],'.-',label = 'P20')
ax02.fill_between(pSA_T, SCM_params_P20['modelEASZ_LA_5'], SCM_params_P20['modelEASZ_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_K20['modelEASZ_LA'],'.-', label='K20')
ax02.fill_between(pSA_T, SCM_params_K20['modelEASZ_LA_5'], SCM_params_K20['modelEASZ_LA_95'], alpha = 0.2)

ax10.loglog(pSA_T, SCM_params_AG20['modelEASZ_LS'],'.-',label = 'AG20', lw = 1)
ax10.fill_between(pSA_T, SCM_params_AG20['modelEASZ_LS_5'], SCM_params_AG20['modelEASZ_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_P20['modelEASZ_LS'],'.-',label = 'P20')
ax10.fill_between(pSA_T, SCM_params_P20['modelEASZ_LS_5'], SCM_params_P20['modelEASZ_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_K20['modelEASZ_LS'],'.-', label='K20')
ax10.fill_between(pSA_T, SCM_params_K20['modelEASZ_LS_5'], SCM_params_K20['modelEASZ_LS_95'], alpha = 0.2)

ax11.loglog(pSA_T, SCM_params_AG20['modelEASZ_LZ'],'.-',label = 'AG20', lw = 1)
ax11.fill_between(pSA_T, SCM_params_AG20['modelEASZ_LZ_5'], SCM_params_AG20['modelEASZ_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_P20['modelEASZ_LZ'],'.-',label = 'P20')
ax11.fill_between(pSA_T, SCM_params_P20['modelEASZ_LZ_5'], SCM_params_P20['modelEASZ_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_K20['modelEASZ_LZ'],'.-', label='K20')
ax11.fill_between(pSA_T, SCM_params_K20['modelEASZ_LZ_5'], SCM_params_K20['modelEASZ_LZ_95'], alpha = 0.2)

ax12.semilogx(pSA_T, SCM_params_AG20['modelEASZ_wS'],'.-',label = 'AG20', lw = 1)
ax12.fill_between(pSA_T, SCM_params_AG20['modelEASZ_wS_5'], SCM_params_AG20['modelEASZ_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_P20['modelEASZ_wS'],'.-',label = 'P20')
ax12.fill_between(pSA_T, SCM_params_P20['modelEASZ_wS_5'], SCM_params_P20['modelEASZ_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_K20['modelEASZ_wS'],'.-', label='K20')
ax12.fill_between(pSA_T, SCM_params_K20['modelEASZ_wS_5'], SCM_params_K20['modelEASZ_wS_95'], alpha = 0.2)

ax30.semilogx(pSA_T, SCM_params_AG20['modelEASZ_w'],'.-',label = 'AG20', lw = 1)
ax30.fill_between(pSA_T, SCM_params_AG20['modelEASZ_w_5'], SCM_params_AG20['modelEASZ_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_P20['modelEASZ_w'],'.-',label = 'P20')
ax30.fill_between(pSA_T, SCM_params_P20['modelEASZ_w_5'], SCM_params_P20['modelEASZ_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_K20['modelEASZ_w'],'.-', label='K20')
ax30.fill_between(pSA_T, SCM_params_K20['modelEASZ_w_5'], SCM_params_K20['modelEASZ_w_95'], alpha = 0.2)

ax00.set_xlabel('Period, T(s)')
ax01.set_xlabel('Period, T(s)')
ax02.set_xlabel('Period, T(s)')
ax10.set_xlabel('Period, T(s)')
ax11.set_xlabel('Period, T(s)')
ax12.set_xlabel('Period, T(s)')
ax30.set_xlabel('Period, T(s)')

ax00.set_ylabel(r'$L_E$ (km)')
ax01.set_ylabel(r'$\gamma_E$')
ax02.set_ylabel(r'$L_A\ (^\circ)$')
ax10.set_ylabel(r'$L_S$ (m/s)')
ax11.set_ylabel(r'$L_Z$ (m)')
ax12.set_ylabel(r'$w_S$')
ax30.set_ylabel(r'$w$')

ax00.set_xlim(0.01, 6)
ax01.set_xlim(0.01, 6)
ax02.set_xlim(0.01, 6)
ax10.set_xlim(0.01, 6)
ax11.set_xlim(0.01, 6)
ax12.set_xlim(0.01, 6)
ax30.set_xlim(0.01, 6)

ax00.set_ylim(10,500)
ax01.set_ylim(0,1)
ax02.set_ylim(0,45)
ax10.set_ylim(10,5000)
ax11.set_ylim(10,5000)
ax12.set_ylim(0,1)
ax30.set_ylim(0,1)

for ax in [ax00, ax01, ax02, ax10, ax11, ax12, ax30]:
    ax.grid(which='both')
ax01.legend(loc='upper center', ncol=3, fontsize = 9.7)
plt.show()

### 3.2.5 Model EAST

In [ ]:
fig = plt.figure(figsize=(12,8),dpi=200, constrained_layout=True)
gs = fig.add_gridspec(3, 3)
ax00 = fig.add_subplot(gs[0,0])
ax01 = fig.add_subplot(gs[0,1])
ax02 = fig.add_subplot(gs[0,2])
ax10 = fig.add_subplot(gs[1,0])
ax11 = fig.add_subplot(gs[1,1])
ax12 = fig.add_subplot(gs[1,2])
ax30 = fig.add_subplot(gs[2,0:3])

ax00.loglog(pSA_T, SCM_params_AG20['modelEAST_LE'],'.-',label = 'AG20', lw = 1)
ax00.fill_between(pSA_T, SCM_params_AG20['modelEAST_LE_5'], SCM_params_AG20['modelEAST_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_P20['modelEAST_LE'],'.-',label = 'P20')
ax00.fill_between(pSA_T, SCM_params_P20['modelEAST_LE_5'], SCM_params_P20['modelEAST_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_K20['modelEAST_LE'],'.-', label='K20')
ax00.fill_between(pSA_T, SCM_params_K20['modelEAST_LE_5'], SCM_params_K20['modelEAST_LE_95'], alpha = 0.2)

ax01.semilogx(pSA_T, SCM_params_AG20['modelEAST_gammaE'],'.-',label = 'AG20', lw = 1)
ax01.fill_between(pSA_T, SCM_params_AG20['modelEAST_gammaE_5'], SCM_params_AG20['modelEAST_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_P20['modelEAST_gammaE'],'.-',label = 'P20')
ax01.fill_between(pSA_T, SCM_params_P20['modelEAST_gammaE_5'], SCM_params_P20['modelEAST_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_K20['modelEAST_gammaE'],'.-', label='K20')
ax01.fill_between(pSA_T, SCM_params_K20['modelEAST_gammaE_5'], SCM_params_K20['modelEAST_gammaE_95'], alpha = 0.2)

ax02.semilogx(pSA_T, SCM_params_AG20['modelEAST_LA'],'.-',label = 'AG20', lw = 1)
ax02.fill_between(pSA_T, SCM_params_AG20['modelEAST_LA_5'], SCM_params_AG20['modelEAST_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_P20['modelEAST_LA'],'.-',label = 'P20')
ax02.fill_between(pSA_T, SCM_params_P20['modelEAST_LA_5'], SCM_params_P20['modelEAST_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_K20['modelEAST_LA'],'.-', label='K20')
ax02.fill_between(pSA_T, SCM_params_K20['modelEAST_LA_5'], SCM_params_K20['modelEAST_LA_95'], alpha = 0.2)

ax10.loglog(pSA_T, SCM_params_AG20['modelEAST_LS'],'.-',label = 'AG20', lw = 1)
ax10.fill_between(pSA_T, SCM_params_AG20['modelEAST_LS_5'], SCM_params_AG20['modelEAST_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_P20['modelEAST_LS'],'.-',label = 'P20')
ax10.fill_between(pSA_T, SCM_params_P20['modelEAST_LS_5'], SCM_params_P20['modelEAST_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_K20['modelEAST_LS'],'.-', label='K20')
ax10.fill_between(pSA_T, SCM_params_K20['modelEAST_LS_5'], SCM_params_K20['modelEAST_LS_95'], alpha = 0.2)

ax11.loglog(pSA_T, SCM_params_AG20['modelEAST_LT'],'.-',label = 'AG20', lw = 1)
ax11.fill_between(pSA_T, SCM_params_AG20['modelEAST_LT_5'], SCM_params_AG20['modelEAST_LT_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_P20['modelEAST_LT'],'.-',label = 'P20')
ax11.fill_between(pSA_T, SCM_params_P20['modelEAST_LT_5'], SCM_params_P20['modelEAST_LT_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_K20['modelEAST_LT'],'.-', label='K20')
ax11.fill_between(pSA_T, SCM_params_K20['modelEAST_LT_5'], SCM_params_K20['modelEAST_LT_95'], alpha = 0.2)

ax12.semilogx(pSA_T, SCM_params_AG20['modelEAST_wS'],'.-',label = 'AG20', lw = 1)
ax12.fill_between(pSA_T, SCM_params_AG20['modelEAST_wS_5'], SCM_params_AG20['modelEAST_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_P20['modelEAST_wS'],'.-',label = 'P20')
ax12.fill_between(pSA_T, SCM_params_P20['modelEAST_wS_5'], SCM_params_P20['modelEAST_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_K20['modelEAST_wS'],'.-', label='K20')
ax12.fill_between(pSA_T, SCM_params_K20['modelEAST_wS_5'], SCM_params_K20['modelEAST_wS_95'], alpha = 0.2)

ax30.semilogx(pSA_T, SCM_params_AG20['modelEAST_w'],'.-',label = 'AG20', lw = 1)
ax30.fill_between(pSA_T, SCM_params_AG20['modelEAST_w_5'], SCM_params_AG20['modelEAST_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_P20['modelEAST_w'],'.-',label = 'P20')
ax30.fill_between(pSA_T, SCM_params_P20['modelEAST_w_5'], SCM_params_P20['modelEAST_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_K20['modelEAST_w'],'.-', label='K20')
ax30.fill_between(pSA_T, SCM_params_K20['modelEAST_w_5'], SCM_params_K20['modelEAST_w_95'], alpha = 0.2)

ax00.set_xlabel('Period, T(s)')
ax01.set_xlabel('Period, T(s)')
ax02.set_xlabel('Period, T(s)')
ax10.set_xlabel('Period, T(s)')
ax11.set_xlabel('Period, T(s)')
ax12.set_xlabel('Period, T(s)')
ax30.set_xlabel('Period, T(s)')

ax00.set_ylabel(r'$L_E$ (km)')
ax01.set_ylabel(r'$\gamma_E$')
ax02.set_ylabel(r'$L_A\ (^\circ)$')
ax10.set_ylabel(r'$L_S$ (m/s)')
ax11.set_ylabel(r'$L_T$ (m)')
ax12.set_ylabel(r'$w_S$')
ax30.set_ylabel(r'$w$')

ax00.set_xlim(0.01, 6)
ax01.set_xlim(0.01, 6)
ax02.set_xlim(0.01, 6)
ax10.set_xlim(0.01, 6)
ax11.set_xlim(0.01, 6)
ax12.set_xlim(0.01, 6)
ax30.set_xlim(0.01, 6)

ax00.set_ylim(10,500)
ax01.set_ylim(0,1)
ax02.set_ylim(0,45)
ax10.set_ylim(10,5000)
ax11.set_ylim(1,500)
ax12.set_ylim(0,1)
ax30.set_ylim(0,1)

for ax in [ax00, ax01, ax02, ax10, ax11, ax12, ax30]:
    ax.grid(which='both')

ax01.legend(loc='upper center', ncol=3, fontsize = 9.7)

plt.show()

### 3.2.6 Model EASZT

In [ ]:
fig = plt.figure(figsize=(12,9),dpi=200, constrained_layout=True)
gs = fig.add_gridspec(4, 3)
ax00 = fig.add_subplot(gs[0,0])
ax01 = fig.add_subplot(gs[0,1])
ax02 = fig.add_subplot(gs[0,2])
ax10 = fig.add_subplot(gs[1,0])
ax11 = fig.add_subplot(gs[1,1])
ax12 = fig.add_subplot(gs[1,2])
ax20 = fig.add_subplot(gs[2,0])
ax21 = fig.add_subplot(gs[2,1])
ax22 = fig.add_subplot(gs[2,2])
ax30 = fig.add_subplot(gs[3,0:3])

ax00.loglog(pSA_T, SCM_params_AG20['modelEASZT_LE'],'.-',label = 'AG20', lw = 1)
ax00.fill_between(pSA_T, SCM_params_AG20['modelEASZT_LE_5'], SCM_params_AG20['modelEASZT_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_P20['modelEASZT_LE'],'.-',label = 'P20')
ax00.fill_between(pSA_T, SCM_params_P20['modelEASZT_LE_5'], SCM_params_P20['modelEASZT_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_K20['modelEASZT_LE'],'.-', label='K20')
ax00.fill_between(pSA_T, SCM_params_K20['modelEASZT_LE_5'], SCM_params_K20['modelEASZT_LE_95'], alpha = 0.2)

ax01.semilogx(pSA_T, SCM_params_AG20['modelEASZT_gammaE'],'.-',label = 'AG20', lw = 1)
ax01.fill_between(pSA_T, SCM_params_AG20['modelEASZT_gammaE_5'], SCM_params_AG20['modelEASZT_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_P20['modelEASZT_gammaE'],'.-',label = 'P20')
ax01.fill_between(pSA_T, SCM_params_P20['modelEASZT_gammaE_5'], SCM_params_P20['modelEASZT_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_K20['modelEASZT_gammaE'],'.-', label='K20')
ax01.fill_between(pSA_T, SCM_params_K20['modelEASZT_gammaE_5'], SCM_params_K20['modelEASZT_gammaE_95'], alpha = 0.2)

ax02.semilogx(pSA_T, SCM_params_AG20['modelEASZT_LA'],'.-',label = 'AG20', lw = 1)
ax02.fill_between(pSA_T, SCM_params_AG20['modelEASZT_LA_5'], SCM_params_AG20['modelEASZT_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_P20['modelEASZT_LA'],'.-',label = 'P20')
ax02.fill_between(pSA_T, SCM_params_P20['modelEASZT_LA_5'], SCM_params_P20['modelEASZT_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_K20['modelEASZT_LA'],'.-', label='K20')
ax02.fill_between(pSA_T, SCM_params_K20['modelEASZT_LA_5'], SCM_params_K20['modelEASZT_LA_95'], alpha = 0.2)

ax10.loglog(pSA_T, SCM_params_AG20['modelEASZT_LS'],'.-',label = 'AG20', lw = 1)
ax10.fill_between(pSA_T, SCM_params_AG20['modelEASZT_LS_5'], SCM_params_AG20['modelEASZT_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_P20['modelEASZT_LS'],'.-',label = 'P20')
ax10.fill_between(pSA_T, SCM_params_P20['modelEASZT_LS_5'], SCM_params_P20['modelEASZT_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_K20['modelEASZT_LS'],'.-', label='K20')
ax10.fill_between(pSA_T, SCM_params_K20['modelEASZT_LS_5'], SCM_params_K20['modelEASZT_LS_95'], alpha = 0.2)

ax11.loglog(pSA_T, SCM_params_AG20['modelEASZT_LZ'],'.-',label = 'AG20', lw = 1)
ax11.fill_between(pSA_T, SCM_params_AG20['modelEASZT_LZ_5'], SCM_params_AG20['modelEASZT_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_P20['modelEASZT_LZ'],'.-',label = 'P20')
ax11.fill_between(pSA_T, SCM_params_P20['modelEASZT_LZ_5'], SCM_params_P20['modelEASZT_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_K20['modelEASZT_LZ'],'.-', label='K20')
ax11.fill_between(pSA_T, SCM_params_K20['modelEASZT_LZ_5'], SCM_params_K20['modelEASZT_LZ_95'], alpha = 0.2)

ax12.loglog(pSA_T, SCM_params_AG20['modelEASZT_LT'],'.-',label = 'AG20', lw = 1)
ax12.fill_between(pSA_T, SCM_params_AG20['modelEASZT_LT_5'], SCM_params_AG20['modelEASZT_LT_95'], alpha = 0.2)
ax12.loglog(pSA_T, SCM_params_P20['modelEASZT_LT'],'.-',label = 'P20')
ax12.fill_between(pSA_T, SCM_params_P20['modelEASZT_LT_5'], SCM_params_P20['modelEASZT_LT_95'], alpha = 0.2)
ax12.loglog(pSA_T, SCM_params_K20['modelEASZT_LT'],'.-', label='K20')
ax12.fill_between(pSA_T, SCM_params_K20['modelEASZT_LT_5'], SCM_params_K20['modelEASZT_LT_95'], alpha = 0.2)

ax20.semilogx(pSA_T, SCM_params_AG20['modelEASZT_wS'],'.-',label = 'AG20', lw = 1)
ax20.fill_between(pSA_T, SCM_params_AG20['modelEASZT_wS_5'], SCM_params_AG20['modelEASZT_wS_95'], alpha = 0.2)
ax20.semilogx(pSA_T, SCM_params_P20['modelEASZT_wS'],'.-',label = 'P20')
ax20.fill_between(pSA_T, SCM_params_P20['modelEASZT_wS_5'], SCM_params_P20['modelEASZT_wS_95'], alpha = 0.2)
ax20.semilogx(pSA_T, SCM_params_K20['modelEASZT_wS'],'.-', label='K20')
ax20.fill_between(pSA_T, SCM_params_K20['modelEASZT_wS_5'], SCM_params_K20['modelEASZT_wS_95'], alpha = 0.2)

ax21.semilogx(pSA_T, SCM_params_AG20['modelEASZT_wZ'],'.-',label = 'AG20', lw = 1)
ax21.fill_between(pSA_T, SCM_params_AG20['modelEASZT_wZ_5'], SCM_params_AG20['modelEASZT_wZ_95'], alpha = 0.2)
ax21.semilogx(pSA_T, SCM_params_P20['modelEASZT_wZ'],'.-',label = 'P20')
ax21.fill_between(pSA_T, SCM_params_P20['modelEASZT_wZ_5'], SCM_params_P20['modelEASZT_wZ_95'], alpha = 0.2)
ax21.semilogx(pSA_T, SCM_params_K20['modelEASZT_wZ'],'.-', label='K20')
ax21.fill_between(pSA_T, SCM_params_K20['modelEASZT_wZ_5'], SCM_params_K20['modelEASZT_wZ_95'], alpha = 0.2)

ax22.semilogx(pSA_T, SCM_params_AG20['modelEASZT_wT'],'.-',label = 'AG20', lw = 1)
ax22.fill_between(pSA_T, SCM_params_AG20['modelEASZT_wT_5'], SCM_params_AG20['modelEASZT_wT_95'], alpha = 0.2)
ax22.semilogx(pSA_T, SCM_params_P20['modelEASZT_wT'],'.-',label = 'P20')
ax22.fill_between(pSA_T, SCM_params_P20['modelEASZT_wT_5'], SCM_params_P20['modelEASZT_wT_95'], alpha = 0.2)
ax22.semilogx(pSA_T, SCM_params_K20['modelEASZT_wT'],'.-', label='K20')
ax22.fill_between(pSA_T, SCM_params_K20['modelEASZT_wT_5'], SCM_params_K20['modelEASZT_wT_95'], alpha = 0.2)

ax30.semilogx(pSA_T, SCM_params_AG20['modelEASZT_w'],'.-',label = 'AG20', lw = 1)
ax30.fill_between(pSA_T, SCM_params_AG20['modelEASZT_w_5'], SCM_params_AG20['modelEASZT_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_P20['modelEASZT_w'],'.-',label = 'P20')
ax30.fill_between(pSA_T, SCM_params_P20['modelEASZT_w_5'], SCM_params_P20['modelEASZT_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_K20['modelEASZT_w'],'.-', label='K20')
ax30.fill_between(pSA_T, SCM_params_K20['modelEASZT_w_5'], SCM_params_K20['modelEASZT_w_95'], alpha = 0.2)

ax00.set_xlabel('Period, T(s)')
ax01.set_xlabel('Period, T(s)')
ax02.set_xlabel('Period, T(s)')
ax10.set_xlabel('Period, T(s)')
ax11.set_xlabel('Period, T(s)')
ax12.set_xlabel('Period, T(s)')
ax20.set_xlabel('Period, T(s)')
ax21.set_xlabel('Period, T(s)')
ax21.set_xlabel('Period, T(s)')

ax00.set_ylabel(r'$L_E$ (km)')
ax01.set_ylabel(r'$\gamma_E$')
ax02.set_ylabel(r'$L_A\ (^\circ)$')
ax10.set_ylabel(r'$L_S$ (m/s)')
ax11.set_ylabel(r'$L_Z$ (m)')
ax12.set_ylabel(r'$L_T$ (m)')
ax20.set_ylabel(r'$w_S$')
ax21.set_ylabel(r'$w_Z$')
ax22.set_ylabel(r'$w_T$')
ax30.set_ylabel(r'$w$')

ax00.set_xlim(0.01, 6)
ax01.set_xlim(0.01, 6)
ax02.set_xlim(0.01, 6)
ax10.set_xlim(0.01, 6)
ax11.set_xlim(0.01, 6)
ax12.set_xlim(0.01, 6)
ax20.set_xlim(0.01, 6)
ax21.set_xlim(0.01, 6)
ax22.set_xlim(0.01, 6)
ax30.set_xlim(0.01, 6)

ax00.set_ylim(10,500)
ax01.set_ylim(0,1)
ax02.set_ylim(0,45)
ax10.set_ylim(10,5000)
ax11.set_ylim(10,5000)
ax12.set_ylim(1,500)
ax20.set_ylim(0,1)
ax21.set_ylim(0,1)
ax22.set_ylim(0,1)
ax30.set_ylim(0,1)

for ax in [ax00, ax01, ax02, ax10, ax11, ax12, ax20, ax21, ax22, ax30]:
    ax.grid(which='both')

ax01.legend(loc='upper center', ncol=3, fontsize = 9.7)

plt.show()

## 3.3 Chiou & Young's 2014 w/ various observations

### 3.3.1 Model E

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(12,4),dpi=200, constrained_layout=True)
ax[0].loglog(pSA_T, SCM_params_CY14_n20['modelE_LE'],'.-',label = 'n > 20', lw = 1)
ax[0].fill_between(pSA_T, SCM_params_CY14_n20['modelE_LE_5'], SCM_params_CY14_n20['modelE_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_CY14_n30['modelE_LE'],'.-',label = 'n > 30')
ax[0].fill_between(pSA_T, SCM_params_CY14_n30['modelE_LE_5'], SCM_params_CY14_n30['modelE_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_CY14_n40['modelE_LE'],'.-', label='n > 40')
ax[0].fill_between(pSA_T, SCM_params_CY14_n40['modelE_LE_5'], SCM_params_CY14_n40['modelE_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_CY14_n50['modelE_LE'],'.-', label='n > 50')
ax[0].fill_between(pSA_T, SCM_params_CY14_n50['modelE_LE_5'], SCM_params_CY14_n50['modelE_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_CY14_n60['modelE_LE'],'.-', label='n > 60')
ax[0].fill_between(pSA_T, SCM_params_CY14_n60['modelE_LE_5'], SCM_params_CY14_n60['modelE_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_CY14_n80['modelE_LE'],'.-', label='n > 80')
ax[0].fill_between(pSA_T, SCM_params_CY14_n80['modelE_LE_5'], SCM_params_CY14_n80['modelE_LE_95'], alpha = 0.2)

ax[1].semilogx(pSA_T, SCM_params_CY14_n20['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, SCM_params_CY14_n20['modelE_gammaE_5'], SCM_params_CY14_n20['modelE_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_CY14_n30['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, SCM_params_CY14_n30['modelE_gammaE_5'], SCM_params_CY14_n30['modelE_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_CY14_n40['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, SCM_params_CY14_n40['modelE_gammaE_5'], SCM_params_CY14_n40['modelE_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_CY14_n50['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, SCM_params_CY14_n50['modelE_gammaE_5'], SCM_params_CY14_n50['modelE_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_CY14_n60['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, SCM_params_CY14_n60['modelE_gammaE_5'], SCM_params_CY14_n60['modelE_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_CY14_n80['modelE_gammaE'],'.-')
ax[1].fill_between(pSA_T, SCM_params_CY14_n80['modelE_gammaE_5'], SCM_params_CY14_n80['modelE_gammaE_95'], alpha = 0.2)

for i in range(2):
    ax[i].set_xlabel('Period, T(s)')
    ax[i].set_xlim([min(pSA_T),max(pSA_T)])
    ax[i].grid(which = 'both')
    ax[i].set_xlim(0.01,6)
# ax[0].set_yscale('log')
ax[0].set_ylabel(r'$L_E$ (km)')
ax[1].set_ylabel(r'$\gamma_E$')
ax[0].set_ylim([10,500])
ax[1].set_ylim([0,1])
ax[0].legend(loc = 'upper left', ncol = 3)
plt.show()

### 3.3.2 Model EA

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(12,4),dpi=200, constrained_layout=True)
ax[0].loglog(pSA_T, SCM_params_CY14_n20['modelEA_LE'],'.-',label = 'n > 20', lw = 1)
ax[0].fill_between(pSA_T, SCM_params_CY14_n20['modelEA_LE_5'], SCM_params_CY14_n20['modelEA_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_CY14_n30['modelEA_LE'],'.-',label = 'n > 30')
ax[0].fill_between(pSA_T, SCM_params_CY14_n30['modelEA_LE_5'], SCM_params_CY14_n30['modelEA_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_CY14_n40['modelEA_LE'],'.-', label='n > 40')
ax[0].fill_between(pSA_T, SCM_params_CY14_n40['modelEA_LE_5'], SCM_params_CY14_n40['modelEA_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_CY14_n50['modelEA_LE'],'.-', label='n > 50')
ax[0].fill_between(pSA_T, SCM_params_CY14_n50['modelEA_LE_5'], SCM_params_CY14_n50['modelEA_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_CY14_n60['modelEA_LE'],'.-', label='n > 60')
ax[0].fill_between(pSA_T, SCM_params_CY14_n60['modelEA_LE_5'], SCM_params_CY14_n60['modelEA_LE_95'], alpha = 0.2)
ax[0].loglog(pSA_T, SCM_params_CY14_n80['modelEA_LE'],'.-', label='n > 80')
ax[0].fill_between(pSA_T, SCM_params_CY14_n80['modelEA_LE_5'], SCM_params_CY14_n80['modelEA_LE_95'], alpha = 0.2)

ax[1].semilogx(pSA_T, SCM_params_CY14_n20['modelEA_gammaE'],'.-',label = 'n > 20', lw = 1)
ax[1].fill_between(pSA_T, SCM_params_CY14_n20['modelEA_gammaE_5'], SCM_params_CY14_n20['modelEA_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_CY14_n30['modelEA_gammaE'],'.-',label = 'n > 30')
ax[1].fill_between(pSA_T, SCM_params_CY14_n30['modelEA_gammaE_5'], SCM_params_CY14_n30['modelEA_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_CY14_n40['modelEA_gammaE'],'.-', label='n > 40')
ax[1].fill_between(pSA_T, SCM_params_CY14_n40['modelEA_gammaE_5'], SCM_params_CY14_n40['modelEA_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_CY14_n50['modelEA_gammaE'],'.-', label='n > 50')
ax[1].fill_between(pSA_T, SCM_params_CY14_n50['modelEA_gammaE_5'], SCM_params_CY14_n50['modelEA_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_CY14_n60['modelEA_gammaE'],'.-', label='n > 60')
ax[1].fill_between(pSA_T, SCM_params_CY14_n60['modelEA_gammaE_5'], SCM_params_CY14_n60['modelEA_gammaE_95'], alpha = 0.2)
ax[1].semilogx(pSA_T, SCM_params_CY14_n80['modelEA_gammaE'],'.-', label='n > 80')
ax[1].fill_between(pSA_T, SCM_params_CY14_n80['modelEA_gammaE_5'], SCM_params_CY14_n80['modelEA_gammaE_95'], alpha = 0.2)

ax[2].semilogx(pSA_T, SCM_params_CY14_n20['modelEA_LA'],'.-',label = 'n > 20', lw = 1)
ax[2].fill_between(pSA_T, SCM_params_CY14_n20['modelEA_LA_5'], SCM_params_CY14_n20['modelEA_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, SCM_params_CY14_n30['modelEA_LA'],'.-',label = 'n > 30')
ax[2].fill_between(pSA_T, SCM_params_CY14_n30['modelEA_LA_5'], SCM_params_CY14_n30['modelEA_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, SCM_params_CY14_n40['modelEA_LA'],'.-', label='n > 40')
ax[2].fill_between(pSA_T, SCM_params_CY14_n40['modelEA_LA_5'], SCM_params_CY14_n40['modelEA_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, SCM_params_CY14_n50['modelEA_LA'],'.-', label='n > 50')
ax[2].fill_between(pSA_T, SCM_params_CY14_n50['modelEA_LA_5'], SCM_params_CY14_n50['modelEA_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, SCM_params_CY14_n60['modelEA_LA'],'.-', label='n > 60')
ax[2].fill_between(pSA_T, SCM_params_CY14_n60['modelEA_LA_5'], SCM_params_CY14_n60['modelEA_LA_95'], alpha = 0.2)
ax[2].semilogx(pSA_T, SCM_params_CY14_n80['modelEA_LA'],'.-', label='n > 80')
ax[2].fill_between(pSA_T, SCM_params_CY14_n80['modelEA_LA_5'], SCM_params_CY14_n80['modelEA_LA_95'], alpha = 0.2)

for i in range(3):
    ax[i].set_xlabel('Period, T(s)')
    ax[i].set_xlim([min(pSA_T),max(pSA_T)])
    ax[i].grid(which = 'both')
    ax[i].set_xlim(0.01,6)
ax[0].set_ylabel(r'$L_E$ (km)')
ax[1].set_ylabel(r'$\gamma_E$')
ax[2].set_ylabel(r'$L_A(\degree)$')
ax[0].set_ylim([10,500])
ax[1].set_ylim([0,1])
ax[2].set_ylim([0,45])
ax[1].legend(loc = 'upper center', ncol = 3)
plt.show()

### 3.3.3 Model EAS

In [ ]:
fig = plt.figure(figsize=(12,6),dpi=200, constrained_layout=True)
gs = fig.add_gridspec(2, 3)
ax00 = fig.add_subplot(gs[0,0])
ax01 = fig.add_subplot(gs[0,1])
ax02 = fig.add_subplot(gs[0,2])
ax10 = fig.add_subplot(gs[1,0])
ax11 = fig.add_subplot(gs[1,1:3])

ax00.loglog(pSA_T, SCM_params_CY14_n20['modelEAS_LE'],'.-',label = 'n > 20', lw = 1)
ax00.fill_between(pSA_T, SCM_params_CY14_n20['modelEAS_LE_5'], SCM_params_CY14_n20['modelEAS_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n30['modelEAS_LE'],'.-',label = 'n > 30')
ax00.fill_between(pSA_T, SCM_params_CY14_n30['modelEAS_LE_5'], SCM_params_CY14_n30['modelEAS_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n40['modelEAS_LE'],'.-', label='n > 40')
ax00.fill_between(pSA_T, SCM_params_CY14_n40['modelEAS_LE_5'], SCM_params_CY14_n40['modelEAS_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n50['modelEAS_LE'],'.-', label='n > 50')
ax00.fill_between(pSA_T, SCM_params_CY14_n50['modelEAS_LE_5'], SCM_params_CY14_n50['modelEAS_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n60['modelEAS_LE'],'.-', label='n > 60')
ax00.fill_between(pSA_T, SCM_params_CY14_n60['modelEAS_LE_5'], SCM_params_CY14_n60['modelEAS_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n80['modelEAS_LE'],'.-', label='n > 80')
ax00.fill_between(pSA_T, SCM_params_CY14_n80['modelEAS_LE_5'], SCM_params_CY14_n80['modelEAS_LE_95'], alpha = 0.2)

ax01.semilogx(pSA_T, SCM_params_CY14_n20['modelEAS_gammaE'],'.-',label = 'n > 20', lw = 1)
ax01.fill_between(pSA_T, SCM_params_CY14_n20['modelEAS_gammaE_5'], SCM_params_CY14_n20['modelEAS_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n30['modelEAS_gammaE'],'.-', label = 'n > 30')
ax01.fill_between(pSA_T, SCM_params_CY14_n30['modelEAS_gammaE_5'], SCM_params_CY14_n30['modelEAS_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n40['modelEAS_gammaE'],'.-', label='n > 40')
ax01.fill_between(pSA_T, SCM_params_CY14_n40['modelEAS_gammaE_5'], SCM_params_CY14_n40['modelEAS_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n50['modelEAS_gammaE'],'.-', label='n > 50')
ax01.fill_between(pSA_T, SCM_params_CY14_n50['modelEAS_gammaE_5'], SCM_params_CY14_n50['modelEAS_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n60['modelEAS_gammaE'],'.-', label='n > 60')
ax01.fill_between(pSA_T, SCM_params_CY14_n60['modelEAS_gammaE_5'], SCM_params_CY14_n60['modelEAS_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n80['modelEAS_gammaE'],'.-', label='n > 80')
ax01.fill_between(pSA_T, SCM_params_CY14_n80['modelEAS_gammaE_5'], SCM_params_CY14_n80['modelEAS_gammaE_95'], alpha = 0.2)

ax02.semilogx(pSA_T, SCM_params_CY14_n20['modelEAS_LA'],'.-',label = 'n > 20', lw = 1)
ax02.fill_between(pSA_T, SCM_params_CY14_n20['modelEAS_LA_5'], SCM_params_CY14_n20['modelEAS_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n30['modelEAS_LA'],'.-',label = 'n > 30')
ax02.fill_between(pSA_T, SCM_params_CY14_n30['modelEAS_LA_5'], SCM_params_CY14_n30['modelEAS_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n40['modelEAS_LA'],'.-', label='n > 40')
ax02.fill_between(pSA_T, SCM_params_CY14_n40['modelEAS_LA_5'], SCM_params_CY14_n40['modelEAS_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n50['modelEAS_LA'],'.-', label='n > 50')
ax02.fill_between(pSA_T, SCM_params_CY14_n50['modelEAS_LA_5'], SCM_params_CY14_n50['modelEAS_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n60['modelEAS_LA'],'.-', label='n > 60')
ax02.fill_between(pSA_T, SCM_params_CY14_n60['modelEAS_LA_5'], SCM_params_CY14_n60['modelEAS_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n80['modelEAS_LA'],'.-', label='n > 80')
ax02.fill_between(pSA_T, SCM_params_CY14_n80['modelEAS_LA_5'], SCM_params_CY14_n80['modelEAS_LA_95'], alpha = 0.2)

ax10.loglog(pSA_T, SCM_params_CY14_n20['modelEAS_LS'],'.-',label = 'n > 20', lw = 1)
ax10.fill_between(pSA_T, SCM_params_CY14_n20['modelEAS_LS_5'], SCM_params_CY14_n20['modelEAS_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n30['modelEAS_LS'],'.-',label = 'n > 30')
ax10.fill_between(pSA_T, SCM_params_CY14_n30['modelEAS_LS_5'], SCM_params_CY14_n30['modelEAS_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n40['modelEAS_LS'],'.-', label='n > 40')
ax10.fill_between(pSA_T, SCM_params_CY14_n40['modelEAS_LS_5'], SCM_params_CY14_n40['modelEAS_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n50['modelEAS_LS'],'.-', label='n > 50')
ax10.fill_between(pSA_T, SCM_params_CY14_n50['modelEAS_LS_5'], SCM_params_CY14_n50['modelEAS_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n60['modelEAS_LS'],'.-', label='n > 60')
ax10.fill_between(pSA_T, SCM_params_CY14_n60['modelEAS_LS_5'], SCM_params_CY14_n60['modelEAS_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n80['modelEAS_LS'],'.-', label='n > 80')
ax10.fill_between(pSA_T, SCM_params_CY14_n80['modelEAS_LS_5'], SCM_params_CY14_n80['modelEAS_LS_95'], alpha = 0.2)

ax11.semilogx(pSA_T, SCM_params_CY14_n20['modelEAS_w'],'.-',label = 'n > 20', lw = 1)
ax11.fill_between(pSA_T, SCM_params_CY14_n20['modelEAS_w_5'], SCM_params_CY14_n20['modelEAS_w_95'], alpha = 0.2)
ax11.semilogx(pSA_T, SCM_params_CY14_n30['modelEAS_w'],'.-',label = 'n > 30')
ax11.fill_between(pSA_T, SCM_params_CY14_n30['modelEAS_w_5'], SCM_params_CY14_n30['modelEAS_w_95'], alpha = 0.2)
ax11.semilogx(pSA_T, SCM_params_CY14_n40['modelEAS_w'],'.-', label='n > 40')
ax11.fill_between(pSA_T, SCM_params_CY14_n40['modelEAS_w_5'], SCM_params_CY14_n40['modelEAS_w_95'], alpha = 0.2)
ax11.semilogx(pSA_T, SCM_params_CY14_n50['modelEAS_w'],'.-', label='n > 50')
ax11.fill_between(pSA_T, SCM_params_CY14_n50['modelEAS_w_5'], SCM_params_CY14_n50['modelEAS_w_95'], alpha = 0.2)
ax11.semilogx(pSA_T, SCM_params_CY14_n60['modelEAS_w'],'.-', label='n > 60')
ax11.fill_between(pSA_T, SCM_params_CY14_n60['modelEAS_w_5'], SCM_params_CY14_n60['modelEAS_w_95'], alpha = 0.2)
ax11.semilogx(pSA_T, SCM_params_CY14_n80['modelEAS_w'],'.-', label='n > 80')
ax11.fill_between(pSA_T, SCM_params_CY14_n80['modelEAS_w_5'], SCM_params_CY14_n80['modelEAS_w_95'], alpha = 0.2)

ax00.set_xlabel('Period, T(s)')
ax01.set_xlabel('Period, T(s)')
ax02.set_xlabel('Period, T(s)')
ax10.set_xlabel('Period, T(s)')
ax11.set_xlabel('Period, T(s)')

ax00.set_ylabel(r'$L_E$ (km)')
ax01.set_ylabel(r'$\gamma_E$')
ax02.set_ylabel(r'$L_A\ (^\circ)$')
ax10.set_ylabel(r'$L_S$ (m/s)')
ax11.set_ylabel(r'$w$')

ax00.set_xlim(0.01, 6)
ax01.set_xlim(0.01, 6)
ax02.set_xlim(0.01, 6)
ax10.set_xlim(0.01, 6)
ax11.set_xlim(0.01, 6)

ax00.set_ylim(10,500)
ax01.set_ylim(0,1)
ax02.set_ylim(0,45)
ax10.set_ylim(10,5000)
ax11.set_ylim(0,1)

for ax in [ax00, ax01, ax02, ax10, ax11]:
    ax.grid(which='both')

ax01.legend(loc='upper center', ncol=3)

plt.show()

### 3.3.4 Model EASZ

In [ ]:
fig = plt.figure(figsize=(12,8),dpi=200, constrained_layout=True)
gs = fig.add_gridspec(3, 3)
ax00 = fig.add_subplot(gs[0,0])
ax01 = fig.add_subplot(gs[0,1])
ax02 = fig.add_subplot(gs[0,2])
ax10 = fig.add_subplot(gs[1,0])
ax11 = fig.add_subplot(gs[1,1])
ax12 = fig.add_subplot(gs[1,2])
ax30 = fig.add_subplot(gs[2,0:3])

ax00.loglog(pSA_T, SCM_params_CY14_n20['modelEASZ_LE'],'.-',label = 'n > 20', lw = 1)
ax00.fill_between(pSA_T, SCM_params_CY14_n20['modelEASZ_LE_5'], SCM_params_CY14_n20['modelEASZ_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n30['modelEASZ_LE'],'.-',label = 'n > 30')
ax00.fill_between(pSA_T, SCM_params_CY14_n30['modelEASZ_LE_5'], SCM_params_CY14_n30['modelEASZ_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n40['modelEASZ_LE'],'.-', label='n > 40')
ax00.fill_between(pSA_T, SCM_params_CY14_n40['modelEASZ_LE_5'], SCM_params_CY14_n40['modelEASZ_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n50['modelEASZ_LE'],'.-', label='n > 50')
ax00.fill_between(pSA_T, SCM_params_CY14_n50['modelEASZ_LE_5'], SCM_params_CY14_n50['modelEASZ_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n60['modelEASZ_LE'],'.-', label='n > 60')
ax00.fill_between(pSA_T, SCM_params_CY14_n60['modelEASZ_LE_5'], SCM_params_CY14_n60['modelEASZ_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n80['modelEASZ_LE'],'.-', label='n > 80')
ax00.fill_between(pSA_T, SCM_params_CY14_n80['modelEASZ_LE_5'], SCM_params_CY14_n80['modelEASZ_LE_95'], alpha = 0.2)

ax01.semilogx(pSA_T, SCM_params_CY14_n20['modelEASZ_gammaE'],'.-',label = 'n > 20', lw = 1)
ax01.fill_between(pSA_T, SCM_params_CY14_n20['modelEASZ_gammaE_5'], SCM_params_CY14_n20['modelEASZ_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n30['modelEASZ_gammaE'],'.-',label = 'n > 30')
ax01.fill_between(pSA_T, SCM_params_CY14_n30['modelEASZ_gammaE_5'], SCM_params_CY14_n30['modelEASZ_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n40['modelEASZ_gammaE'],'.-', label='n > 40')
ax01.fill_between(pSA_T, SCM_params_CY14_n40['modelEASZ_gammaE_5'], SCM_params_CY14_n40['modelEASZ_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n50['modelEASZ_gammaE'],'.-', label='n > 50')
ax01.fill_between(pSA_T, SCM_params_CY14_n50['modelEASZ_gammaE_5'], SCM_params_CY14_n50['modelEASZ_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n60['modelEASZ_gammaE'],'.-', label='n > 60')
ax01.fill_between(pSA_T, SCM_params_CY14_n60['modelEASZ_gammaE_5'], SCM_params_CY14_n60['modelEASZ_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n80['modelEASZ_gammaE'],'.-', label='n > 80')
ax01.fill_between(pSA_T, SCM_params_CY14_n80['modelEASZ_gammaE_5'], SCM_params_CY14_n80['modelEASZ_gammaE_95'], alpha = 0.2)

ax02.semilogx(pSA_T, SCM_params_CY14_n20['modelEASZ_LA'],'.-',label = 'n > 20', lw = 1)
ax02.fill_between(pSA_T, SCM_params_CY14_n20['modelEASZ_LA_5'], SCM_params_CY14_n20['modelEASZ_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n30['modelEASZ_LA'],'.-',label = 'n > 30')
ax02.fill_between(pSA_T, SCM_params_CY14_n30['modelEASZ_LA_5'], SCM_params_CY14_n30['modelEASZ_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n40['modelEASZ_LA'],'.-', label='n > 40')
ax02.fill_between(pSA_T, SCM_params_CY14_n40['modelEASZ_LA_5'], SCM_params_CY14_n40['modelEASZ_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n50['modelEASZ_LA'],'.-', label='n > 50')
ax02.fill_between(pSA_T, SCM_params_CY14_n50['modelEASZ_LA_5'], SCM_params_CY14_n50['modelEASZ_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n60['modelEASZ_LA'],'.-', label='n > 60')
ax02.fill_between(pSA_T, SCM_params_CY14_n60['modelEASZ_LA_5'], SCM_params_CY14_n60['modelEASZ_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n80['modelEASZ_LA'],'.-', label='n > 80')
ax02.fill_between(pSA_T, SCM_params_CY14_n80['modelEASZ_LA_5'], SCM_params_CY14_n80['modelEASZ_LA_95'], alpha = 0.2)

ax10.loglog(pSA_T, SCM_params_CY14_n20['modelEASZ_LS'],'.-',label = 'n > 20', lw = 1)
ax10.fill_between(pSA_T, SCM_params_CY14_n20['modelEASZ_LS_5'], SCM_params_CY14_n20['modelEASZ_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n30['modelEASZ_LS'],'.-',label = 'n > 30')
ax10.fill_between(pSA_T, SCM_params_CY14_n30['modelEASZ_LS_5'], SCM_params_CY14_n30['modelEASZ_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n40['modelEASZ_LS'],'.-', label='n > 40')
ax10.fill_between(pSA_T, SCM_params_CY14_n40['modelEASZ_LS_5'], SCM_params_CY14_n40['modelEASZ_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n50['modelEASZ_LS'],'.-', label='n > 50')
ax10.fill_between(pSA_T, SCM_params_CY14_n50['modelEASZ_LS_5'], SCM_params_CY14_n50['modelEASZ_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n60['modelEASZ_LS'],'.-', label='n > 60')
ax10.fill_between(pSA_T, SCM_params_CY14_n60['modelEASZ_LS_5'], SCM_params_CY14_n60['modelEASZ_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n80['modelEASZ_LS'],'.-', label='n > 80')
ax10.fill_between(pSA_T, SCM_params_CY14_n80['modelEASZ_LS_5'], SCM_params_CY14_n80['modelEASZ_LS_95'], alpha = 0.2)

ax11.loglog(pSA_T, SCM_params_CY14_n20['modelEASZ_LZ'],'.-',label = 'n > 20', lw = 1)
ax11.fill_between(pSA_T, SCM_params_CY14_n20['modelEASZ_LZ_5'], SCM_params_CY14_n20['modelEASZ_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CY14_n30['modelEASZ_LZ'],'.-',label = 'n > 30')
ax11.fill_between(pSA_T, SCM_params_CY14_n30['modelEASZ_LZ_5'], SCM_params_CY14_n30['modelEASZ_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CY14_n40['modelEASZ_LZ'],'.-', label='n > 40')
ax11.fill_between(pSA_T, SCM_params_CY14_n40['modelEASZ_LZ_5'], SCM_params_CY14_n40['modelEASZ_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CY14_n50['modelEASZ_LZ'],'.-', label='n > 50')
ax11.fill_between(pSA_T, SCM_params_CY14_n50['modelEASZ_LZ_5'], SCM_params_CY14_n50['modelEASZ_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CY14_n60['modelEASZ_LZ'],'.-', label='n > 60')
ax11.fill_between(pSA_T, SCM_params_CY14_n60['modelEASZ_LZ_5'], SCM_params_CY14_n60['modelEASZ_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CY14_n80['modelEASZ_LZ'],'.-', label='n > 80')
ax11.fill_between(pSA_T, SCM_params_CY14_n80['modelEASZ_LZ_5'], SCM_params_CY14_n80['modelEASZ_LZ_95'], alpha = 0.2)

ax12.semilogx(pSA_T, SCM_params_CY14_n20['modelEASZ_wS'],'.-',label = 'n > 20', lw = 1)
ax12.fill_between(pSA_T, SCM_params_CY14_n20['modelEASZ_wS_5'], SCM_params_CY14_n20['modelEASZ_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_CY14_n30['modelEASZ_wS'],'.-',label = 'n > 30')
ax12.fill_between(pSA_T, SCM_params_CY14_n30['modelEASZ_wS_5'], SCM_params_CY14_n30['modelEASZ_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_CY14_n40['modelEASZ_wS'],'.-', label='n > 40')
ax12.fill_between(pSA_T, SCM_params_CY14_n40['modelEASZ_wS_5'], SCM_params_CY14_n40['modelEASZ_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_CY14_n50['modelEASZ_wS'],'.-', label='n > 50')
ax12.fill_between(pSA_T, SCM_params_CY14_n50['modelEASZ_wS_5'], SCM_params_CY14_n50['modelEASZ_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_CY14_n60['modelEASZ_wS'],'.-', label='n > 60')
ax12.fill_between(pSA_T, SCM_params_CY14_n60['modelEASZ_wS_5'], SCM_params_CY14_n60['modelEASZ_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_CY14_n80['modelEASZ_wS'],'.-', label='n > 80')
ax12.fill_between(pSA_T, SCM_params_CY14_n80['modelEASZ_wS_5'], SCM_params_CY14_n80['modelEASZ_wS_95'], alpha = 0.2)

ax30.semilogx(pSA_T, SCM_params_CY14_n20['modelEASZ_w'],'.-',label = 'n > 20', lw = 1)
ax30.fill_between(pSA_T, SCM_params_CY14_n20['modelEASZ_w_5'], SCM_params_CY14_n20['modelEASZ_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CY14_n30['modelEASZ_w'],'.-',label = 'n > 30')
ax30.fill_between(pSA_T, SCM_params_CY14_n30['modelEASZ_w_5'], SCM_params_CY14_n30['modelEASZ_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CY14_n40['modelEASZ_w'],'.-', label='n > 40')
ax30.fill_between(pSA_T, SCM_params_CY14_n40['modelEASZ_w_5'], SCM_params_CY14_n40['modelEASZ_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CY14_n50['modelEASZ_w'],'.-', label='n > 50')
ax30.fill_between(pSA_T, SCM_params_CY14_n50['modelEASZ_w_5'], SCM_params_CY14_n50['modelEASZ_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CY14_n60['modelEASZ_w'],'.-', label='n > 60')
ax30.fill_between(pSA_T, SCM_params_CY14_n60['modelEASZ_w_5'], SCM_params_CY14_n60['modelEASZ_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CY14_n80['modelEASZ_w'],'.-', label='n > 80')
ax30.fill_between(pSA_T, SCM_params_CY14_n80['modelEASZ_w_5'], SCM_params_CY14_n80['modelEASZ_w_95'], alpha = 0.2)

ax00.set_xlabel('Period, T(s)')
ax01.set_xlabel('Period, T(s)')
ax02.set_xlabel('Period, T(s)')
ax10.set_xlabel('Period, T(s)')
ax11.set_xlabel('Period, T(s)')
ax12.set_xlabel('Period, T(s)')
ax30.set_xlabel('Period, T(s)')

ax00.set_ylabel(r'$L_E$ (km)')
ax01.set_ylabel(r'$\gamma_E$')
ax02.set_ylabel(r'$L_A\ (^\circ)$')
ax10.set_ylabel(r'$L_S$ (m/s)')
ax11.set_ylabel(r'$L_Z$ (m)')
ax12.set_ylabel(r'$w_S$')
ax30.set_ylabel(r'$w$')

ax00.set_xlim(0.01, 6)
ax01.set_xlim(0.01, 6)
ax02.set_xlim(0.01, 6)
ax10.set_xlim(0.01, 6)
ax11.set_xlim(0.01, 6)
ax12.set_xlim(0.01, 6)
ax30.set_xlim(0.01, 6)

ax00.set_ylim(10,500)
ax01.set_ylim(0,1)
ax02.set_ylim(0,45)
ax10.set_ylim(10,5000)
ax11.set_ylim(10,5000)
ax12.set_ylim(0,1)
ax30.set_ylim(0,1)

for ax in [ax00, ax01, ax02, ax10, ax11, ax12, ax30]:
    ax.grid(which='both')

ax01.legend(loc='upper center', ncol=3, fontsize = 9.7)

plt.show()

### 3.3.5 Model EAST

In [ ]:
fig = plt.figure(figsize=(12,8),dpi=200, constrained_layout=True)
gs = fig.add_gridspec(3, 3)
ax00 = fig.add_subplot(gs[0,0])
ax01 = fig.add_subplot(gs[0,1])
ax02 = fig.add_subplot(gs[0,2])
ax10 = fig.add_subplot(gs[1,0])
ax11 = fig.add_subplot(gs[1,1])
ax12 = fig.add_subplot(gs[1,2])
ax30 = fig.add_subplot(gs[2,0:3])

ax00.loglog(pSA_T, SCM_params_CY14_n20['modelEAST_LE'],'.-',label = 'n > 20', lw = 1)
ax00.fill_between(pSA_T, SCM_params_CY14_n20['modelEAST_LE_5'], SCM_params_CY14_n20['modelEAST_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n30['modelEAST_LE'],'.-',label = 'n > 30')
ax00.fill_between(pSA_T, SCM_params_CY14_n30['modelEAST_LE_5'], SCM_params_CY14_n30['modelEAST_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n40['modelEAST_LE'],'.-', label='n > 40')
ax00.fill_between(pSA_T, SCM_params_CY14_n40['modelEAST_LE_5'], SCM_params_CY14_n40['modelEAST_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n50['modelEAST_LE'],'.-', label='n > 50')
ax00.fill_between(pSA_T, SCM_params_CY14_n50['modelEAST_LE_5'], SCM_params_CY14_n50['modelEAST_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n60['modelEAST_LE'],'.-', label='n > 60')
ax00.fill_between(pSA_T, SCM_params_CY14_n60['modelEAST_LE_5'], SCM_params_CY14_n60['modelEAST_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n80['modelEAST_LE'],'.-', label='n > 80')
ax00.fill_between(pSA_T, SCM_params_CY14_n80['modelEAST_LE_5'], SCM_params_CY14_n80['modelEAST_LE_95'], alpha = 0.2)

ax01.semilogx(pSA_T, SCM_params_CY14_n20['modelEAST_gammaE'],'.-',label = 'n > 20', lw = 1)
ax01.fill_between(pSA_T, SCM_params_CY14_n20['modelEAST_gammaE_5'], SCM_params_CY14_n20['modelEAST_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n30['modelEAST_gammaE'],'.-',label = 'n > 30')
ax01.fill_between(pSA_T, SCM_params_CY14_n30['modelEAST_gammaE_5'], SCM_params_CY14_n30['modelEAST_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n40['modelEAST_gammaE'],'.-', label='n > 40')
ax01.fill_between(pSA_T, SCM_params_CY14_n40['modelEAST_gammaE_5'], SCM_params_CY14_n40['modelEAST_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n50['modelEAST_gammaE'],'.-', label='n > 50')
ax01.fill_between(pSA_T, SCM_params_CY14_n50['modelEAST_gammaE_5'], SCM_params_CY14_n50['modelEAST_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n60['modelEAST_gammaE'],'.-', label='n > 60')
ax01.fill_between(pSA_T, SCM_params_CY14_n60['modelEAST_gammaE_5'], SCM_params_CY14_n60['modelEAST_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n80['modelEAST_gammaE'],'.-', label='n > 80')
ax01.fill_between(pSA_T, SCM_params_CY14_n80['modelEAST_gammaE_5'], SCM_params_CY14_n80['modelEAST_gammaE_95'], alpha = 0.2)

ax02.semilogx(pSA_T, SCM_params_CY14_n20['modelEAST_LA'],'.-',label = 'n > 20', lw = 1)
ax02.fill_between(pSA_T, SCM_params_CY14_n20['modelEAST_LA_5'], SCM_params_CY14_n20['modelEAST_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n30['modelEAST_LA'],'.-',label = 'n > 30')
ax02.fill_between(pSA_T, SCM_params_CY14_n30['modelEAST_LA_5'], SCM_params_CY14_n30['modelEAST_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n40['modelEAST_LA'],'.-', label='n > 40')
ax02.fill_between(pSA_T, SCM_params_CY14_n40['modelEAST_LA_5'], SCM_params_CY14_n40['modelEAST_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n50['modelEAST_LA'],'.-', label='n > 50')
ax02.fill_between(pSA_T, SCM_params_CY14_n50['modelEAST_LA_5'], SCM_params_CY14_n50['modelEAST_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n60['modelEAST_LA'],'.-', label='n > 60')
ax02.fill_between(pSA_T, SCM_params_CY14_n60['modelEAST_LA_5'], SCM_params_CY14_n60['modelEAST_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n80['modelEAST_LA'],'.-', label='n > 80')
ax02.fill_between(pSA_T, SCM_params_CY14_n80['modelEAST_LA_5'], SCM_params_CY14_n80['modelEAST_LA_95'], alpha = 0.2)

ax10.loglog(pSA_T, SCM_params_CY14_n20['modelEAST_LS'],'.-',label = 'n > 20', lw = 1)
ax10.fill_between(pSA_T, SCM_params_CY14_n20['modelEAST_LS_5'], SCM_params_CY14_n20['modelEAST_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n30['modelEAST_LS'],'.-',label = 'n > 30')
ax10.fill_between(pSA_T, SCM_params_CY14_n30['modelEAST_LS_5'], SCM_params_CY14_n30['modelEAST_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n40['modelEAST_LS'],'.-', label='n > 40')
ax10.fill_between(pSA_T, SCM_params_CY14_n40['modelEAST_LS_5'], SCM_params_CY14_n40['modelEAST_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n50['modelEAST_LS'],'.-', label='n > 50')
ax10.fill_between(pSA_T, SCM_params_CY14_n50['modelEAST_LS_5'], SCM_params_CY14_n50['modelEAST_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n60['modelEAST_LS'],'.-', label='n > 60')
ax10.fill_between(pSA_T, SCM_params_CY14_n60['modelEAST_LS_5'], SCM_params_CY14_n60['modelEAST_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n80['modelEAST_LS'],'.-', label='n > 80')
ax10.fill_between(pSA_T, SCM_params_CY14_n80['modelEAST_LS_5'], SCM_params_CY14_n80['modelEAST_LS_95'], alpha = 0.2)

ax11.loglog(pSA_T, SCM_params_CY14_n20['modelEAST_LT'],'.-',label = 'n > 20', lw = 1)
ax11.fill_between(pSA_T, SCM_params_CY14_n20['modelEAST_LT_5'], SCM_params_CY14_n20['modelEAST_LT_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CY14_n30['modelEAST_LT'],'.-',label = 'n > 30')
ax11.fill_between(pSA_T, SCM_params_CY14_n30['modelEAST_LT_5'], SCM_params_CY14_n30['modelEAST_LT_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CY14_n40['modelEAST_LT'],'.-', label='n > 40')
ax11.fill_between(pSA_T, SCM_params_CY14_n40['modelEAST_LT_5'], SCM_params_CY14_n40['modelEAST_LT_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CY14_n50['modelEAST_LT'],'.-', label='n > 50')
ax11.fill_between(pSA_T, SCM_params_CY14_n50['modelEAST_LT_5'], SCM_params_CY14_n50['modelEAST_LT_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CY14_n60['modelEAST_LT'],'.-', label='n > 60')
ax11.fill_between(pSA_T, SCM_params_CY14_n60['modelEAST_LT_5'], SCM_params_CY14_n60['modelEAST_LT_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CY14_n80['modelEAST_LT'],'.-', label='n > 80')
ax11.fill_between(pSA_T, SCM_params_CY14_n80['modelEAST_LT_5'], SCM_params_CY14_n80['modelEAST_LT_95'], alpha = 0.2)

ax12.semilogx(pSA_T, SCM_params_CY14_n20['modelEAST_wS'],'.-',label = 'n > 20', lw = 1)
ax12.fill_between(pSA_T, SCM_params_CY14_n20['modelEAST_wS_5'], SCM_params_CY14_n20['modelEAST_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_CY14_n30['modelEAST_wS'],'.-',label = 'n > 30')
ax12.fill_between(pSA_T, SCM_params_CY14_n30['modelEAST_wS_5'], SCM_params_CY14_n30['modelEAST_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_CY14_n40['modelEAST_wS'],'.-', label='n > 40')
ax12.fill_between(pSA_T, SCM_params_CY14_n40['modelEAST_wS_5'], SCM_params_CY14_n40['modelEAST_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_CY14_n50['modelEAST_wS'],'.-', label='n > 50')
ax12.fill_between(pSA_T, SCM_params_CY14_n50['modelEAST_wS_5'], SCM_params_CY14_n50['modelEAST_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_CY14_n60['modelEAST_wS'],'.-', label='n > 60')
ax12.fill_between(pSA_T, SCM_params_CY14_n60['modelEAST_wS_5'], SCM_params_CY14_n60['modelEAST_wS_95'], alpha = 0.2)
ax12.semilogx(pSA_T, SCM_params_CY14_n80['modelEAST_wS'],'.-', label='n > 80')
ax12.fill_between(pSA_T, SCM_params_CY14_n80['modelEAST_wS_5'], SCM_params_CY14_n80['modelEAST_wS_95'], alpha = 0.2)

ax30.semilogx(pSA_T, SCM_params_CY14_n20['modelEAST_w'],'.-',label = 'n > 20', lw = 1)
ax30.fill_between(pSA_T, SCM_params_CY14_n20['modelEAST_w_5'], SCM_params_CY14_n20['modelEAST_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CY14_n30['modelEAST_w'],'.-',label = 'n > 30')
ax30.fill_between(pSA_T, SCM_params_CY14_n30['modelEAST_w_5'], SCM_params_CY14_n30['modelEAST_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CY14_n40['modelEAST_w'],'.-', label='n > 40')
ax30.fill_between(pSA_T, SCM_params_CY14_n40['modelEAST_w_5'], SCM_params_CY14_n40['modelEAST_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CY14_n50['modelEAST_w'],'.-', label='n > 50')
ax30.fill_between(pSA_T, SCM_params_CY14_n50['modelEAST_w_5'], SCM_params_CY14_n50['modelEAST_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CY14_n60['modelEAST_w'],'.-', label='n > 60')
ax30.fill_between(pSA_T, SCM_params_CY14_n60['modelEAST_w_5'], SCM_params_CY14_n60['modelEAST_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CY14_n80['modelEAST_w'],'.-', label='n > 80')
ax30.fill_between(pSA_T, SCM_params_CY14_n80['modelEAST_w_5'], SCM_params_CY14_n80['modelEAST_w_95'], alpha = 0.2)

ax00.set_xlabel('Period, T(s)')
ax01.set_xlabel('Period, T(s)')
ax02.set_xlabel('Period, T(s)')
ax10.set_xlabel('Period, T(s)')
ax11.set_xlabel('Period, T(s)')
ax12.set_xlabel('Period, T(s)')
ax30.set_xlabel('Period, T(s)')

ax00.set_ylabel(r'$L_E$ (km)')
ax01.set_ylabel(r'$\gamma_E$')
ax02.set_ylabel(r'$L_A\ (^\circ)$')
ax10.set_ylabel(r'$L_S$ (m/s)')
ax11.set_ylabel(r'$L_T$ (m)')
ax12.set_ylabel(r'$w_S$')
ax30.set_ylabel(r'$w$')

ax00.set_xlim(0.01, 6)
ax01.set_xlim(0.01, 6)
ax02.set_xlim(0.01, 6)
ax10.set_xlim(0.01, 6)
ax11.set_xlim(0.01, 6)
ax12.set_xlim(0.01, 6)
ax30.set_xlim(0.01, 6)

ax00.set_ylim(10,500)
ax01.set_ylim(0,1)
ax02.set_ylim(0,45)
ax10.set_ylim(10,5000)
ax11.set_ylim(1,500)
ax12.set_ylim(0,1)
ax30.set_ylim(0,1)

for ax in [ax00, ax01, ax02, ax10, ax11, ax12, ax30]:
    ax.grid(which='both')

ax01.legend(loc='upper center', ncol=3, fontsize = 9.7)

plt.show()

### 3.3.6 Model EASZT

In [ ]:
fig = plt.figure(figsize=(12,9),dpi=200, constrained_layout=True)
gs = fig.add_gridspec(4, 3)
ax00 = fig.add_subplot(gs[0,0])
ax01 = fig.add_subplot(gs[0,1])
ax02 = fig.add_subplot(gs[0,2])
ax10 = fig.add_subplot(gs[1,0])
ax11 = fig.add_subplot(gs[1,1])
ax12 = fig.add_subplot(gs[1,2])
ax20 = fig.add_subplot(gs[2,0])
ax21 = fig.add_subplot(gs[2,1])
ax22 = fig.add_subplot(gs[2,2])
ax30 = fig.add_subplot(gs[3,0:3])

ax00.loglog(pSA_T, SCM_params_CY14_n20['modelEASZT_LE'],'.-',label = 'n > 20', lw = 1)
ax00.fill_between(pSA_T, SCM_params_CY14_n20['modelEASZT_LE_5'], SCM_params_CY14_n20['modelEASZT_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n30['modelEASZT_LE'],'.-',label = 'n > 30')
ax00.fill_between(pSA_T, SCM_params_CY14_n30['modelEASZT_LE_5'], SCM_params_CY14_n30['modelEASZT_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n40['modelEASZT_LE'],'.-', label='n > 40')
ax00.fill_between(pSA_T, SCM_params_CY14_n40['modelEASZT_LE_5'], SCM_params_CY14_n40['modelEASZT_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n50['modelEASZT_LE'],'.-', label='n > 50')
ax00.fill_between(pSA_T, SCM_params_CY14_n50['modelEASZT_LE_5'], SCM_params_CY14_n50['modelEASZT_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n60['modelEASZT_LE'],'.-', label='n > 60')
ax00.fill_between(pSA_T, SCM_params_CY14_n60['modelEASZT_LE_5'], SCM_params_CY14_n60['modelEASZT_LE_95'], alpha = 0.2)
ax00.loglog(pSA_T, SCM_params_CY14_n80['modelEASZT_LE'],'.-', label='n > 80')
ax00.fill_between(pSA_T, SCM_params_CY14_n80['modelEASZT_LE_5'], SCM_params_CY14_n80['modelEASZT_LE_95'], alpha = 0.2)

ax01.semilogx(pSA_T, SCM_params_CY14_n20['modelEASZT_gammaE'],'.-',label = 'n > 20', lw = 1)
ax01.fill_between(pSA_T, SCM_params_CY14_n20['modelEASZT_gammaE_5'], SCM_params_CY14_n20['modelEASZT_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n30['modelEASZT_gammaE'],'.-',label = 'n > 30')
ax01.fill_between(pSA_T, SCM_params_CY14_n30['modelEASZT_gammaE_5'], SCM_params_CY14_n30['modelEASZT_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n40['modelEASZT_gammaE'],'.-', label='n > 40')
ax01.fill_between(pSA_T, SCM_params_CY14_n40['modelEASZT_gammaE_5'], SCM_params_CY14_n40['modelEASZT_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n50['modelEASZT_gammaE'],'.-', label='n > 50')
ax01.fill_between(pSA_T, SCM_params_CY14_n50['modelEASZT_gammaE_5'], SCM_params_CY14_n50['modelEASZT_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n60['modelEAST_gammaE'],'.-', label='n > 60')
ax01.fill_between(pSA_T, SCM_params_CY14_n60['modelEASZT_gammaE_5'], SCM_params_CY14_n60['modelEASZT_gammaE_95'], alpha = 0.2)
ax01.semilogx(pSA_T, SCM_params_CY14_n80['modelEASZT_gammaE'],'.-', label='n > 80')
ax01.fill_between(pSA_T, SCM_params_CY14_n80['modelEASZT_gammaE_5'], SCM_params_CY14_n80['modelEASZT_gammaE_95'], alpha = 0.2)

ax02.semilogx(pSA_T, SCM_params_CY14_n20['modelEASZT_LA'],'.-',label = 'n > 20', lw = 1)
ax02.fill_between(pSA_T, SCM_params_CY14_n20['modelEASZT_LA_5'], SCM_params_CY14_n20['modelEASZT_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n30['modelEASZT_LA'],'.-',label = 'n > 30')
ax02.fill_between(pSA_T, SCM_params_CY14_n30['modelEASZT_LA_5'], SCM_params_CY14_n30['modelEASZT_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n40['modelEASZT_LA'],'.-', label='n > 40')
ax02.fill_between(pSA_T, SCM_params_CY14_n40['modelEASZT_LA_5'], SCM_params_CY14_n40['modelEASZT_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n50['modelEASZT_LA'],'.-', label='n > 50')
ax02.fill_between(pSA_T, SCM_params_CY14_n50['modelEASZT_LA_5'], SCM_params_CY14_n50['modelEASZT_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n60['modelEASZT_LA'],'.-', label='n > 60')
ax02.fill_between(pSA_T, SCM_params_CY14_n60['modelEASZT_LA_5'], SCM_params_CY14_n60['modelEASZT_LA_95'], alpha = 0.2)
ax02.semilogx(pSA_T, SCM_params_CY14_n80['modelEASZT_LA'],'.-', label='n > 80')
ax02.fill_between(pSA_T, SCM_params_CY14_n80['modelEASZT_LA_5'], SCM_params_CY14_n80['modelEASZT_LA_95'], alpha = 0.2)

ax10.loglog(pSA_T, SCM_params_CY14_n20['modelEASZT_LS'],'.-',label = 'n > 20', lw = 1)
ax10.fill_between(pSA_T, SCM_params_CY14_n20['modelEASZT_LS_5'], SCM_params_CY14_n20['modelEASZT_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n30['modelEASZT_LS'],'.-',label = 'n > 30')
ax10.fill_between(pSA_T, SCM_params_CY14_n30['modelEASZT_LS_5'], SCM_params_CY14_n30['modelEASZT_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n40['modelEASZT_LS'],'.-', label='n > 40')
ax10.fill_between(pSA_T, SCM_params_CY14_n40['modelEASZT_LS_5'], SCM_params_CY14_n40['modelEASZT_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n50['modelEASZT_LS'],'.-', label='n > 50')
ax10.fill_between(pSA_T, SCM_params_CY14_n50['modelEASZT_LS_5'], SCM_params_CY14_n50['modelEASZT_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n60['modelEASZT_LS'],'.-', label='n > 60')
ax10.fill_between(pSA_T, SCM_params_CY14_n60['modelEASZT_LS_5'], SCM_params_CY14_n60['modelEASZT_LS_95'], alpha = 0.2)
ax10.loglog(pSA_T, SCM_params_CY14_n80['modelEASZT_LS'],'.-', label='n > 80')
ax10.fill_between(pSA_T, SCM_params_CY14_n80['modelEASZT_LS_5'], SCM_params_CY14_n80['modelEASZT_LS_95'], alpha = 0.2)

ax11.loglog(pSA_T, SCM_params_CY14_n20['modelEASZT_LZ'],'.-',label = 'n > 20', lw = 1)
ax11.fill_between(pSA_T, SCM_params_CY14_n20['modelEASZT_LZ_5'], SCM_params_CY14_n20['modelEASZT_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CY14_n30['modelEASZT_LZ'],'.-',label = 'n > 30')
ax11.fill_between(pSA_T, SCM_params_CY14_n30['modelEASZT_LZ_5'], SCM_params_CY14_n30['modelEASZT_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CY14_n40['modelEASZT_LZ'],'.-', label='n > 40')
ax11.fill_between(pSA_T, SCM_params_CY14_n40['modelEASZT_LZ_5'], SCM_params_CY14_n40['modelEASZT_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CY14_n50['modelEASZT_LZ'],'.-', label='n > 50')
ax11.fill_between(pSA_T, SCM_params_CY14_n50['modelEASZT_LZ_5'], SCM_params_CY14_n50['modelEASZT_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CY14_n60['modelEASZT_LZ'],'.-', label='n > 60')
ax11.fill_between(pSA_T, SCM_params_CY14_n60['modelEASZT_LZ_5'], SCM_params_CY14_n60['modelEASZT_LZ_95'], alpha = 0.2)
ax11.loglog(pSA_T, SCM_params_CY14_n80['modelEASZT_LZ'],'.-', label='n > 80')
ax11.fill_between(pSA_T, SCM_params_CY14_n80['modelEASZT_LZ_5'], SCM_params_CY14_n80['modelEASZT_LZ_95'], alpha = 0.2)

ax12.loglog(pSA_T, SCM_params_CY14_n20['modelEASZT_LT'],'.-',label = 'n > 20', lw = 1)
ax12.fill_between(pSA_T, SCM_params_CY14_n20['modelEASZT_LT_5'], SCM_params_CY14_n20['modelEASZT_LT_95'], alpha = 0.2)
ax12.loglog(pSA_T, SCM_params_CY14_n30['modelEASZT_LT'],'.-',label = 'n > 30')
ax12.fill_between(pSA_T, SCM_params_CY14_n30['modelEASZT_LT_5'], SCM_params_CY14_n30['modelEASZT_LT_95'], alpha = 0.2)
ax12.loglog(pSA_T, SCM_params_CY14_n40['modelEASZT_LT'],'.-', label='n > 40')
ax12.fill_between(pSA_T, SCM_params_CY14_n40['modelEASZT_LT_5'], SCM_params_CY14_n40['modelEASZT_LT_95'], alpha = 0.2)
ax12.loglog(pSA_T, SCM_params_CY14_n50['modelEASZT_LT'],'.-', label='n > 50')
ax12.fill_between(pSA_T, SCM_params_CY14_n50['modelEASZT_LT_5'], SCM_params_CY14_n50['modelEASZT_LT_95'], alpha = 0.2)
ax12.loglog(pSA_T, SCM_params_CY14_n60['modelEASZT_LT'],'.-', label='n > 60')
ax12.fill_between(pSA_T, SCM_params_CY14_n60['modelEASZT_LT_5'], SCM_params_CY14_n60['modelEASZT_LT_95'], alpha = 0.2)
ax12.loglog(pSA_T, SCM_params_CY14_n80['modelEASZT_LT'],'.-', label='n > 80')
ax12.fill_between(pSA_T, SCM_params_CY14_n80['modelEASZT_LT_5'], SCM_params_CY14_n80['modelEASZT_LT_95'], alpha = 0.2)

ax20.semilogx(pSA_T, SCM_params_CY14_n20['modelEASZT_wS'],'.-',label = 'n > 20', lw = 1)
ax20.fill_between(pSA_T, SCM_params_CY14_n20['modelEASZT_wS_5'], SCM_params_CY14_n20['modelEASZT_wS_95'], alpha = 0.2)
ax20.semilogx(pSA_T, SCM_params_CY14_n30['modelEASZT_wS'],'.-',label = 'n > 30')
ax20.fill_between(pSA_T, SCM_params_CY14_n30['modelEASZT_wS_5'], SCM_params_CY14_n30['modelEASZT_wS_95'], alpha = 0.2)
ax20.semilogx(pSA_T, SCM_params_CY14_n40['modelEASZT_wS'],'.-', label='n > 40')
ax20.fill_between(pSA_T, SCM_params_CY14_n40['modelEASZT_wS_5'], SCM_params_CY14_n40['modelEASZT_wS_95'], alpha = 0.2)
ax20.semilogx(pSA_T, SCM_params_CY14_n50['modelEASZT_wS'],'.-', label='n > 50')
ax20.fill_between(pSA_T, SCM_params_CY14_n50['modelEASZT_wS_5'], SCM_params_CY14_n50['modelEASZT_wS_95'], alpha = 0.2)
ax20.semilogx(pSA_T, SCM_params_CY14_n60['modelEASZT_wS'],'.-', label='n > 60')
ax20.fill_between(pSA_T, SCM_params_CY14_n60['modelEASZT_wS_5'], SCM_params_CY14_n60['modelEASZT_wS_95'], alpha = 0.2)
ax20.semilogx(pSA_T, SCM_params_CY14_n80['modelEASZT_wS'],'.-', label='n > 80')
ax20.fill_between(pSA_T, SCM_params_CY14_n80['modelEASZT_wS_5'], SCM_params_CY14_n80['modelEASZT_wS_95'], alpha = 0.2)

ax21.semilogx(pSA_T, SCM_params_CY14_n20['modelEASZT_wZ'],'.-',label = 'n > 20', lw = 1)
ax21.fill_between(pSA_T, SCM_params_CY14_n20['modelEASZT_wZ_5'], SCM_params_CY14_n20['modelEASZT_wZ_95'], alpha = 0.2)
ax21.semilogx(pSA_T, SCM_params_CY14_n30['modelEASZT_wZ'],'.-',label = 'n > 30')
ax21.fill_between(pSA_T, SCM_params_CY14_n30['modelEASZT_wZ_5'], SCM_params_CY14_n30['modelEASZT_wZ_95'], alpha = 0.2)
ax21.semilogx(pSA_T, SCM_params_CY14_n40['modelEASZT_wZ'],'.-', label='n > 40')
ax21.fill_between(pSA_T, SCM_params_CY14_n40['modelEASZT_wZ_5'], SCM_params_CY14_n40['modelEASZT_wZ_95'], alpha = 0.2)
ax21.semilogx(pSA_T, SCM_params_CY14_n50['modelEASZT_wZ'],'.-', label='n > 50')
ax21.fill_between(pSA_T, SCM_params_CY14_n50['modelEASZT_wZ_5'], SCM_params_CY14_n50['modelEASZT_wZ_95'], alpha = 0.2)
ax21.semilogx(pSA_T, SCM_params_CY14_n60['modelEASZT_wZ'],'.-', label='n > 60')
ax21.fill_between(pSA_T, SCM_params_CY14_n60['modelEASZT_wZ_5'], SCM_params_CY14_n60['modelEASZT_wZ_95'], alpha = 0.2)
ax21.semilogx(pSA_T, SCM_params_CY14_n80['modelEASZT_wZ'],'.-', label='n > 80')
ax21.fill_between(pSA_T, SCM_params_CY14_n80['modelEASZT_wZ_5'], SCM_params_CY14_n80['modelEASZT_wZ_95'], alpha = 0.2)

ax22.semilogx(pSA_T, SCM_params_CY14_n20['modelEASZT_wT'],'.-',label = 'n > 20', lw = 1)
ax22.fill_between(pSA_T, SCM_params_CY14_n20['modelEASZT_wT_5'], SCM_params_CY14_n20['modelEASZT_wT_95'], alpha = 0.2)
ax22.semilogx(pSA_T, SCM_params_CY14_n30['modelEASZT_wT'],'.-',label = 'n > 30')
ax22.fill_between(pSA_T, SCM_params_CY14_n30['modelEASZT_wT_5'], SCM_params_CY14_n30['modelEASZT_wT_95'], alpha = 0.2)
ax22.semilogx(pSA_T, SCM_params_CY14_n40['modelEASZT_wT'],'.-', label='n > 40')
ax22.fill_between(pSA_T, SCM_params_CY14_n40['modelEASZT_wT_5'], SCM_params_CY14_n40['modelEASZT_wT_95'], alpha = 0.2)
ax22.semilogx(pSA_T, SCM_params_CY14_n50['modelEASZT_wT'],'.-', label='n > 50')
ax22.fill_between(pSA_T, SCM_params_CY14_n50['modelEASZT_wT_5'], SCM_params_CY14_n50['modelEASZT_wT_95'], alpha = 0.2)
ax22.semilogx(pSA_T, SCM_params_CY14_n60['modelEASZT_wT'],'.-', label='n > 60')
ax22.fill_between(pSA_T, SCM_params_CY14_n60['modelEASZT_wT_5'], SCM_params_CY14_n60['modelEASZT_wT_95'], alpha = 0.2)
ax22.semilogx(pSA_T, SCM_params_CY14_n80['modelEASZT_wT'],'.-', label='n > 80')
ax22.fill_between(pSA_T, SCM_params_CY14_n80['modelEASZT_wT_5'], SCM_params_CY14_n80['modelEASZT_wT_95'], alpha = 0.2)

ax30.semilogx(pSA_T, SCM_params_CY14_n20['modelEASZT_w'],'.-',label = 'n > 20', lw = 1)
ax30.fill_between(pSA_T, SCM_params_CY14_n20['modelEASZT_w_5'], SCM_params_CY14_n20['modelEASZT_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CY14_n30['modelEASZT_w'],'.-',label = 'n > 30')
ax30.fill_between(pSA_T, SCM_params_CY14_n30['modelEASZT_w_5'], SCM_params_CY14_n30['modelEASZT_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CY14_n40['modelEASZT_w'],'.-', label='n > 40')
ax30.fill_between(pSA_T, SCM_params_CY14_n40['modelEASZT_w_5'], SCM_params_CY14_n40['modelEASZT_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CY14_n50['modelEASZT_w'],'.-', label='n > 50')
ax30.fill_between(pSA_T, SCM_params_CY14_n50['modelEASZT_w_5'], SCM_params_CY14_n50['modelEASZT_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CY14_n60['modelEASZT_w'],'.-', label='n > 60')
ax30.fill_between(pSA_T, SCM_params_CY14_n60['modelEASZT_w_5'], SCM_params_CY14_n60['modelEASZT_w_95'], alpha = 0.2)
ax30.semilogx(pSA_T, SCM_params_CY14_n80['modelEASZT_w'],'.-', label='n > 80')
ax30.fill_between(pSA_T, SCM_params_CY14_n80['modelEASZT_w_5'], SCM_params_CY14_n80['modelEASZT_w_95'], alpha = 0.2)

ax00.set_xlabel('Period, T(s)')
ax01.set_xlabel('Period, T(s)')
ax02.set_xlabel('Period, T(s)')
ax10.set_xlabel('Period, T(s)')
ax11.set_xlabel('Period, T(s)')
ax12.set_xlabel('Period, T(s)')
ax20.set_xlabel('Period, T(s)')
ax21.set_xlabel('Period, T(s)')
ax21.set_xlabel('Period, T(s)')

ax00.set_ylabel(r'$L_E$ (km)')
ax01.set_ylabel(r'$\gamma_E$')
ax02.set_ylabel(r'$L_A\ (^\circ)$')
ax10.set_ylabel(r'$L_S$ (m/s)')
ax11.set_ylabel(r'$L_Z$ (m)')
ax12.set_ylabel(r'$L_T$ (m)')
ax20.set_ylabel(r'$w_S$')
ax21.set_ylabel(r'$w_Z$')
ax22.set_ylabel(r'$w_T$')
ax30.set_ylabel(r'$w$')

ax00.set_xlim(0.01, 6)
ax01.set_xlim(0.01, 6)
ax02.set_xlim(0.01, 6)
ax10.set_xlim(0.01, 6)
ax11.set_xlim(0.01, 6)
ax12.set_xlim(0.01, 6)
ax20.set_xlim(0.01, 6)
ax21.set_xlim(0.01, 6)
ax22.set_xlim(0.01, 6)
ax30.set_xlim(0.01, 6)

ax00.set_ylim(10,500)
ax01.set_ylim(0,1)
ax02.set_ylim(0,45)
ax10.set_ylim(10,5000)
ax11.set_ylim(10,5000)
ax12.set_ylim(1,500)
ax20.set_ylim(0,1)
ax21.set_ylim(0,1)
ax22.set_ylim(0,1)
ax30.set_ylim(0,1)

for ax in [ax00, ax01, ax02, ax10, ax11, ax12, ax20, ax21, ax22, ax30]:
    ax.grid(which='both')

ax01.legend(loc='upper center', ncol=3, fontsize = 9.7)

plt.show()

# 4.0 Correlation deviation computation

## 4.1 Crustal GMMs: NGA-West2 and NZ-GMM

In [ ]:
# compared S22 dataset
e_ASK14 = BM23EXT_CBB21(cbb21_CY14, pSA_T, SCM_params_ASK14)
e_BSSA = BM23EXT_CBB21(cbb21_CY14, pSA_T, SCM_params_BSSA14)
e_CB14 = BM23EXT_CBB21(cbb21_CY14, pSA_T, SCM_params_CB14)
e_CY14 = BM23EXT_CBB21(cbb21_CY14, pSA_T, SCM_params_CY14)
e_ASK22 = BM23EXT_CBB21(cbb21_CY14, pSA_T, SCM_params_ASK22)
e_S22 = BM23EXT_CBB21(cbb21_CY14, pSA_T, SCM_params_S22)

## 4.2 Subduction GMMs: NGA-sub

In [ ]:
e_AG20 = BM23EXT_CBB21(cbb21_K20, pSA_T, SCM_params_AG20)
e_P20 = BM23EXT_CBB21(cbb21_K20, pSA_T, SCM_params_P20)
e_K20 = BM23EXT_CBB21(cbb21_K20, pSA_T, SCM_params_K20)

## 4.3 Chiou & Young's 2014 w/ various observations

In [ ]:
e_CY14_n20 = BM23EXT_CBB21(cbb21_CY14_n20, pSA_T, SCM_params_CY14_n20)
e_CY14_n30 = BM23EXT_CBB21(cbb21_CY14_n20, pSA_T, SCM_params_CY14_n30)
e_CY14_n40 = BM23EXT_CBB21(cbb21_CY14_n20, pSA_T, SCM_params_CY14_n40)
e_CY14_n50 = BM23EXT_CBB21(cbb21_CY14_n20, pSA_T, SCM_params_CY14_n50)
e_CY14_n60 = BM23EXT_CBB21(cbb21_CY14_n20, pSA_T, SCM_params_CY14_n60)
e_CY14_n80 = BM23EXT_CBB21(cbb21_CY14_n20, pSA_T, SCM_params_CY14_n80)

# 5.0 Correlation deviation plots

## 5.1 Crustal GMMs: NGA-West2 and NZ-GMM

In [ ]:
fig, ax = plt.subplots(2,3,figsize = (12,6), dpi = 200, constrained_layout = True)
# model E
ax[0,0].plot(pSA_T, e_ASK14['E']['mean'], '-o', label = 'ASK14', lw = 1)
ax[0,0].fill_between(pSA_T, e_ASK14['E']['mean5'], e_ASK14['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_BSSA['E']['mean'], '-o', label = 'BSSA14', lw = 1)
ax[0,0].fill_between(pSA_T, e_BSSA['E']['mean5'], e_BSSA['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_CB14['E']['mean'], '-o', label = 'CB14', lw = 1)
ax[0,0].fill_between(pSA_T, e_CB14['E']['mean5'], e_CB14['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_CY14['E']['mean'], '-o', label = 'CY14', lw = 1)
ax[0,0].fill_between(pSA_T, e_CY14['E']['mean5'], e_CY14['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_ASK22['E']['mean'], '-o', label = 'A22', lw = 1)
ax[0,0].fill_between(pSA_T, e_ASK22['E']['mean5'], e_ASK22['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_S22['E']['mean'], '-o', label = 'S22', lw = 1)
ax[0,0].fill_between(pSA_T, e_S22['E']['mean5'], e_S22['E']['mean95'], alpha = 0.2)

# model EA
ax[0,1].plot(pSA_T, e_ASK14['EA']['mean'], '-o', label = 'ASK14', lw = 1)
ax[0,1].fill_between(pSA_T, e_ASK14['EA']['mean5'], e_ASK14['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_BSSA['EA']['mean'], '-o', label = 'BSSA14', lw = 1)
ax[0,1].fill_between(pSA_T, e_BSSA['EA']['mean5'], e_BSSA['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_CB14['EA']['mean'], '-o', label = 'CB14', lw = 1)
ax[0,1].fill_between(pSA_T, e_CB14['EA']['mean5'], e_CB14['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_CY14['EA']['mean'], '-o', label = 'CY14', lw = 1)
ax[0,1].fill_between(pSA_T, e_CY14['EA']['mean5'], e_CY14['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_ASK22['EA']['mean'], '-o', label = 'A22', lw = 1)
ax[0,1].fill_between(pSA_T, e_ASK22['EA']['mean5'], e_ASK22['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_S22['EA']['mean'], '-o', label = 'S22', lw = 1)
ax[0,1].fill_between(pSA_T, e_S22['EA']['mean5'], e_S22['EA']['mean95'], alpha = 0.2)

# model EAS
ax[0,2].plot(pSA_T, e_ASK14['EAS']['mean'], '-o', label = 'ASK14', lw = 1)
ax[0,2].fill_between(pSA_T, e_ASK14['EAS']['mean5'], e_ASK14['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_BSSA['EAS']['mean'], '-o', label = 'BSSA14', lw = 1)
ax[0,2].fill_between(pSA_T, e_BSSA['EAS']['mean5'], e_BSSA['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_CB14['EAS']['mean'], '-o', label = 'CB14', lw = 1)
ax[0,2].fill_between(pSA_T, e_CB14['EAS']['mean5'], e_CB14['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_CY14['EAS']['mean'], '-o', label = 'CY14', lw = 1)
ax[0,2].fill_between(pSA_T, e_CY14['EAS']['mean5'], e_CY14['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_ASK22['EAS']['mean'], '-o', label = 'A22', lw = 1)
ax[0,2].fill_between(pSA_T, e_ASK22['EAS']['mean5'], e_ASK22['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_S22['EAS']['mean'], '-o', label = 'S22', lw = 1)
ax[0,2].fill_between(pSA_T, e_S22['EAS']['mean5'], e_S22['EAS']['mean95'], alpha = 0.2)

# model EASZ
ax[1,0].plot(pSA_T, e_ASK14['EASZ']['mean'], '-o', label = 'ASK14', lw = 1)
ax[1,0].fill_between(pSA_T, e_ASK14['EASZ']['mean5'], e_ASK14['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_BSSA['EASZ']['mean'], '-o', label = 'BSSA14', lw = 1)
ax[1,0].fill_between(pSA_T, e_BSSA['EASZ']['mean5'], e_BSSA['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_CB14['EASZ']['mean'], '-o', label = 'CB14', lw = 1)
ax[1,0].fill_between(pSA_T, e_CB14['EASZ']['mean5'], e_CB14['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_CY14['EASZ']['mean'], '-o', label = 'CY14', lw = 1)
ax[1,0].fill_between(pSA_T, e_CY14['EASZ']['mean5'], e_CY14['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_ASK22['EASZ']['mean'], '-o', label = 'A22', lw = 1)
ax[1,0].fill_between(pSA_T, e_ASK22['EASZ']['mean5'], e_ASK22['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_S22['EASZ']['mean'], '-o', label = 'S22', lw = 1)
ax[1,0].fill_between(pSA_T, e_S22['EASZ']['mean5'], e_S22['EASZ']['mean95'], alpha = 0.2)

# model EAST
ax[1,1].plot(pSA_T, e_ASK14['EAST']['mean'], '-o', label = 'ASK14', lw = 1)
ax[1,1].fill_between(pSA_T, e_ASK14['EAST']['mean5'], e_ASK14['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_BSSA['EAST']['mean'], '-o', label = 'BSSA14', lw = 1)
ax[1,1].fill_between(pSA_T, e_BSSA['EAST']['mean5'], e_BSSA['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_CB14['EAST']['mean'], '-o', label = 'CB14', lw = 1)
ax[1,1].fill_between(pSA_T, e_CB14['EAST']['mean5'], e_CB14['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_CY14['EAST']['mean'], '-o', label = 'CY14', lw = 1)
ax[1,1].fill_between(pSA_T, e_CY14['EAST']['mean5'], e_CY14['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_ASK22['EAST']['mean'], '-o', label = 'A22', lw = 1)
ax[1,1].fill_between(pSA_T, e_ASK22['EAST']['mean5'], e_ASK22['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_S22['EAST']['mean'], '-o', label = 'S22', lw = 1)
ax[1,1].fill_between(pSA_T, e_S22['EAST']['mean5'], e_S22['EAST']['mean95'], alpha = 0.2)

# model EASZT
ax[1,2].plot(pSA_T, e_ASK14['EASZT']['mean'], '-o', label = 'ASK14', lw = 1)
ax[1,2].fill_between(pSA_T, e_ASK14['EASZT']['mean5'], e_ASK14['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_BSSA['EASZT']['mean'], '-o', label = 'BSSA14', lw = 1)
ax[1,2].fill_between(pSA_T, e_BSSA['EASZT']['mean5'], e_BSSA['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_CB14['EASZT']['mean'], '-o', label = 'CB14', lw = 1)
ax[1,2].fill_between(pSA_T, e_CB14['EASZT']['mean5'], e_CB14['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_CY14['EASZT']['mean'], '-o', label = 'CY14', lw = 1)
ax[1,2].fill_between(pSA_T, e_CY14['EASZT']['mean5'], e_CY14['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_ASK22['EASZT']['mean'], '-o', label = 'A22', lw = 1)
ax[1,2].fill_between(pSA_T, e_ASK22['EASZT']['mean5'], e_ASK22['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_S22['EASZT']['mean'], '-o', label = 'S22', lw = 1)
ax[1,2].fill_between(pSA_T, e_S22['EASZT']['mean5'], e_S22['EASZT']['mean95'], alpha = 0.2)

for i in range(2):
    for j in range(3):
        ax[i,j].set_ylim([-1.5,1.5])
        ax[i,j].set_xlim([0.01,6])
        ax[i,j].plot([0.01,6],[0,0],'--k')
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_xscale('log')
        ax[1,j].set_xlabel('Period, T (s)')
        ax[i,0].set_ylabel(r'Correlation Mean, $\mu_e$')
ax[0,0].set_title('Model E')
ax[0,1].set_title('Model EA')
ax[0,2].set_title('Model EAS')
ax[1,0].set_title('Model EASZ')
ax[1,1].set_title('Model EAST')
ax[1,2].set_title('Model EASZT')
ax[0,1].legend(loc = 'upper center', ncol=3, fontsize = 10)
plt.show()

# plot: correlation deviation standard deviation
fig, ax = plt.subplots(2,3,figsize = (12,6), dpi = 200, constrained_layout = True)
# model E
ax[0,0].plot(pSA_T, e_ASK14['E']['std'], '-o', label = 'ASK14', lw = 1)
ax[0,0].fill_between(pSA_T, e_ASK14['E']['std5'], e_ASK14['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_BSSA['E']['std'], '-o', label = 'BSSA14', lw = 1)
ax[0,0].fill_between(pSA_T, e_BSSA['E']['std5'], e_BSSA['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_CB14['E']['std'], '-o', label = 'CB14', lw = 1)
ax[0,0].fill_between(pSA_T, e_CB14['E']['std5'], e_CB14['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_CY14['E']['std'], '-o', label = 'CY14', lw = 1)
ax[0,0].fill_between(pSA_T, e_CY14['E']['std5'], e_CY14['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_ASK22['E']['std'], '-o', label = 'A22', lw = 1)
ax[0,0].fill_between(pSA_T, e_ASK22['E']['std5'], e_ASK22['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_S22['E']['std'], '-o', label = 'S22', lw = 1)
ax[0,0].fill_between(pSA_T, e_S22['E']['std5'], e_ASK22['E']['std95'], alpha = 0.2)

# model EA
ax[0,1].plot(pSA_T, e_ASK14['EA']['std'], '-o', label = 'ASK14', lw = 1)
ax[0,1].fill_between(pSA_T, e_ASK14['EA']['std5'], e_ASK14['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_BSSA['EA']['std'], '-o', label = 'BSSA14', lw = 1)
ax[0,1].fill_between(pSA_T, e_BSSA['EA']['std5'], e_BSSA['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_CB14['EA']['std'], '-o', label = 'CB14', lw = 1)
ax[0,1].fill_between(pSA_T, e_CB14['EA']['std5'], e_CB14['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_CY14['EA']['std'], '-o', label = 'CY14', lw = 1)
ax[0,1].fill_between(pSA_T, e_CY14['EA']['std5'], e_CY14['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_ASK22['EA']['std'], '-o', label = 'A22', lw = 1)
ax[0,1].fill_between(pSA_T, e_ASK22['EA']['std5'], e_ASK22['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_S22['EA']['std'], '-o', label = 'S22', lw = 1)
ax[0,1].fill_between(pSA_T, e_S22['EA']['std5'], e_S22['EA']['std95'], alpha = 0.2)

# model EAS
ax[0,2].plot(pSA_T, e_ASK14['EAS']['std'], '-o', label = 'ASK14', lw = 1)
ax[0,2].fill_between(pSA_T, e_ASK14['EAS']['std5'], e_ASK14['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_BSSA['EAS']['std'], '-o', label = 'BSSA14', lw = 1)
ax[0,2].fill_between(pSA_T, e_BSSA['EAS']['std5'], e_BSSA['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_CB14['EAS']['std'], '-o', label = 'CB14', lw = 1)
ax[0,2].fill_between(pSA_T, e_CB14['EAS']['std5'], e_CB14['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_CY14['EAS']['std'], '-o', label = 'CY14', lw = 1)
ax[0,2].fill_between(pSA_T, e_CY14['EAS']['std5'], e_CY14['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_ASK22['EAS']['std'], '-o', label = 'A22', lw = 1)
ax[0,2].fill_between(pSA_T, e_ASK22['EAS']['std5'], e_ASK22['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_S22['EAS']['std'], '-o', label = 'ASK22', lw = 1)
ax[0,2].fill_between(pSA_T, e_S22['EAS']['std5'], e_S22['EAS']['std95'], alpha = 0.2)

# model EASZ
ax[1,0].plot(pSA_T, e_ASK14['EASZ']['std'], '-o', label = 'ASK14', lw = 1)
ax[1,0].fill_between(pSA_T, e_ASK14['EASZ']['std5'], e_ASK14['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_BSSA['EASZ']['std'], '-o', label = 'BSSA14', lw = 1)
ax[1,0].fill_between(pSA_T, e_BSSA['EASZ']['std5'], e_BSSA['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_CB14['EASZ']['std'], '-o', label = 'CB14', lw = 1)
ax[1,0].fill_between(pSA_T, e_CB14['EASZ']['std5'], e_CB14['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_CY14['EASZ']['std'], '-o', label = 'CY14', lw = 1)
ax[1,0].fill_between(pSA_T, e_CY14['EASZ']['std5'], e_CY14['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_ASK22['EASZ']['std'], '-o', label = 'A22', lw = 1)
ax[1,0].fill_between(pSA_T, e_ASK22['EASZ']['std5'], e_ASK22['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_S22['EASZ']['std'], '-o', label = 'ASK22', lw = 1)
ax[1,0].fill_between(pSA_T, e_S22['EASZ']['std5'], e_S22['EASZ']['std95'], alpha = 0.2)

# model EAST
ax[1,1].plot(pSA_T, e_ASK14['EAST']['std'], '-o', label = 'ASK14', lw = 1)
ax[1,1].fill_between(pSA_T, e_ASK14['EAST']['std5'], e_ASK14['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_BSSA['EAST']['std'], '-o', label = 'BSSA14', lw = 1)
ax[1,1].fill_between(pSA_T, e_BSSA['EAST']['std5'], e_BSSA['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_CB14['EAST']['std'], '-o', label = 'CB14', lw = 1)
ax[1,1].fill_between(pSA_T, e_CB14['EAST']['std5'], e_CB14['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_CY14['EAST']['std'], '-o', label = 'CY14', lw = 1)
ax[1,1].fill_between(pSA_T, e_CY14['EAST']['std5'], e_CY14['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_ASK22['EAST']['std'], '-o', label = 'A22', lw = 1)
ax[1,1].fill_between(pSA_T, e_ASK22['EAST']['std5'], e_ASK22['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_S22['EAST']['std'], '-o', label = 'S22', lw = 1)
ax[1,1].fill_between(pSA_T, e_S22['EAST']['std5'], e_S22['EAST']['std95'], alpha = 0.2)

# model EASZT
ax[1,2].plot(pSA_T, e_ASK14['EASZT']['std'], '-o', label = 'ASK14', lw = 1)
ax[1,2].fill_between(pSA_T, e_ASK14['EASZT']['std5'], e_ASK14['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_BSSA['EASZT']['std'], '-o', label = 'BSSA14', lw = 1)
ax[1,2].fill_between(pSA_T, e_BSSA['EASZT']['std5'], e_BSSA['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_CB14['EASZT']['std'], '-o', label = 'CB14', lw = 1)
ax[1,2].fill_between(pSA_T, e_CB14['EASZT']['std5'], e_CB14['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_CY14['EASZT']['std'], '-o', label = 'CY14', lw = 1)
ax[1,2].fill_between(pSA_T, e_CY14['EASZT']['std5'], e_CY14['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_ASK22['EASZT']['std'], '-o', label = 'A22', lw = 1)
ax[1,2].fill_between(pSA_T, e_ASK22['EASZT']['std5'], e_ASK22['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_S22['EASZT']['std'], '-o', label = 'S22', lw = 1)
ax[1,2].fill_between(pSA_T, e_S22['EASZT']['std5'], e_S22['EASZT']['std95'], alpha = 0.2)

for i in range(2):
    for j in range(3):
        ax[i,j].set_ylim([1,3])
        ax[i,j].set_xlim([0.01,6])
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_xscale('log')
        ax[1,j].set_xlabel('Period, T (s)')
        ax[i,0].set_ylabel(r'Correlation Deviation, $\sigma_{e}$')
ax[0,0].set_title('Model E')
ax[0,1].set_title('Model EA')
ax[0,2].set_title('Model EAS')
ax[1,0].set_title('Model EASZ')
ax[1,1].set_title('Model EAST')
ax[1,2].set_title('Model EASZT')
ax[0,1].legend(loc = 'upper center', ncol=3, fontsize = 10)
plt.show()

## 5.2 Subduction GMMs: NGA-Sub

In [ ]:
fig, ax = plt.subplots(2,3,figsize = (12,6), dpi = 200, constrained_layout = True)
# model E
ax[0,0].plot(pSA_T, e_AG20['E']['mean'], '-o', label = 'AG20', lw = 1)
ax[0,0].fill_between(pSA_T, e_AG20['E']['mean5'], e_AG20['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_P20['E']['mean'], '-o', label = 'P20', lw = 1)
ax[0,0].fill_between(pSA_T, e_P20['E']['mean5'], e_P20['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_K20['E']['mean'], '-o', label = 'K20', lw = 1)
ax[0,0].fill_between(pSA_T, e_K20['E']['mean5'], e_K20['E']['mean95'], alpha = 0.2)

# model EA
ax[0,1].plot(pSA_T, e_AG20['EA']['mean'], '-o', label = 'AG20', lw = 1)
ax[0,1].fill_between(pSA_T, e_AG20['EA']['mean5'], e_AG20['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_P20['EA']['mean'], '-o', label = 'P20', lw = 1)
ax[0,1].fill_between(pSA_T, e_P20['EA']['mean5'], e_P20['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_K20['EA']['mean'], '-o', label = 'K20', lw = 1)
ax[0,1].fill_between(pSA_T, e_K20['EA']['mean5'], e_K20['EA']['mean95'], alpha = 0.2)

# model EAS
ax[0,2].plot(pSA_T, e_AG20['EAS']['mean'], '-o', label = 'AG20', lw = 1)
ax[0,2].fill_between(pSA_T, e_AG20['EAS']['mean5'], e_AG20['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_P20['EAS']['mean'], '-o', label = 'P20', lw = 1)
ax[0,2].fill_between(pSA_T, e_P20['EAS']['mean5'], e_P20['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_K20['EAS']['mean'], '-o', label = 'K20', lw = 1)
ax[0,2].fill_between(pSA_T, e_K20['EAS']['mean5'], e_K20['EAS']['mean95'], alpha = 0.2)

# model EASZ
ax[1,0].plot(pSA_T, e_AG20['EASZ']['mean'], '-o', label = 'AG20', lw = 1)
ax[1,0].fill_between(pSA_T, e_AG20['EASZ']['mean5'], e_AG20['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_P20['EASZ']['mean'], '-o', label = 'P20', lw = 1)
ax[1,0].fill_between(pSA_T, e_P20['EASZ']['mean5'], e_P20['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_K20['EASZ']['mean'], '-o', label = 'K20', lw = 1)
ax[1,0].fill_between(pSA_T, e_K20['EASZ']['mean5'], e_K20['EASZ']['mean95'], alpha = 0.2)

# model EAST
ax[1,1].plot(pSA_T, e_AG20['EAST']['mean'], '-o', label = 'AG20', lw = 1)
ax[1,1].fill_between(pSA_T, e_AG20['EAST']['mean5'], e_AG20['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_P20['EAST']['mean'], '-o', label = 'P20', lw = 1)
ax[1,1].fill_between(pSA_T, e_P20['EAST']['mean5'], e_P20['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_K20['EAST']['mean'], '-o', label = 'K20', lw = 1)
ax[1,1].fill_between(pSA_T, e_K20['EAST']['mean5'], e_K20['EAST']['mean95'], alpha = 0.2)

# model EASZT
ax[1,2].plot(pSA_T, e_AG20['EASZT']['mean'], '-o', label = 'AG20', lw = 1)
ax[1,2].fill_between(pSA_T, e_AG20['EASZT']['mean5'], e_AG20['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_P20['EASZT']['mean'], '-o', label = 'P20', lw = 1)
ax[1,2].fill_between(pSA_T, e_P20['EASZT']['mean5'], e_P20['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_K20['EASZT']['mean'], '-o', label = 'K20', lw = 1)
ax[1,2].fill_between(pSA_T, e_K20['EASZT']['mean5'], e_K20['EASZT']['mean95'], alpha = 0.2)

for i in range(2):
    for j in range(3):
        ax[i,j].set_ylim([-1.5,1.5])
        ax[i,j].set_xlim([0.01,6])
        ax[i,j].plot([0.01,6],[0,0],'--k')
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_xscale('log')
        ax[1,j].set_xlabel('Period, T (s)')
        ax[i,0].set_ylabel(r'Correlation Mean, $\mu_e$')
ax[0,0].set_title('Model E')
ax[0,1].set_title('Model EA')
ax[0,2].set_title('Model EAS')
ax[1,0].set_title('Model EASZ')
ax[1,1].set_title('Model EAST')
ax[1,2].set_title('Model EASZT')
ax[0,1].legend(loc = 'upper center', ncol=3, fontsize = 10)
plt.show()

# plot: correlation deviation standard deviation
fig, ax = plt.subplots(2,3,figsize = (12,6), dpi = 200, constrained_layout = True)
# model E
ax[0,0].plot(pSA_T, e_AG20['E']['std'], '-o', label = 'AG20', lw = 1)
ax[0,0].fill_between(pSA_T, e_AG20['E']['std5'], e_AG20['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_P20['E']['std'], '-o', label = 'P20', lw = 1)
ax[0,0].fill_between(pSA_T, e_P20['E']['std5'], e_P20['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_K20['E']['std'], '-o', label = 'K20', lw = 1)
ax[0,0].fill_between(pSA_T, e_K20['E']['std5'], e_K20['E']['std95'], alpha = 0.2)

# model EA
ax[0,1].plot(pSA_T, e_AG20['EA']['std'], '-o', label = 'AG20', lw = 1)
ax[0,1].fill_between(pSA_T, e_AG20['EA']['std5'], e_AG20['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_P20['EA']['std'], '-o', label = 'P20', lw = 1)
ax[0,1].fill_between(pSA_T, e_P20['EA']['std5'], e_P20['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_K20['EA']['std'], '-o', label = 'K20', lw = 1)
ax[0,1].fill_between(pSA_T, e_K20['EA']['std5'], e_K20['EA']['std95'], alpha = 0.2)

# model EAS
ax[0,2].plot(pSA_T, e_AG20['EAS']['std'], '-o', label = 'AG20', lw = 1)
ax[0,2].fill_between(pSA_T, e_AG20['EAS']['std5'], e_AG20['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_P20['EAS']['std'], '-o', label = 'P20', lw = 1)
ax[0,2].fill_between(pSA_T, e_P20['EAS']['std5'], e_P20['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_K20['EAS']['std'], '-o', label = 'K20', lw = 1)
ax[0,2].fill_between(pSA_T, e_K20['EAS']['std5'], e_K20['EAS']['std95'], alpha = 0.2)

# model EASZ
ax[1,0].plot(pSA_T, e_AG20['EASZ']['std'], '-o', label = 'AG20', lw = 1)
ax[1,0].fill_between(pSA_T, e_AG20['EASZ']['std5'], e_AG20['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_P20['EASZ']['std'], '-o', label = 'P20', lw = 1)
ax[1,0].fill_between(pSA_T, e_P20['EASZ']['std5'], e_P20['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_K20['EASZ']['std'], '-o', label = 'K20', lw = 1)
ax[1,0].fill_between(pSA_T, e_K20['EASZ']['std5'], e_K20['EASZ']['std95'], alpha = 0.2)

# model EAST
ax[1,1].plot(pSA_T, e_AG20['EAST']['std'], '-o', label = 'AG20', lw = 1)
ax[1,1].fill_between(pSA_T, e_AG20['EAST']['std5'], e_AG20['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_P20['EAST']['std'], '-o', label = 'P20', lw = 1)
ax[1,1].fill_between(pSA_T, e_P20['EAST']['std5'], e_P20['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_K20['EAST']['std'], '-o', label = 'K20', lw = 1)
ax[1,1].fill_between(pSA_T, e_K20['EAST']['std5'], e_K20['EAST']['std95'], alpha = 0.2)

# model EASZT
ax[1,2].plot(pSA_T, e_AG20['EASZT']['std'], '-o', label = 'AG20', lw = 1)
ax[1,2].fill_between(pSA_T, e_AG20['EASZT']['std5'], e_AG20['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_P20['EASZT']['std'], '-o', label = 'P20', lw = 1)
ax[1,2].fill_between(pSA_T, e_P20['EASZT']['std5'], e_P20['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_K20['EASZT']['std'], '-o', label = 'K20', lw = 1)
ax[1,2].fill_between(pSA_T, e_K20['EASZT']['std5'], e_K20['EASZT']['std95'], alpha = 0.2)

for i in range(2):
    for j in range(3):
        ax[i,j].set_ylim([1,3])
        ax[i,j].set_xlim([0.01,6])
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_xscale('log')
        ax[1,j].set_xlabel('Period, T (s)')
        ax[i,0].set_ylabel(r'Correlation Deviation, $\sigma_{e}$')
ax[0,0].set_title('Model E')
ax[0,1].set_title('Model EA')
ax[0,2].set_title('Model EAS')
ax[1,0].set_title('Model EASZ')
ax[1,1].set_title('Model EAST')
ax[1,2].set_title('Model EASZT')
ax[0,1].legend(loc = 'upper center', ncol=3, fontsize = 10)
plt.show()

## 5.3 Chiou & Young's 2014 w/ various observations

In [ ]:
fig, ax = plt.subplots(2,3,figsize = (12,6), dpi = 200, constrained_layout = True)
# model E
ax[0,0].plot(pSA_T, e_CY14_n20['E']['mean'], '-o', label = 'n > 20', lw = 1)
ax[0,0].fill_between(pSA_T, e_CY14_n20['E']['mean5'], e_CY14_n20['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_CY14_n30['E']['mean'], '-o', label = 'n > 30', lw = 1)
ax[0,0].fill_between(pSA_T, e_CY14_n30['E']['mean5'], e_CY14_n30['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_CY14_n40['E']['mean'], '-o', label = 'n > 40', lw = 1)
ax[0,0].fill_between(pSA_T, e_CY14_n40['E']['mean5'], e_CY14_n40['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_CY14_n50['E']['mean'], '-o', label = 'n > 50', lw = 1)
ax[0,0].fill_between(pSA_T, e_CY14_n50['E']['mean5'], e_CY14_n50['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_CY14_n60['E']['mean'], '-o', label = 'n > 60', lw = 1)
ax[0,0].fill_between(pSA_T, e_CY14_n60['E']['mean5'], e_CY14_n60['E']['mean95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_CY14_n80['E']['mean'], '-o', label = 'n > 80', lw = 1)
ax[0,0].fill_between(pSA_T, e_CY14_n80['E']['mean5'], e_CY14_n80['E']['mean95'], alpha = 0.2)

# model EA
ax[0,1].plot(pSA_T, e_CY14_n20['EA']['mean'], '-o', label = 'n > 20', lw = 1)
ax[0,1].fill_between(pSA_T, e_CY14_n20['EA']['mean5'], e_CY14_n20['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_CY14_n30['EA']['mean'], '-o', label = 'n > 30', lw = 1)
ax[0,1].fill_between(pSA_T, e_CY14_n30['EA']['mean5'], e_CY14_n30['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_CY14_n40['EA']['mean'], '-o', label = 'n > 40', lw = 1)
ax[0,1].fill_between(pSA_T, e_CY14_n40['EA']['mean5'], e_CY14_n40['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_CY14_n50['EA']['mean'], '-o', label = 'n > 50', lw = 1)
ax[0,1].fill_between(pSA_T, e_CY14_n50['EA']['mean5'], e_CY14_n50['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_CY14_n60['EA']['mean'], '-o', label = 'n > 60', lw = 1)
ax[0,1].fill_between(pSA_T, e_CY14_n60['EA']['mean5'], e_CY14_n60['EA']['mean95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_CY14_n80['EA']['mean'], '-o', label = 'n > 80', lw = 1)
ax[0,1].fill_between(pSA_T, e_CY14_n80['EA']['mean5'], e_CY14_n80['EA']['mean95'], alpha = 0.2)

# model EAS
ax[0,2].plot(pSA_T, e_CY14_n20['EAS']['mean'], '-o', label = 'n > 20', lw = 1)
ax[0,2].fill_between(pSA_T, e_CY14_n20['EAS']['mean5'], e_CY14_n20['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_CY14_n30['EAS']['mean'], '-o', label = 'n > 30', lw = 1)
ax[0,2].fill_between(pSA_T, e_CY14_n30['EAS']['mean5'], e_CY14_n30['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_CY14_n40['EAS']['mean'], '-o', label = 'n > 40', lw = 1)
ax[0,2].fill_between(pSA_T, e_CY14_n40['EAS']['mean5'], e_CY14_n40['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_CY14_n50['EAS']['mean'], '-o', label = 'n > 50', lw = 1)
ax[0,2].fill_between(pSA_T, e_CY14_n50['EAS']['mean5'], e_CY14_n50['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_CY14_n60['EAS']['mean'], '-o', label = 'n > 60', lw = 1)
ax[0,2].fill_between(pSA_T, e_CY14_n60['EAS']['mean5'], e_CY14_n60['EAS']['mean95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_CY14_n80['EAS']['mean'], '-o', label = 'n > 80', lw = 1)
ax[0,2].fill_between(pSA_T, e_CY14_n80['EAS']['mean5'], e_CY14_n80['EAS']['mean95'], alpha = 0.2)

# model EASZ
ax[1,0].plot(pSA_T, e_CY14_n20['EASZ']['mean'], '-o', label = 'n > 20', lw = 1)
ax[1,0].fill_between(pSA_T, e_CY14_n20['EASZ']['mean5'], e_CY14_n20['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_CY14_n30['EASZ']['mean'], '-o', label = 'n > 30', lw = 1)
ax[1,0].fill_between(pSA_T, e_CY14_n30['EASZ']['mean5'], e_CY14_n30['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_CY14_n40['EASZ']['mean'], '-o', label = 'n > 40', lw = 1)
ax[1,0].fill_between(pSA_T, e_CY14_n40['EASZ']['mean5'], e_CY14_n40['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_CY14_n50['EASZ']['mean'], '-o', label = 'n > 50', lw = 1)
ax[1,0].fill_between(pSA_T, e_CY14_n50['EASZ']['mean5'], e_CY14_n50['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_CY14_n60['EASZ']['mean'], '-o', label = 'n > 60', lw = 1)
ax[1,0].fill_between(pSA_T, e_CY14_n60['EASZ']['mean5'], e_CY14_n60['EASZ']['mean95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_CY14_n80['EASZ']['mean'], '-o', label = 'n > 80', lw = 1)
ax[1,0].fill_between(pSA_T, e_CY14_n80['EASZ']['mean5'], e_CY14_n80['EASZ']['mean95'], alpha = 0.2)

# model EAST
ax[1,1].plot(pSA_T, e_CY14_n20['EAST']['mean'], '-o', label = 'n > 20', lw = 1)
ax[1,1].fill_between(pSA_T, e_CY14_n20['EAST']['mean5'], e_CY14_n20['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_CY14_n30['EAST']['mean'], '-o', label = 'n > 30', lw = 1)
ax[1,1].fill_between(pSA_T, e_CY14_n30['EAST']['mean5'], e_CY14_n30['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_CY14_n40['EAST']['mean'], '-o', label = 'n > 40', lw = 1)
ax[1,1].fill_between(pSA_T, e_CY14_n40['EAST']['mean5'], e_CY14_n40['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_CY14_n50['EAST']['mean'], '-o', label = 'n > 50', lw = 1)
ax[1,1].fill_between(pSA_T, e_CY14_n50['EAST']['mean5'], e_CY14_n50['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_CY14_n60['EAST']['mean'], '-o', label = 'n > 60', lw = 1)
ax[1,1].fill_between(pSA_T, e_CY14_n60['EAST']['mean5'], e_CY14_n60['EAST']['mean95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_CY14_n80['EAST']['mean'], '-o', label = 'n > 80', lw = 1)
ax[1,1].fill_between(pSA_T, e_CY14_n80['EAST']['mean5'], e_CY14_n80['EAST']['mean95'], alpha = 0.2)

# model EASZT
ax[1,2].plot(pSA_T, e_CY14_n20['EASZT']['mean'], '-o', label = 'n > 20', lw = 1)
ax[1,2].fill_between(pSA_T, e_CY14_n20['EASZT']['mean5'], e_CY14_n20['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_CY14_n30['EASZT']['mean'], '-o', label = 'n > 30', lw = 1)
ax[1,2].fill_between(pSA_T, e_CY14_n30['EASZT']['mean5'], e_CY14_n30['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_CY14_n40['EASZT']['mean'], '-o', label = 'n > 40', lw = 1)
ax[1,2].fill_between(pSA_T, e_CY14_n40['EASZT']['mean5'], e_CY14_n40['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_CY14_n50['EASZT']['mean'], '-o', label = 'n > 50', lw = 1)
ax[1,2].fill_between(pSA_T, e_CY14_n50['EASZT']['mean5'], e_CY14_n50['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_CY14_n60['EASZT']['mean'], '-o', label = 'n > 60', lw = 1)
ax[1,2].fill_between(pSA_T, e_CY14_n60['EASZT']['mean5'], e_CY14_n60['EASZT']['mean95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_CY14_n80['EASZT']['mean'], '-o', label = 'n > 80', lw = 1)
ax[1,2].fill_between(pSA_T, e_CY14_n80['EASZT']['mean5'], e_CY14_n80['EASZT']['mean95'], alpha = 0.2)

for i in range(2):
    for j in range(3):
        ax[i,j].set_ylim([-1.5,1.5])
        ax[i,j].set_xlim([0.01,6])
        ax[i,j].plot([0.01,6],[0,0],'--k')
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_xscale('log')
        ax[1,j].set_xlabel('Period, T (s)')
        ax[i,0].set_ylabel(r'Correlation Mean, $\mu_e$')
ax[0,0].set_title('Model E')
ax[0,1].set_title('Model EA')
ax[0,2].set_title('Model EAS')
ax[1,0].set_title('Model EASZ')
ax[1,1].set_title('Model EAST')
ax[1,2].set_title('Model EASZT')
ax[0,1].legend(loc = 'upper center', ncol=3, fontsize = 10)
plt.show()

# plot: correlation deviation standard deviation
fig, ax = plt.subplots(2,3,figsize = (12,6), dpi = 200, constrained_layout = True)
# model E
ax[0,0].plot(pSA_T, e_CY14_n20['E']['std'], '-o', label = 'n > 20', lw = 1)
ax[0,0].fill_between(pSA_T, e_CY14_n20['E']['std5'], e_CY14_n20['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_CY14_n30['E']['std'], '-o', label = 'n > 30', lw = 1)
ax[0,0].fill_between(pSA_T, e_CY14_n30['E']['std5'], e_CY14_n30['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_CY14_n40['E']['std'], '-o', label = 'n > 40', lw = 1)
ax[0,0].fill_between(pSA_T, e_CY14_n40['E']['std5'], e_CY14_n40['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_CY14_n50['E']['std'], '-o', label = 'n > 50', lw = 1)
ax[0,0].fill_between(pSA_T, e_CY14_n50['E']['std5'], e_CY14_n50['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_CY14_n60['E']['std'], '-o', label = 'n > 60', lw = 1)
ax[0,0].fill_between(pSA_T, e_CY14_n60['E']['std5'], e_CY14_n60['E']['std95'], alpha = 0.2)
ax[0,0].plot(pSA_T, e_CY14_n80['E']['std'], '-o', label = 'n > 80', lw = 1)
ax[0,0].fill_between(pSA_T, e_CY14_n80['E']['std5'], e_CY14_n80['E']['std95'], alpha = 0.2)

# model EA
ax[0,1].plot(pSA_T, e_CY14_n20['EA']['std'], '-o', label = 'n > 20', lw = 1)
ax[0,1].fill_between(pSA_T, e_CY14_n20['EA']['std5'], e_CY14_n20['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_CY14_n30['EA']['std'], '-o', label = 'n > 30', lw = 1)
ax[0,1].fill_between(pSA_T, e_CY14_n30['EA']['std5'], e_CY14_n30['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_CY14_n40['EA']['std'], '-o', label = 'n > 40', lw = 1)
ax[0,1].fill_between(pSA_T, e_CY14_n40['EA']['std5'], e_CY14_n40['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_CY14_n50['EA']['std'], '-o', label = 'n > 50', lw = 1)
ax[0,1].fill_between(pSA_T, e_CY14_n50['EA']['std5'], e_CY14_n50['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_CY14_n60['EA']['std'], '-o', label = 'n > 60', lw = 1)
ax[0,1].fill_between(pSA_T, e_CY14_n60['EA']['std5'], e_CY14_n60['EA']['std95'], alpha = 0.2)
ax[0,1].plot(pSA_T, e_CY14_n80['EA']['std'], '-o', label = 'n > 80', lw = 1)
ax[0,1].fill_between(pSA_T, e_CY14_n80['EA']['std5'], e_CY14_n80['EA']['std95'], alpha = 0.2)

# model EAS
ax[0,2].plot(pSA_T, e_CY14_n20['EAS']['std'], '-o', label = 'n > 20', lw = 1)
ax[0,2].fill_between(pSA_T, e_CY14_n20['EAS']['std5'], e_CY14_n20['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_CY14_n30['EAS']['std'], '-o', label = 'n > 30', lw = 1)
ax[0,2].fill_between(pSA_T, e_CY14_n30['EAS']['std5'], e_CY14_n30['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_CY14_n40['EAS']['std'], '-o', label = 'n > 40', lw = 1)
ax[0,2].fill_between(pSA_T, e_CY14_n40['EAS']['std5'], e_CY14_n40['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_CY14_n50['EAS']['std'], '-o', label = 'n > 50', lw = 1)
ax[0,2].fill_between(pSA_T, e_CY14_n50['EAS']['std5'], e_CY14_n50['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_CY14_n60['EAS']['std'], '-o', label = 'n > 60', lw = 1)
ax[0,2].fill_between(pSA_T, e_CY14_n60['EAS']['std5'], e_CY14_n60['EAS']['std95'], alpha = 0.2)
ax[0,2].plot(pSA_T, e_CY14_n80['EAS']['std'], '-o', label = 'n > 80', lw = 1)
ax[0,2].fill_between(pSA_T, e_CY14_n80['EAS']['std5'], e_CY14_n80['EAS']['std95'], alpha = 0.2)

# model EASZ
ax[1,0].plot(pSA_T, e_CY14_n20['EASZ']['std'], '-o', label = 'n > 20', lw = 1)
ax[1,0].fill_between(pSA_T, e_CY14_n20['EASZ']['std5'], e_CY14_n20['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_CY14_n30['EASZ']['std'], '-o', label = 'n > 30', lw = 1)
ax[1,0].fill_between(pSA_T, e_CY14_n30['EASZ']['std5'], e_CY14_n30['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_CY14_n40['EASZ']['std'], '-o', label = 'n > 40', lw = 1)
ax[1,0].fill_between(pSA_T, e_CY14_n40['EASZ']['std5'], e_CY14_n40['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_CY14_n50['EASZ']['std'], '-o', label = 'n > 50', lw = 1)
ax[1,0].fill_between(pSA_T, e_CY14_n50['EASZ']['std5'], e_CY14_n50['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_CY14_n60['EASZ']['std'], '-o', label = 'n > 60', lw = 1)
ax[1,0].fill_between(pSA_T, e_CY14_n60['EASZ']['std5'], e_CY14_n60['EASZ']['std95'], alpha = 0.2)
ax[1,0].plot(pSA_T, e_CY14_n80['EASZ']['std'], '-o', label = 'n > 80', lw = 1)
ax[1,0].fill_between(pSA_T, e_CY14_n80['EASZ']['std5'], e_CY14_n80['EASZ']['std95'], alpha = 0.2)

# model EAST
ax[1,1].plot(pSA_T, e_CY14_n20['EAST']['std'], '-o', label = 'n > 20', lw = 1)
ax[1,1].fill_between(pSA_T, e_CY14_n20['EAST']['std5'], e_CY14_n20['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_CY14_n30['EAST']['std'], '-o', label = 'n > 30', lw = 1)
ax[1,1].fill_between(pSA_T, e_CY14_n30['EAST']['std5'], e_CY14_n30['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_CY14_n40['EAST']['std'], '-o', label = 'n > 40', lw = 1)
ax[1,1].fill_between(pSA_T, e_CY14_n40['EAST']['std5'], e_CY14_n40['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_CY14_n50['EAST']['std'], '-o', label = 'n > 50', lw = 1)
ax[1,1].fill_between(pSA_T, e_CY14_n50['EAST']['std5'], e_CY14_n50['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_CY14_n60['EAST']['std'], '-o', label = 'n > 60', lw = 1)
ax[1,1].fill_between(pSA_T, e_CY14_n60['EAST']['std5'], e_CY14_n60['EAST']['std95'], alpha = 0.2)
ax[1,1].plot(pSA_T, e_CY14_n80['EAST']['std'], '-o', label = 'n > 80', lw = 1)
ax[1,1].fill_between(pSA_T, e_CY14_n80['EAST']['std5'], e_CY14_n80['EAST']['std95'], alpha = 0.2)

# model EASZT
ax[1,2].plot(pSA_T, e_CY14_n20['EASZT']['std'], '-o', label = 'n > 20', lw = 1)
ax[1,2].fill_between(pSA_T, e_CY14_n20['EASZT']['std5'], e_CY14_n20['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_CY14_n30['EASZT']['std'], '-o', label = 'n > 30', lw = 1)
ax[1,2].fill_between(pSA_T, e_CY14_n30['EASZT']['std5'], e_CY14_n30['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_CY14_n40['EASZT']['std'], '-o', label = 'n > 40', lw = 1)
ax[1,2].fill_between(pSA_T, e_CY14_n40['EASZT']['std5'], e_CY14_n40['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_CY14_n50['EASZT']['std'], '-o', label = 'n > 50', lw = 1)
ax[1,2].fill_between(pSA_T, e_CY14_n50['EASZT']['std5'], e_CY14_n50['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_CY14_n60['EASZT']['std'], '-o', label = 'n > 60', lw = 1)
ax[1,2].fill_between(pSA_T, e_CY14_n60['EASZT']['std5'], e_CY14_n60['EASZT']['std95'], alpha = 0.2)
ax[1,2].plot(pSA_T, e_CY14_n80['EASZT']['std'], '-o', label = 'n > 80', lw = 1)
ax[1,2].fill_between(pSA_T, e_CY14_n80['EASZT']['std5'], e_CY14_n80['EASZT']['std95'], alpha = 0.2)

for i in range(2):
    for j in range(3):
        ax[i,j].set_ylim([1,3])
        ax[i,j].set_xlim([0.01,6])
        ax[i,j].grid(which='both', linestyle='--', color='k', alpha=0.25)
        ax[i,j].set_xscale('log')
        ax[1,j].set_xlabel('Period, T (s)')
        ax[i,0].set_ylabel(r'Correlation Deviation, $\sigma_{e}$')
ax[0,0].set_title('Model E')
ax[0,1].set_title('Model EA')
ax[0,2].set_title('Model EAS')
ax[1,0].set_title('Model EASZ')
ax[1,1].set_title('Model EAST')
ax[1,2].set_title('Model EASZT')
ax[0,1].legend(loc = 'upper center', ncol=3, fontsize = 10)
plt.show()

# 6.0 Non-stationary SCM performance

## 6.1 Crustal GMMs

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7), dpi=200, constrained_layout=True)
axes = axes.flatten()

datasets = {
    "BSSA14": LPPDP_reldiff_BSSA14,
    "CB14":   LPPDP_reldiff_CB14,
    "CY14":   LPPDP_reldiff_CY14,
    "ASK14":  LPPDP_reldiff_ASK14,
    "A22":  LPPDP_reldiff_ASK22,
    "S22":    LPPDP_reldiff_S22,
}

colors = {
    'EA': 'royalblue',
    'EAS': 'crimson',
    'EASZ': 'seagreen',
    'EAST': 'darkorange',
    'EASZT': 'saddlebrown',
}
markers = {
    'EA': 's',
    'EAS': 'o',
    'EASZ': '^',
    'EAST': 'D',
    'EASZT': '<',
}

for ax, (label, df) in zip(axes, datasets.items()):
    # keep your diffs logic exactly
    diffs = {
        'EA':   df[df.model == 'modelEA']['rel_diff'].reset_index(drop=True) -
                df[df.model == 'modelE']['rel_diff'].reset_index(drop=True),
        'EAS':  df[df.model == 'modelEAS']['rel_diff'].reset_index(drop=True) -
                df[df.model == 'modelE']['rel_diff'].reset_index(drop=True),
        'EASZ': df[df.model == 'modelEASZ']['rel_diff'].reset_index(drop=True) -
                df[df.model == 'modelE']['rel_diff'].reset_index(drop=True),
        'EAST': df[df.model == 'modelEAST']['rel_diff'].reset_index(drop=True) -
                df[df.model == 'modelE']['rel_diff'].reset_index(drop=True),
        'EASZT':df[df.model == 'modelEASZT']['rel_diff'].reset_index(drop=True) -
                df[df.model == 'modelE']['rel_diff'].reset_index(drop=True),
    }

    sorted_models = sorted(diffs.keys(), key=lambda m: diffs[m].iloc[-1])
    previous_diff = pd.Series([0] * len(pSA_T[:12]))

    for model in sorted_models:
        current_diff = diffs[model]
        ax.plot(pSA_T[:12], current_diff, marker=markers[model], linestyle='-', color=colors[model],
                markersize=6, alpha=0.9, label=f'Model {model}')
        ax.fill_between(pSA_T[:12], previous_diff, current_diff, color=colors[model], alpha=0.4)
        previous_diff = current_diff

    # baseline line
    ax.axhline(y=0, color='black', linewidth=2, zorder=0)
    ax.set_title(label, fontsize=11)
    ax.set_xscale('log')
    ax.set_xlim(0.01, 6)
    ax.set_ylim(0, 15)
    ax.grid(which='both', linestyle='-', linewidth=0.75, color='gray', alpha=0.2)
    ax.set_xticks([0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 6.0])
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.get_xaxis().set_minor_formatter(plt.NullFormatter())
    ax.set_xlabel('Period, T [s]')
    ax.set_ylabel('Relative LPPD Difference (%)')

# shared legend
handles, labels = axes[0].get_legend_handles_labels()
axes[0].legend(handles, labels,
               loc='upper left',   # adjust location as needed
               fontsize=10,
               frameon=True,
               framealpha=0.8)
plt.show()

## 6.1 Subduction GMMs

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), dpi=200, constrained_layout=True)
axes = axes.flatten()

datasets = {
    "AG20": LPPDP_reldiff_AG20,
    "K20":   LPPDP_reldiff_K20,
    "P20":   LPPDP_reldiff_P20,
}

colors = {
    'EA': 'royalblue',
    'EAS': 'crimson',
    'EASZ': 'seagreen',
    'EAST': 'darkorange',
    'EASZT': 'saddlebrown',
}
markers = {
    'EA': 's',
    'EAS': 'o',
    'EASZ': '^',
    'EAST': 'D',
    'EASZT': '<',
}

for ax, (label, df) in zip(axes, datasets.items()):
    # keep your diffs logic exactly
    diffs = {
        'EA':   df[df.model == 'modelEA']['rel_diff'].reset_index(drop=True) -
                df[df.model == 'modelE']['rel_diff'].reset_index(drop=True),
        'EAS':  df[df.model == 'modelEAS']['rel_diff'].reset_index(drop=True) -
                df[df.model == 'modelE']['rel_diff'].reset_index(drop=True),
        'EASZ': df[df.model == 'modelEASZ']['rel_diff'].reset_index(drop=True) -
                df[df.model == 'modelE']['rel_diff'].reset_index(drop=True),
        'EAST': df[df.model == 'modelEAST']['rel_diff'].reset_index(drop=True) -
                df[df.model == 'modelE']['rel_diff'].reset_index(drop=True),
        'EASZT':df[df.model == 'modelEASZT']['rel_diff'].reset_index(drop=True) -
                df[df.model == 'modelE']['rel_diff'].reset_index(drop=True),
    }

    sorted_models = sorted(diffs.keys(), key=lambda m: diffs[m].iloc[-1])
    previous_diff = pd.Series([0] * len(pSA_T[:12]))

    for model in sorted_models:
        current_diff = diffs[model]
        ax.plot(pSA_T[:12], current_diff, marker=markers[model], linestyle='-', color=colors[model],
                markersize=6, alpha=0.9, label=f'Model {model}')
        ax.fill_between(pSA_T[:12], previous_diff, current_diff, color=colors[model], alpha=0.4)
        previous_diff = current_diff

    # baseline line
    ax.axhline(y=0, color='black', linewidth=2, zorder=0)
    ax.set_title(label, fontsize=11)
    ax.set_xscale('log')
    ax.set_xlim(0.01, 6)
    ax.set_ylim(0, 6)
    ax.grid(which='both', linestyle='-', linewidth=0.75, color='gray', alpha=0.2)
    ax.set_xticks([0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 6.0])
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.get_xaxis().set_minor_formatter(plt.NullFormatter())
    ax.set_xlabel('Period, T [s]')
    ax.set_ylabel('Relative LPPD Difference (%)')

# shared legend
handles, labels = axes[0].get_legend_handles_labels()
axes[0].legend(handles, labels,
               loc='upper left',   # adjust location as needed
               fontsize=10,
               frameon=True,
               framealpha=0.8)
plt.show()